
<div dir="rtl" align="right">

# NB22E-v4 — ادامهٔ ارزیابی Retrieval فاز دوم

**پروژه:** RAG_finance  
**شاخهٔ مبنا:** `rag-phase2`  
**زیرکار:** R11.2  
**نقطهٔ ادامه:** خروجی موفق `NB22E-v3:C342`

## هدف

ادامهٔ آماده‌سازی و اجرای کنترل‌شدهٔ ارزیابی جمعیت Test، با مصرف شواهد تاریخی، بدون بازاجرای کارهای موفق.

## وضعیت انتقال

- جمعیت تاریخی: ۱۱۴۹ سؤال.
- ۲۹ اجرای دقیق هدف احراز شده‌اند.
- ۲ مورد با هویت اجرا نامحرز باقی مانده‌اند.
- ۲ مورد خطای ثبت‌شده محفوظ دارند.
- ۱۱۱۶ سؤال نیازمند تعیین و تثبیت مرز رتبه‌بندی و context هستند.
- R11.2 باز است و Gate نسخهٔ هدف BLOCKED باقی می‌ماند.

## ورودی‌ها

- `C342.summary` و `C342.upstream_candidate_manifest`
- `C341.question_bindings`
- قراردادهای فریز‌شده و رسیدهای Producer ارجاع‌شده در C342

## مسیر اجرا

۱. بارگذاری و اتصال دقیق Artifactهای انتقال.  
۲. آماده‌سازی مرز نسخهٔ جمعیتی مستقل.  
۳. کنترل محدود Index و Cache مدل.  
۴. طراحی Pilot محلی و اجرای آن فقط پس از مجوز.  
۵. تولید رتبه و context نسخه‌دار.  
۶. آماده‌سازی ورودی پاسخ و کنترل Cache و دفتر تلاش.  
۷. ادامهٔ ارزیابی، Regression و Gateهای قرارداد.

## مرزهای ممنوع

- تغییر مستقیم مخزن یا Artifactهای تاریخی.
- بازاجرای نوت‌بوک v3 یا سل‌های موفق C340 تا C342.
- تغییر Gold، مدل یا تنظیمات فریز‌شده بدون تصمیم قراردادی.
- اجرای خودکار API، Retrieval یا مدل در مرحلهٔ انتقال.

## خروجی‌های جدید

فقط در مسیر مستقل کنار مخزن:

`_RAG_finance_NB22E_v4_outputs`

## شرط پایان

Artifactهای رتبه، context، پاسخ، شواهد ارزیابی و Gateها باید مطابق قرارداد نسخه‌دار تولید، Reload و اعتبارسنجی شوند. موفقیت Loader به‌تنهایی پایان R11.2 نیست.

**وضعیت اولیه:** `OBJECTIVE_INCOMPLETE`

</div>



<div dir="rtl" align="right">

### NB22E-v4:C001–C002 — R11.2: انتقال مستقل و فقط‌خواندنی

**هدف:** ساخت Kernel سبک و مصرف مستقیم سه Artifact قطعی ادامه.

**ورودی‌ها:**
- Summary و Manifest نسخهٔ C342
- Question Bindings نسخهٔ C341
- هش‌های SHA-256 بایتی ثبت‌شده

**عملیات:**
- C001: واردکردن وابستگی‌های استاندارد و تنظیمات پایه.
- C002: یافتن محدود ریشهٔ پروژه، خواندن سه فایل مشخص، بررسی هش، Schema، تعداد ردیف‌ها، یکتایی شناسه و اتصال سؤال به Manifest.

**کنترل:**
- دقیقاً ۱۱۱۶ شناسهٔ یکتا در هر دو JSONL.
- تطبیق هش متن هر سؤال.
- کنترل وضعیت Producerها از Summary ثبت‌شده.
- حفظ موارد نامحرز و وضعیت BLOCKED.
- عدم تغییر مخزن و Artifactها.

**نتیجهٔ مورد انتظار:** `TRANSFER_LOADER_PASS` در صورت موفقیت تمام کنترل‌های محدود.

**API و هزینه:** صفر درخواست؛ صفر هزینهٔ API.

**شرط پایان اقدام:** موفقیت کنترل ورودی‌ها و آماده‌بودن شواهد برای تصمیم نسخهٔ جمعیتی؛ بدون ادعای آمادگی اجرای مدل.

</div>


In [1]:
# NB22E-v4 | C001 | R11.2
# Centralized imports and deferred optional dependencies; no network or writes.

import ast
import collections
import dataclasses
import hashlib
import html
import importlib
import importlib.metadata
import io
import json
import math
import os
import pickle
import platform
import re
import resource
import statistics
import subprocess
import sys
import tarfile
import tempfile
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
import uuid
import warnings

from collections import Counter, OrderedDict, defaultdict
from collections.abc import Mapping
from difflib import SequenceMatcher
from itertools import combinations
from pathlib import Path
from urllib.parse import urlsplit

NB22E_V4_NOTEBOOK_ID = "NB22E-v4"
NB22E_V4_REPOSITORY_NAME = "RAG_finance"
NB22E_V4_BRANCH = "rag-phase2"
NB22E_V4_SUBTASK = "R11.2"

NB22E_V4_READ_ONLY = True
NB22E_V4_ALLOW_API = False
NB22E_V4_ALLOW_MODEL_EXECUTION = False
NB22E_V4_ALLOW_RETRIEVAL = False
NB22E_V4_ALLOW_REPOSITORY_WRITES = False

NB22E_V4_PREVIOUS_NOTEBOOK = "NB22E-v3"
NB22E_V4_PREVIOUS_SUCCESSFUL_CELL = "C342"

_metadata = importlib.metadata
_platform = platform
_subprocess = subprocess

NB22E_IMPORT_SPECS = [
    ("qdrant_client", "QdrantClient", "QdrantClient"),
    ("dotenv", "dotenv_values", "dotenv_values"),
    ("numpy", None, "np"),
    ("onnxruntime", None, "ort"),
    ("torch", None, "torch"),
    ("transformers", "AutoTokenizer", "AutoTokenizer"),
    (
        "transformers",
        "AutoModelForSequenceClassification",
        "AutoModelForSequenceClassification",
    ),
    ("IPython.display", "display", "display"),
    ("IPython.display", "HTML", "HTML"),
    ("threadpoolctl", "threadpool_limits", "threadpool_limits"),
    ("sklearn", None, "sklearn"),
    (
        "sklearn.ensemble",
        "RandomForestClassifier",
        "RandomForestClassifier",
    ),
    ("lightgbm", None, "lgb"),
    ("xgboost", None, "xgb"),
    (
        "sentence_transformers",
        "SentenceTransformer",
        "SentenceTransformer",
    ),
]

NB22E_IMPORT_STATUS = {}
NB22E_V4_IMPORT_REGISTRY = {}

for module_name, attribute_name, binding_name in NB22E_IMPORT_SPECS:
    if binding_name in NB22E_V4_IMPORT_REGISTRY:
        raise RuntimeError(
            f"Duplicate optional import binding: {binding_name}"
        )

    NB22E_V4_IMPORT_REGISTRY[binding_name] = (
        module_name,
        attribute_name,
    )

    NB22E_IMPORT_STATUS[binding_name] = {
        "status": "DEFERRED",
        "module": module_name,
    }

    globals().pop(binding_name, None)


def nb22e_v4_require_imports(*binding_names):
    """Load only explicitly requested registered dependencies."""

    loaded = {}

    for binding_name in binding_names:
        if binding_name not in NB22E_V4_IMPORT_REGISTRY:
            raise KeyError(
                f"Unregistered optional dependency: {binding_name}"
            )

        module_name, attribute_name = (
            NB22E_V4_IMPORT_REGISTRY[binding_name]
        )

        if NB22E_IMPORT_STATUS[binding_name]["status"] == "AVAILABLE":
            loaded[binding_name] = globals()[binding_name]
            continue

        try:
            module = importlib.import_module(module_name)

            value = (
                module
                if attribute_name is None
                else getattr(module, attribute_name)
            )

            globals()[binding_name] = value

            NB22E_IMPORT_STATUS[binding_name] = {
                "status": "AVAILABLE",
                "module": module_name,
            }

            loaded[binding_name] = value

        except Exception as exc:
            globals().pop(binding_name, None)

            NB22E_IMPORT_STATUS[binding_name] = {
                "status": "UNAVAILABLE",
                "module": module_name,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }

            raise RuntimeError(
                f"Optional dependency unavailable: {binding_name}: "
                f"{type(exc).__name__}: {exc}"
            ) from exc

    return loaded


assert NB22E_V4_READ_ONLY is True
assert NB22E_V4_ALLOW_API is False
assert NB22E_V4_ALLOW_MODEL_EXECUTION is False
assert NB22E_V4_ALLOW_RETRIEVAL is False
assert NB22E_V4_ALLOW_REPOSITORY_WRITES is False

assert len(NB22E_IMPORT_SPECS) == len(
    NB22E_V4_IMPORT_REGISTRY
)

assert hasattr(uuid, "uuid5")
assert hasattr(uuid, "UUID")

NB22E_V4_BOOTSTRAP_STATUS = {
    "cell": "NB22E-v4:C001",
    "status": "LIGHTWEIGHT_BOOTSTRAP_READY",
    "repository_policy": "READ_ONLY",
    "uuid_import": "AVAILABLE",
    "optional_dependencies_registered": len(
        NB22E_V4_IMPORT_REGISTRY
    ),
    "optional_dependencies_deferred": len(
        NB22E_V4_IMPORT_REGISTRY
    ),
    "api_calls": 0,
    "model_runs": 0,
    "repository_modified": False,
}

print(json.dumps(
    NB22E_V4_BOOTSTRAP_STATUS,
    ensure_ascii=False,
    indent=2,
))

{
  "cell": "NB22E-v4:C001",
  "status": "LIGHTWEIGHT_BOOTSTRAP_READY",
  "repository_policy": "READ_ONLY",
  "uuid_import": "AVAILABLE",
  "optional_dependencies_registered": 15,
  "optional_dependencies_deferred": 15,
  "api_calls": 0,
  "model_runs": 0,
  "repository_modified": false
}


In [2]:
# NB22E-v4 | C002 | R11.2
# Read-only C341/C342 transfer loader: exact hashes, schemas and candidate join.

if not NB22E_V4_READ_ONLY:
    raise RuntimeError("C002_REQUIRES_READ_ONLY_MODE")

def c002_require(condition, message):
    if not condition:
        raise RuntimeError("C002_" + message)

# 1. Bounded project-root resolution; no recursive directory search.

c002_candidates = []

c002_env_root = os.environ.get("RAG_FINANCE_ROOT")
if c002_env_root:
    c002_candidates.append(Path(c002_env_root).expanduser())

c002_cwd = Path.cwd().resolve()
c002_candidates.extend((c002_cwd, *c002_cwd.parents))

# Known project layout, relative to the user's home directory.
c002_candidates.append(
    Path.home()
    / "AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

c002_root = None
for candidate in c002_candidates:
    candidate = candidate.resolve()
    if (
        candidate.name == NB22E_V4_REPOSITORY_NAME
        and (candidate / "notebooks/phase2").is_dir()
    ):
        c002_root = candidate
        break

c002_require(c002_root is not None, "PROJECT_ROOT_NOT_FOUND")

c002_parent = c002_root.parent.resolve()

# 2. Exact receipts from the successful historical producer outputs.

c002_specs = {
    "summary": {
        "folder": "_RAG_finance_C342_rank_route_preflight_outputs",
        "stem": "C342.summary",
        "sha": "fe34926715265f4595dc10084ab18a3dcaca0fbbaf72b37de323466648616b3b",
        "size": 8623,
        "extension": "json",
    },
    "manifest": {
        "folder": "_RAG_finance_C342_rank_route_preflight_outputs",
        "stem": "C342.upstream_candidate_manifest",
        "sha": "4167708704c10ae1b556c0ee82dfdb517d8a9511d61b0723b1f8b0b1c8649dcc",
        "size": 1365688,
        "extension": "jsonl",
    },
    "questions": {
        "folder": "_RAG_finance_C341_question_binding_outputs",
        "stem": "C341.question_bindings",
        "sha": "9a4f0c044b459e78930ab6f0aded8f7d1bb75b677dbe170679333011cd94b5e3",
        "size": 654033,
        "extension": "jsonl",
    },
}

c002_paths = {}

for role, spec in c002_specs.items():
    expected_parent = (c002_parent / spec["folder"]).resolve()
    name = f'{spec["stem"]}.{spec["sha"]}.{spec["extension"]}'
    path = (expected_parent / name).resolve()

    c002_require(
        expected_parent.parent == c002_parent
        and path.parent == expected_parent
        and not path.is_relative_to(c002_root),
        role.upper() + "_UNSAFE_PATH",
    )

    c002_require(path.is_file(), role.upper() + "_FILE_MISSING")
    c002_require(
        path.stat().st_size == spec["size"],
        role.upper() + "_SIZE_MISMATCH",
    )
    c002_paths[role] = path


def c002_read_summary(path, spec):
    before = path.stat()
    raw = path.read_bytes()
    after = path.stat()

    c002_require(
        before.st_size == len(raw) == spec["size"]
        and (before.st_size, before.st_mtime_ns)
        == (after.st_size, after.st_mtime_ns),
        "SUMMARY_STABILITY_CONFLICT",
    )

    c002_require(
        hashlib.sha256(raw).hexdigest() == spec["sha"],
        "SUMMARY_SHA256_CONFLICT",
    )

    result = json.loads(raw)
    c002_require(isinstance(result, dict), "SUMMARY_SCHEMA")
    return result


def c002_read_jsonl(role, consumer, expected_rows=1116):
    spec = c002_specs[role]
    path = c002_paths[role]

    before = path.stat()
    digest = hashlib.sha256()
    count = 0

    with path.open("rb") as handle:
        for line_number, raw in enumerate(handle, 1):
            c002_require(
                line_number <= expected_rows
                and len(raw) <= 512 * 1024
                and raw.endswith(b"\n")
                and bool(raw.strip()),
                role.upper() + "_JSONL_LIMIT_OR_FORMAT",
            )

            digest.update(raw)
            row = json.loads(raw)

            c002_require(
                isinstance(row, dict),
                role.upper() + "_ROW_SCHEMA",
            )

            consumer(row, line_number)
            count += 1

    after = path.stat()

    c002_require(
        (before.st_size, before.st_mtime_ns)
        == (after.st_size, after.st_mtime_ns),
        role.upper() + "_CHANGED_DURING_READ",
    )
    c002_require(
        digest.hexdigest() == spec["sha"],
        role.upper() + "_SHA256_CONFLICT",
    )
    c002_require(
        count == expected_rows,
        role.upper() + "_ROW_COUNT_CONFLICT",
    )

    return count


# 3. Validate the exact C342 summary and its historical status.

c002_summary = c002_read_summary(
    c002_paths["summary"],
    c002_specs["summary"],
)

c002_require(
    c002_summary.get("cell") == "C342"
    and c002_summary.get("roadmap_subtask") == "R11.2"
    and c002_summary.get("status")
    == "FROZEN_RANK_PRODUCER_ROUTE_PREFLIGHT_NO_EXECUTION",
    "SUMMARY_IDENTITY_CONFLICT",
)

c002_expected_preserved = {
    "EXACT_PRESERVED": 29,
    "MODEL_IDENTITY_HELD": 2,
    "RECORDED_ERROR_HELD": 2,
    "OUTSIDE_PACKET": 1116,
}

c002_require(
    c002_summary.get("candidate_count") == 1116
    and c002_summary.get("historical_input_candidates_verified") == 1116
    and c002_summary.get("preserved") == c002_expected_preserved
    and c002_summary.get("grouped_candidate_input_blockers") == [],
    "SUMMARY_COUNTS_OR_BLOCKERS_CONFLICT",
)

c002_require(
    c002_summary.get("new_requests_ready_for_dispatch") == 0
    and c002_summary.get("new_api_calls") == 0
    and c002_summary.get("new_retrieval_runs") == 0
    and c002_summary.get("new_rank_runs") == 0
    and c002_summary.get("original_target_identity_reused") is False
    and c002_summary.get("new_population_requires_version_decision") is True,
    "SUMMARY_EXECUTION_OR_VERSION_CONFLICT",
)

c002_receipt = c002_summary.get("manifest_receipt", {})

c002_require(
    isinstance(c002_receipt, dict)
    and Path(c002_receipt.get("path", "")).resolve()
    == c002_paths["manifest"]
    and c002_receipt.get("file_bytes_sha256")
    == c002_specs["manifest"]["sha"]
    and c002_receipt.get("size_bytes")
    == c002_specs["manifest"]["size"],
    "MANIFEST_RECEIPT_CONFLICT",
)

c002_producers = c002_summary.get("producer_contract_checks", {})
c002_sources = c002_summary.get("source_checks", {})
c002_inputs = c002_summary.get("input_file_checks", {})

c002_require(
    isinstance(c002_producers, dict)
    and bool(c002_producers)
    and all(value is True for value in c002_producers.values()),
    "PRODUCER_CONTRACT_STATUS",
)

c002_require(
    isinstance(c002_sources, dict)
    and len(c002_sources) == 12
    and all(
        isinstance(item, dict)
        and item.get("status") == "FILE_BYTES_VERIFIED"
        for item in c002_sources.values()
    ),
    "SOURCE_RECEIPT_STATUS",
)

c002_roles = ("GOLDEN", "HYBRID", "RERANKER")

c002_require(
    isinstance(c002_inputs, dict)
    and set(c002_inputs) == {*c002_roles, "CHUNKS"}
    and all(
        isinstance(item, dict)
        and item.get("status") == "FILE_BYTES_VERIFIED"
        for item in c002_inputs.values()
    ),
    "INPUT_RECEIPT_STATUS",
)

# 4. Stream C341 question bindings and verify text/hash identity.

c002_questions = {}

def c002_consume_question(row, line_number):
    cid = row.get("candidate_id")
    question = row.get("question")
    recorded_sha = row.get("question_sha256")

    c002_require(
        isinstance(cid, str) and bool(cid)
        and cid not in c002_questions
        and isinstance(question, str) and bool(question)
        and isinstance(recorded_sha, str)
        and row.get("question_only_not_response_or_context") is True
        and isinstance(row.get("source_line_number"), int)
        and row["source_line_number"] > 0,
        "QUESTION_SCHEMA_OR_DUPLICATE",
    )

    actual_sha = hashlib.sha256(
        question.encode("utf-8")
    ).hexdigest()

    c002_require(
        actual_sha == recorded_sha,
        "QUESTION_TEXT_HASH_CONFLICT",
    )

    c002_questions[cid] = {
        "question": question,
        "question_sha256": recorded_sha,
    }

c002_question_count = c002_read_jsonl(
    "questions",
    c002_consume_question,
)

# 5. Stream C342 manifest and perform the exact 1116-ID join.

c002_remaining = set(c002_questions)
c002_manifest_status_count = 0

def c002_consume_manifest(row, line_number):
    global c002_manifest_status_count

    cid = row.get("candidate_id")

    c002_require(
        isinstance(cid, str)
        and cid in c002_remaining,
        "MANIFEST_UNKNOWN_OR_DUPLICATE_ID",
    )

    c002_remaining.remove(cid)

    c002_require(
        row.get("question_sha256")
        == c002_questions[cid]["question_sha256"],
        "MANIFEST_QUESTION_HASH_CONFLICT",
    )

    c002_require(
        row.get("historical_input_status")
        == "CANDIDATE_FOR_FROZEN_REPRODUCTION"
        and row.get("blockers") == []
        and row.get("target_char_rrf_rank_proven") is False
        and row.get("target_identity_8d5875_reused") is False
        and row.get("send_authorized") is False,
        "MANIFEST_VERSION_OR_READINESS_CONFLICT",
    )

    receipts = row.get("source_row_receipts")

    c002_require(
        isinstance(receipts, dict)
        and set(receipts) == set(c002_roles),
        "MANIFEST_SOURCE_RECEIPT_SCHEMA",
    )

    for role in c002_roles:
        receipt = receipts[role]
        pinned = c002_inputs[role]

        c002_require(
            isinstance(receipt, dict)
            and receipt.get("path") == pinned.get("path")
            and receipt.get("file_bytes_sha256")
            == pinned.get("file_bytes_sha256")
            and type(receipt.get("line_number")) is int
            and receipt["line_number"] > 0
            and isinstance(
                receipt.get("row_sha256_excluding_line_ending"),
                str,
            )
            and len(receipt["row_sha256_excluding_line_ending"]) == 64,
            "MANIFEST_SOURCE_RECEIPT_CONFLICT",
        )

    c002_manifest_status_count += 1


c002_manifest_count = c002_read_jsonl(
    "manifest",
    c002_consume_manifest,
)

c002_require(
    not c002_remaining
    and c002_manifest_status_count == 1116,
    "QUESTION_MANIFEST_JOIN_INCOMPLETE",
)

# 6. Preserve the pending Index/Cache gates without executing them.

c002_index = c002_summary.get("dense_index_precheck", {})
c002_vectors = c002_summary.get("query_vector_precheck", {})
c002_model_cache = c002_summary.get("model_cache_precheck", {})

c002_require(
    isinstance(c002_index, dict)
    and c002_index.get("frozen_index_hash_reverified") is False
    and isinstance(c002_vectors, dict)
    and c002_vectors.get("vector_candidate_id_coverage") == 1116
    and isinstance(c002_model_cache, dict),
    "PENDING_RUNTIME_PRECHECK_SCHEMA",
)

NB22E_V4_TRANSFER = {
    "status": "TRANSFER_LOADER_PASS",
    "candidate_count": c002_manifest_count,
    "questions_bound": c002_question_count,
    "artifact_paths": {
        role: str(path)
        for role, path in c002_paths.items()
    },
    "artifact_sha256": {
        role: spec["sha"]
        for role, spec in c002_specs.items()
    },
    "version_decision": c002_summary["version_decision"],
    "index_status": c002_index.get("status"),
    "model_cache_status": c002_model_cache.get("status"),
    "send_authorized": False,
}

print(json.dumps({
    "cell": "NB22E-v4:C002",
    "status": NB22E_V4_TRANSFER["status"],
    "file_bytes_verified": list(c002_specs),
    "question_count": c002_question_count,
    "manifest_count": c002_manifest_count,
    "joined_ids": 1116 - len(c002_remaining),
    "producer_contract_checks": c002_producers,
    "preserved": c002_expected_preserved,
    "dense_index_status": c002_index.get("status"),
    "dense_index_hash_reverified": False,
    "model_cache_status": c002_model_cache.get("status"),
    "version_decision": c002_summary["version_decision"],
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "repository_modified": False,
    "next": "VERSION_BOUNDARY_PREPARATION",
}, ensure_ascii=False, indent=2))


{
  "cell": "NB22E-v4:C002",
  "status": "TRANSFER_LOADER_PASS",
  "file_bytes_verified": [
    "summary",
    "manifest",
    "questions"
  ],
  "question_count": 1116,
  "manifest_count": 1116,
  "joined_ids": 1116,
  "producer_contract_checks": {
    "C118_runtime_matches": true,
    "C119_parity_verified": true,
    "C139_caller_bound": true,
    "C186_input_pins_match": true,
    "C188_33_only": true,
    "NB22C_functions_pinned": true,
    "NB22D_functions_pinned": true
  },
  "preserved": {
    "EXACT_PRESERVED": 29,
    "MODEL_IDENTITY_HELD": 2,
    "RECORDED_ERROR_HELD": 2,
    "OUTSIDE_PACKET": 1116
  },
  "dense_index_status": "DIRECTORY_PRESENT_VERSION_UNREVALIDATED",
  "dense_index_hash_reverified": false,
  "model_cache_status": "PRESENT_REQUIRES_RECHECK_BEFORE_LOCAL_EXECUTION",
  "version_decision": "NEW_VERSIONED_POPULATION_AND_TARGET_GENERATION_BOUNDARY_RECEIPTS_REQUIRED_BEFORE_LABELLING_NEW_RESULTS_AS_TARGET",
  "target_release_gate": "BLOCKED",
  "api_calls": 0,
  "r


<div dir="rtl" align="right">

### NB22E-v4:C003 — R11.2: آماده‌سازی مرز نسخهٔ جمعیتی

**هدف:** ایجاد پیش‌نویس مستقل و قابل‌ردیابی از مرز نسخهٔ جدید برای ۱۱۱۶ سؤال، بدون اعطای هویت نهایی یا اجرای محاسبات Retrieval.

**ورودی‌ها:**
- Artifactهای تأییدشدهٔ C002.
- نسخهٔ فریز‌شدهٔ قالب درخواست C340.
- وضعیت Producerها، ورودی‌ها و محدودیت‌های C342.

**عملیات:**
۱. کنترل اتصال هویت تاریخی و تنظیمات واقعی مدل.  
۲. ثبت منابع فریز‌شده و مرز جمعیت جدید.  
۳. تفکیک رتبه و context تاریخی از خروجی‌های هنوز تولیدنشده.  
۴. محاسبهٔ Fingerprint پیشنهادی از محتوای واقعی.  
۵. ذخیرهٔ Atomic پیش‌نویس فقط در پوشهٔ خروجی مستقل کنار مخزن.

**خروجی:** فایل JSON نسخه‌دار با وضعیت `DRAFT_AWAITING_VERSION_APPROVAL`.

**کنترل:** هش بایتی، کنترل نسخه، عدم بازنویسی Artifact موجود و Reload فایل نهایی.

**محدودیت:** Fingerprint پیشنهادی، هویت نسخهٔ نهایی پاسخ نیست. Index، Cache مدل، تولید رتبه و context همچنان نیازمند اقدامات بعدی‌اند.

**API:** صفر.  
**تغییر مخزن:** صفر.

**شرط پایان:** ثبت موفق و بازخوانی پیش‌نویس معتبر؛ بدون تغییر وضعیت Gate اصلی.

</div>


In [3]:
# NB22E-v4 | C003 | R11.2
# Prepare a content-bound population version proposal; no repository writes or execution.

# ============================================================
# 1. Require the successful C002 transfer state
# ============================================================

if (
    not NB22E_V4_READ_ONLY
    or NB22E_V4_TRANSFER.get("status") != "TRANSFER_LOADER_PASS"
):
    raise RuntimeError("C003_REQUIRES_SUCCESSFUL_READ_ONLY_C002")

if (
    NB22E_V4_TRANSFER.get("candidate_count") != 1116
    or NB22E_V4_TRANSFER.get("questions_bound") != 1116
    or NB22E_V4_TRANSFER.get("send_authorized") is not False
):
    raise RuntimeError("C003_TRANSFER_BOUNDARY_CONFLICT")

c003_root = c002_root.resolve()
c003_parent = c003_root.parent.resolve()

if c003_root.name != "RAG_finance":
    raise RuntimeError("C003_PROJECT_ROOT_CONFLICT")

for role in ("summary", "manifest", "questions"):
    if (
        NB22E_V4_TRANSFER["artifact_sha256"].get(role)
        != c002_specs[role]["sha"]
        or Path(NB22E_V4_TRANSFER["artifact_paths"][role]).resolve()
        != c002_paths[role]
    ):
        raise RuntimeError("C003_TRANSFER_RECEIPT_CONFLICT:" + role)

# ============================================================
# 2. Read only the exact C340 frozen-template receipt
# ============================================================

c003_template_sha = (
    "f72301a2309c5c66aa84b0a5ba3eda96dcee703e1d93d67c9df71e40942240d9"
)

c003_template_path = (
    c003_parent
    / "_RAG_finance_C340_preflight_outputs"
    / ("C340.frozen_template." + c003_template_sha + ".json")
).resolve()

if (
    c003_template_path.parent
    != (c003_parent / "_RAG_finance_C340_preflight_outputs").resolve()
    or not c003_template_path.is_file()
):
    raise RuntimeError("C003_FROZEN_TEMPLATE_MISSING_OR_UNSAFE")

c003_before = c003_template_path.stat()

if c003_before.st_size != 47393:
    raise RuntimeError("C003_FROZEN_TEMPLATE_SIZE_CONFLICT")

c003_raw = c003_template_path.read_bytes()
c003_after = c003_template_path.stat()

if (
    c003_before.st_size != c003_after.st_size
    or c003_before.st_mtime_ns != c003_after.st_mtime_ns
    or hashlib.sha256(c003_raw).hexdigest() != c003_template_sha
):
    raise RuntimeError("C003_FROZEN_TEMPLATE_SHA256_CONFLICT")

c003_template = json.loads(c003_raw)

def c003_canonical(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

def c003_sha(raw):
    return hashlib.sha256(raw).hexdigest()

# ============================================================
# 3. Verify frozen model and historical identity
# ============================================================

c003_original_identity = c002_summary.get(
    "target_identity_sha256_original"
)

c003_request = c003_template.get("reference_request", {})
c003_identity = c003_template.get("target_identity", {})

if not (
    isinstance(c003_request, dict)
    and isinstance(c003_identity, dict)
    and isinstance(c003_request.get("messages"), list)
    and len(c003_request["messages"]) == 2
):
    raise RuntimeError("C003_TEMPLATE_SCHEMA_CONFLICT")

c003_profile = {
    key: value
    for key, value in c003_request.items()
    if key != "messages"
}

c003_expected_profile = {
    "model": "DeepSeek-V4-Flash",
    "temperature": 0,
    "max_tokens": 600,
    "thinking": {"type": "disabled"},
}

c003_messages = c003_request["messages"]

if not (
    c003_profile == c003_expected_profile
    and c003_identity.get("profile") == c003_expected_profile
    and c003_messages[0].get("role") == "system"
    and c003_messages[1].get("role") == "user"
    and isinstance(c003_messages[0].get("content"), str)
):
    raise RuntimeError("C003_FROZEN_MODEL_PROFILE_CONFLICT")

c003_prompt_sha = c003_sha(
    c003_messages[0]["content"].encode("utf-8")
)

if not (
    c003_sha(c003_canonical(c003_identity))
    == c003_template.get("target_identity_sha256")
    == c003_original_identity
    == "8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065"
    and c003_prompt_sha
    == c003_template.get("system_prompt_sha256")
    == c003_identity.get("system_prompt_sha256")
):
    raise RuntimeError("C003_HISTORICAL_TARGET_IDENTITY_CONFLICT")

# ============================================================
# 4. Verify source coverage and pending version conditions
# ============================================================

c003_sources = c002_summary["source_checks"]
c003_inputs = c002_summary["input_file_checks"]
c003_producers = c002_summary["producer_contract_checks"]

if not (
    all(v is True for v in c003_producers.values())
    and all(
        value.get("status") == "FILE_BYTES_VERIFIED"
        for value in c003_sources.values()
    )
    and all(
        value.get("status") == "FILE_BYTES_VERIFIED"
        for value in c003_inputs.values()
    )
    and c002_summary.get("existing_final_rank_coverage")
    == "C189_ONLY_33_VERIFIED"
    and c002_summary.get("frozen_final_char_rrf_rank_newly_bound") == 0
    and c002_summary.get("original_target_identity_reused") is False
    and c002_summary.get("new_population_requires_version_decision") is True
):
    raise RuntimeError("C003_PRODUCER_OR_VERSION_STATUS_CONFLICT")

c003_index = c002_summary["dense_index_precheck"]
c003_vector = c002_summary["query_vector_precheck"]
c003_cache = c002_summary["model_cache_precheck"]

if not (
    c003_index.get("frozen_index_hash_reverified") is False
    and c003_vector.get("vector_candidate_id_coverage") == 1116
    and c003_cache.get("status")
    == "PRESENT_REQUIRES_RECHECK_BEFORE_LOCAL_EXECUTION"
):
    raise RuntimeError("C003_RUNTIME_PENDING_STATE_CONFLICT")

# ============================================================
# 5. Build a proposal anchored to verified artifact content
# ============================================================

c003_core = {
    "schema": "NB22E_V4_R11_2_POPULATION_BOUNDARY_DRAFT_V1",
    "cell": "NB22E-v4:C003",
    "subtask": "R11.2",
    "status": "DRAFT_AWAITING_VERSION_APPROVAL",
    "population": {
        "historical_test_count": 1149,
        "new_candidate_count": 1116,
        "manifest_file_bytes_sha256": c002_specs["manifest"]["sha"],
        "questions_file_bytes_sha256": c002_specs["questions"]["sha"],
        "source_summary_file_bytes_sha256": c002_specs["summary"]["sha"],
        "preserved": c002_summary["preserved"],
    },
    "historical_target_lineage": {
        "identity_sha256": c003_original_identity,
        "rank_input_mode": c002_summary["original_rank_input_mode"],
        "existing_rank_coverage": "C189_ONLY_33_VERIFIED",
        "frozen_template_file_bytes_sha256": c003_template_sha,
    },
    "frozen_generation_profile": {
        "request_profile": c003_profile,
        "system_prompt_sha256": c003_prompt_sha,
        "request_wrapper_source": "C340_FROZEN_TEMPLATE",
        "historical_identity_only": True,
    },
    "producer_evidence": {
        "source_checks": c003_sources,
        "input_file_checks": c003_inputs,
        "producer_contract_checks": c003_producers,
        "upstream_candidate_manifest_sha256": (
            c002_specs["manifest"]["sha"]
        ),
    },
    "proposed_new_boundary": {
        "rank_input_mode": "PENDING_VERSION_BOUND_RANK_RECEIPTS",
        "retrieval_input_identity": "PENDING_VERSION_VALIDATION",
        "rank_receipts_for_new_population": "NOT_YET_PRODUCED",
        "ordered_context_receipts": "NOT_YET_PRODUCED",
        "rendered_context_receipts": "NOT_YET_PRODUCED",
        "generation_request_receipts": "NOT_YET_PRODUCED",
        "new_target_identity": "NOT_ASSIGNED",
        "historical_identity_transfer_authorized": False,
        "configuration_selection_on_test_authorized": False,
        "generation_dispatch_authorized": False,
    },
    "runtime_prechecks": {
        "dense_index": c003_index,
        "query_vectors": c003_vector,
        "model_cache": c003_cache,
    },
    "planned_execution_sequence": (
        c002_summary["next_execution_sequence_after_explicit_version_approval"]
    ),
    "pending": [
        "SUPERVISOR_VERSION_BOUNDARY_DECISION",
        "DENSE_INDEX_EXACT_VERSION_VERIFICATION",
        "LOCAL_MODEL_CACHE_EXACT_VERSION_VERIFICATION",
        "HISTORICAL_POOL50_VERSION_MEMBERSHIP_CHECK",
        "NEW_RANK_AND_CONTEXT_RECEIPTS",
        "NEW_GENERATION_REQUEST_AND_ATTEMPT_CONTROL",
        "R11_2_EVALUATION_AND_REGRESSION_GATES",
    ],
    "constraints": {
        "repository_read_only": True,
        "historical_artifacts_immutable": True,
        "no_retrieval_in_this_cell": True,
        "no_ranking_in_this_cell": True,
        "no_model_execution_in_this_cell": True,
        "no_api_in_this_cell": True,
        "proposal_is_not_target_release_identity": True,
    },
}

c003_proposal_sha = c003_sha(c003_canonical(c003_core))

c003_document = {
    **c003_core,
    "proposal_content_sha256": c003_proposal_sha,
    "fingerprint_kind": "CANONICAL_JSON_PROPOSAL_CONTENT",
}

# ============================================================
# 6. Publish an immutable draft outside the repository
# ============================================================

c003_out = (
    c003_parent / "_RAG_finance_NB22E_v4_outputs"
).resolve()

if (
    c003_out == c003_root
    or c003_out.is_relative_to(c003_root)
):
    raise RuntimeError("C003_OUTPUT_INSIDE_REPOSITORY")

c003_out.mkdir(parents=True, exist_ok=True)

c003_bytes = c003_canonical(c003_document) + b"\n"
c003_file_sha = c003_sha(c003_bytes)

c003_target = c003_out / (
    "C003.boundary_proposal."
    + c003_file_sha
    + ".json"
)

# Atomic, no-overwrite publication.
# The temporary file and final file are on the same filesystem.

if not c003_target.exists():
    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=c003_out,
        prefix=".C003.boundary.",
        suffix=".tmp",
        delete=False,
    ) as c003_handle:
        c003_temp = Path(c003_handle.name)
        c003_handle.write(c003_bytes)
        c003_handle.flush()
        os.fsync(c003_handle.fileno())

    try:
        if c003_temp.read_bytes() != c003_bytes:
            raise RuntimeError("C003_TEMP_RELOAD_CONFLICT")

        try:
            os.link(c003_temp, c003_target)
        except FileExistsError:
            pass
    finally:
        c003_temp.unlink(missing_ok=True)

# ============================================================
# 7. Independent reload and fingerprint verification
# ============================================================

c003_reloaded_bytes = c003_target.read_bytes()

if (
    c003_reloaded_bytes != c003_bytes
    or c003_sha(c003_reloaded_bytes) != c003_file_sha
):
    raise RuntimeError("C003_FINAL_FILE_BYTES_CONFLICT")

c003_reloaded = json.loads(c003_reloaded_bytes)

c003_reloaded_core = {
    key: value
    for key, value in c003_reloaded.items()
    if key not in (
        "proposal_content_sha256",
        "fingerprint_kind",
    )
}

if not (
    c003_reloaded["proposal_content_sha256"]
    == c003_sha(c003_canonical(c003_reloaded_core))
    == c003_proposal_sha
    and c003_reloaded["status"]
    == "DRAFT_AWAITING_VERSION_APPROVAL"
    and c003_reloaded["population"]["new_candidate_count"] == 1116
):
    raise RuntimeError("C003_RELOAD_VALIDATION_CONFLICT")

NB22E_V4_BOUNDARY_PROPOSAL = {
    "status": "DRAFT_RELOAD_PASS",
    "path": str(c003_target),
    "file_bytes_sha256": c003_file_sha,
    "proposal_content_sha256": c003_proposal_sha,
    "new_target_identity_assigned": False,
    "approved": False,
}

print(json.dumps({
    "cell": "NB22E-v4:C003",
    "status": "DRAFT_RELOAD_PASS",
    "population": 1116,
    "historical_identity": c003_original_identity,
    "historical_rank_mode": c002_summary["original_rank_input_mode"],
    "proposed_rank_mode": "PENDING_VERSION_BOUND_RANK_RECEIPTS",
    "model": c003_profile["model"],
    "frozen_profile_verified": True,
    "proposal_content_sha256": c003_proposal_sha,
    "output_receipt": {
        "path": str(c003_target),
        "file_bytes_sha256": c003_file_sha,
        "size_bytes": len(c003_bytes),
    },
    "new_target_identity_assigned": False,
    "version_approval": "PENDING",
    "dense_index_version": "PENDING",
    "local_model_cache_version": "PENDING",
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "repository_modified": False,
    "next": "INDEX_AND_MODEL_CACHE_LIMITED_VERIFICATION",
}, ensure_ascii=False, indent=2))


{
  "cell": "NB22E-v4:C003",
  "status": "DRAFT_RELOAD_PASS",
  "population": 1116,
  "historical_identity": "8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065",
  "historical_rank_mode": "VERSION_BOUND_STORED_C189_RANKS",
  "proposed_rank_mode": "PENDING_VERSION_BOUND_RANK_RECEIPTS",
  "model": "DeepSeek-V4-Flash",
  "frozen_profile_verified": true,
  "proposal_content_sha256": "72a6dbc8de189b4add1032f9d9c0330b75935bfb7ff95eda982a332ea1edb327",
  "output_receipt": {
    "path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/_RAG_finance_NB22E_v4_outputs/C003.boundary_proposal.0dbfd6e4c00de3d4e4ec75f616f2cbc09a1b63d025a4648249684f43593fb492.json",
    "file_bytes_sha256": "0dbfd6e4c00de3d4e4ec75f616f2cbc09a1b63d025a4648249684f43593fb492",
    "size_bytes": 9588
  },
  "new_target_identity_assigned": false,
  "version_approval": "PENDING",
  "dense_index_version": "PENDING",
  "local_model_cache_version": "PENDING",
  "target_release_gate": "BLOCKED",
  "a

<div dir="rtl" align="right">

### NB22E-v4:C004 — R11.2: کنترل محدود Index و Cache مدل

**هدف:** بررسی اعتبار فعلی فایل‌های قرارداد Index و مدل محلی، بدون اجرای Retrieval یا مدل.

**ورودی‌ها:**
- خروجی موفق C002 و پیش‌نویس معتبر C003.
- چهار فایل فریز‌شدهٔ Index با هش‌های ثبت‌شده.
- مدل ONNX و تنظیمات ثبت‌شده در C342.

**عملیات:**
۱. بررسی SHA-256 قراردادهای Index و تطبیق شناسهٔ نسخه، Collection و مسیر ذخیره‌سازی.  
۲. کنترل وجود مسیر فیزیکی Index، بدون بازکردن Qdrant.  
۳. محاسبهٔ قطعه‌ای هش مدل ONNX و مقایسه با نسخهٔ فریز‌شده.  
۴. بررسی فایل‌های Tokenizer و نسخهٔ نصب‌شدهٔ ONNX Runtime.  
۵. گزارش جداگانهٔ کنترل‌های موفق و موارد هنوز نامحرز.

**محدودیت:** تطبیق Metadata اثبات سلامت محتوای زندهٔ Qdrant نیست. این کنترل جداگانه و بدون امکان تغییر Index اصلی انجام خواهد شد.

**خروجی:** گزارش وضعیت در همان Kernel؛ بدون نوشتن فایل.

**API:** صفر.  
**تغییر مخزن:** صفر.  
**شرط پایان:** مشخص‌شدن اعتبار قراردادهای Index، هش مدل و موانع واقعی اجرای محلی.

</div>

In [4]:
# NB22E-v4 | C004 | R11.2
# Repair: verified historical SHA-256 pins; read-only Index and ONNX preflight.

# ============================================================
# 1. Previous cell and repository safety
# ============================================================

if not (
    NB22E_V4_READ_ONLY is True
    and NB22E_V4_TRANSFER.get("status") == "TRANSFER_LOADER_PASS"
    and NB22E_V4_BOUNDARY_PROPOSAL.get("status") == "DRAFT_RELOAD_PASS"
):
    raise RuntimeError("C004_REQUIRES_VERIFIED_C002_C003")

c004_root = c002_root.resolve()

if c004_root.name != "RAG_finance":
    raise RuntimeError("C004_PROJECT_ROOT_CONFLICT")

# ============================================================
# 2. Exact historical file-byte pins from NB22E-v3:C148
# ============================================================

c004_pins = {
    "release": (
        "artifacts/phase2/dense_index/dense_index_release_v1.json",
        "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
    ),
    "handoff": (
        "artifacts/phase2/dense_index/nb16_dense_index_handoff_v1.json",
        "da7f812a79a21b1876c74668a548badb0f271b2d18ac2f1b2cf9663f789f301f",
    ),
    "manifest": (
        "data/phase2/index/dense_index_v1/dense_index_manifest.json",
        "739539f7abdfd0df435389e103658e28244b078f425f33878458034058bfae47",
    ),
    "lifecycle": (
        "data/phase2/index/dense_index_v1/index_lifecycle_evidence.json",
        "bcea3bda1ee7ad86c9c6ca40ab674aa9bc23394a668dce86a99e5b03a1a3d2ef",
    ),
}

for role, (relative, expected_sha) in c004_pins.items():
    if (
        len(expected_sha) != 64
        or any(ch not in "0123456789abcdef" for ch in expected_sha)
    ):
        raise RuntimeError(f"C004_INVALID_PIN:{role}")

# ============================================================
# 3. Strict read-only file verification
# ============================================================

def c004_read_pinned(relative, expected_sha):
    relative_path = Path(relative)
    path = (c004_root / relative_path).resolve()

    if (
        relative_path.is_absolute()
        or not path.is_relative_to(c004_root)
        or not path.is_file()
    ):
        raise RuntimeError(f"C004_FILE_MISSING_OR_UNSAFE:{relative}")

    before = path.stat()

    if before.st_size > 16 * 1024 * 1024:
        raise RuntimeError(f"C004_METADATA_TOO_LARGE:{relative}")

    raw = path.read_bytes()
    after = path.stat()

    if (
        before.st_size != after.st_size
        or before.st_size != len(raw)
        or before.st_mtime_ns != after.st_mtime_ns
    ):
        raise RuntimeError(f"C004_FILE_STABILITY_CONFLICT:{relative}")

    actual_sha = hashlib.sha256(raw).hexdigest()

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"C004_FILE_HASH_CONFLICT:{relative}:"
            f"expected={expected_sha}:actual={actual_sha}"
        )

    document = json.loads(raw)

    if not isinstance(document, dict):
        raise RuntimeError(f"C004_JSON_SCHEMA_CONFLICT:{relative}")

    return document, {
        "path": relative,
        "status": "FILE_BYTES_VERIFIED",
        "file_bytes_sha256": actual_sha,
        "size_bytes": len(raw),
    }

c004_docs = {}
c004_file_checks = {}

for role, (relative, expected_sha) in c004_pins.items():
    document, receipt = c004_read_pinned(
        relative,
        expected_sha,
    )
    c004_docs[role] = document
    c004_file_checks[role] = receipt

# ============================================================
# 4. Index boundary: metadata is not live storage validation
# ============================================================

c004_dense_precheck = c002_summary["dense_index_precheck"]

c004_expected_index_relative = (
    "data/phase2/index/dense_index_v1/qdrant"
)

if (
    c004_dense_precheck.get("path")
    != c004_expected_index_relative
    or c004_dense_precheck.get("frozen_index_hash_reverified")
    is not False
):
    raise RuntimeError("C004_DENSE_INDEX_BOUNDARY_CONFLICT")

c004_index_path = (
    c004_root / c004_expected_index_relative
).resolve()

c004_index_present = (
    c004_index_path.is_relative_to(c004_root)
    and c004_index_path.is_dir()
)

c004_index_status = (
    "PINNED_METADATA_VERIFIED_LIVE_INDEX_PENDING"
    if c004_index_present
    else "PINNED_METADATA_VERIFIED_STORAGE_MISSING"
)

c004_index_observed_fields = {
    role: {
        "status": doc.get("status"),
        "final_status": doc.get("final_status"),
        "logical_index_sha256": doc.get("logical_index_sha256"),
    }
    for role, doc in c004_docs.items()
}

# Do not assume that all documents use the same schema.
# Historical release semantics remain bound to their original evidence.
# This cell verifies file bytes, not live Qdrant integrity.

# ============================================================
# 5. Verify exact ONNX model from the C342 cache precheck
# ============================================================

c004_cache = c002_summary["model_cache_precheck"]

c004_expected_model_sha = (
    "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"
)

if (
    not isinstance(c004_cache, dict)
    or c004_cache.get("expected_onnx_sha256")
    != c004_expected_model_sha
    or not isinstance(c004_cache.get("path"), str)
    or not c004_cache["path"]
):
    raise RuntimeError("C004_MODEL_CONTRACT_CONFLICT")

c004_model_path = Path(
    c004_cache["path"]
).expanduser()

c004_model_actual_sha = None
c004_model_status = "MODEL_FILE_MISSING"

if c004_model_path.is_file():
    before = c004_model_path.stat()
    digest = hashlib.sha256()

    with c004_model_path.open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)

    after = c004_model_path.stat()

    c004_model_actual_sha = digest.hexdigest()

    if (
        before.st_size != after.st_size
        or before.st_mtime_ns != after.st_mtime_ns
    ):
        c004_model_status = "MODEL_CHANGED_DURING_READ"

    elif c004_model_actual_sha == c004_expected_model_sha:
        c004_model_status = "ONNX_FILE_HASH_MATCHED"

    else:
        c004_model_status = "ONNX_FILE_HASH_MISMATCH"

# ============================================================
# 6. Lightweight tokenizer and runtime inspection
# ============================================================

c004_snapshot = c004_model_path.parent.parent

c004_tokenizer_files = {
    name: (c004_snapshot / name).is_file()
    for name in (
        "config.json",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "vocab.txt",
    )
}

try:
    c004_onnxruntime_version = importlib.metadata.version(
        "onnxruntime"
    )
except importlib.metadata.PackageNotFoundError:
    c004_onnxruntime_version = None

c004_expected_runtime_version = "1.16.3"

c004_runtime_version_matches = (
    c004_onnxruntime_version
    == c004_expected_runtime_version
)

c004_historical_parity_preserved = (
    c002_summary["producer_contract_checks"].get(
        "C119_parity_verified"
    ) is True
)

# ============================================================
# 7. Store read-only preflight result in the current kernel
# ============================================================

c004_model_ready_for_later_validation = (
    c004_model_status == "ONNX_FILE_HASH_MATCHED"
    and c004_runtime_version_matches
)

NB22E_V4_RUNTIME_PREFLIGHT = {
    "cell": "NB22E-v4:C004",
    "status": "LIMITED_READ_ONLY_PREFLIGHT_COMPLETE",
    "index_file_checks": c004_file_checks,
    "index_metadata_files_verified": len(c004_file_checks),
    "index_status": c004_index_status,
    "index_directory_present": c004_index_present,
    "index_observed_fields": c004_index_observed_fields,
    "index_live_storage_reverified": False,
    "index_opened": False,
    "model_file_status": c004_model_status,
    "model_file_sha256": c004_model_actual_sha,
    "tokenizer_file_presence": c004_tokenizer_files,
    "tokenizer_runtime_loadability_reverified": False,
    "onnxruntime_version": c004_onnxruntime_version,
    "onnxruntime_version_matches": c004_runtime_version_matches,
    "historical_four_pair_parity_preserved": (
        c004_historical_parity_preserved
    ),
    "model_ready_for_later_validation": (
        c004_model_ready_for_later_validation
    ),
    "local_execution_ready": False,
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "model_runs": 0,
    "repository_modified": False,
}

print(json.dumps(
    NB22E_V4_RUNTIME_PREFLIGHT,
    ensure_ascii=False,
    indent=2,
))

{
  "cell": "NB22E-v4:C004",
  "status": "LIMITED_READ_ONLY_PREFLIGHT_COMPLETE",
  "index_file_checks": {
    "release": {
      "path": "artifacts/phase2/dense_index/dense_index_release_v1.json",
      "status": "FILE_BYTES_VERIFIED",
      "file_bytes_sha256": "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
      "size_bytes": 3592
    },
    "handoff": {
      "path": "artifacts/phase2/dense_index/nb16_dense_index_handoff_v1.json",
      "status": "FILE_BYTES_VERIFIED",
      "file_bytes_sha256": "da7f812a79a21b1876c74668a548badb0f271b2d18ac2f1b2cf9663f789f301f",
      "size_bytes": 1061
    },
    "manifest": {
      "path": "data/phase2/index/dense_index_v1/dense_index_manifest.json",
      "status": "FILE_BYTES_VERIFIED",
      "file_bytes_sha256": "739539f7abdfd0df435389e103658e28244b078f425f33878458034058bfae47",
      "size_bytes": 8493
    },
    "lifecycle": {
      "path": "data/phase2/index/dense_index_v1/index_lifecycle_evidence.json",
      "status": 

<div dir="rtl" align="right">

### NB22E-v4:C005 — R11.2: کنترل ایمن Tokenizer و Collection محلی

**هدف:** تکمیل کنترل بارگذاری Tokenizer و آزمایش خواندن Collection روی کپی موقت Index، بدون تغییر مخزن.

**ورودی‌ها:**
- خروجی‌های معتبر C002 تا C004.
- Snapshot مدل ONNX با هش تأییدشده.
- مسیر Qdrant و قراردادهای فریز‌شدهٔ Index.

**عملیات:**
۱. کنترل تطبیق تنظیمات Release و Handoff.
۲. بارگذاری Tokenizer فقط از Cache محلی، بدون دانلود.
۳. بررسی محدود اندازه و تعداد فایل‌های Index.
۴. در صورت کافی‌بودن منابع، تهیهٔ کپی موقت بیرون مخزن.
۵. بازکردن فقط کپی Qdrant و کنترل Collection و تعداد Pointها.
۶. بستن Qdrant و حذف خودکار کپی موقت.

**خروجی:** گزارش وضعیت Tokenizer، قابلیت خواندن Collection و موانع باقی‌مانده.

**محدودیت‌ها:**
- از اصل Qdrant هیچ Client ساخته نمی‌شود.
- حداکثر حجم کپی: ۵۱۲ مگابایت.
- هیچ Retrieval، رتبه‌بندی یا استنتاج مدل اجرا نمی‌شود.
- کنترل Collection به‌تنهایی معادل تأیید کامل Logical Index Hash نیست.
- هیچ نسخهٔ جدیدی تصویب نمی‌شود.

**API:** صفر.

**تغییر مخزن:** صفر.

**شرط پایان:** ثبت وضعیت واقعی هر دو کنترل و تعیین امکان طراحی Pilot محدود، بدون تغییر Gate رسمی.

</div>

In [5]:
# NB22E-v4 | C005 | R11.2
# Local tokenizer and isolated Qdrant-clone verification; no retrieval, inference or repository writes.

# ============================================================
# 1. Validate the successful prior checkpoints
# ============================================================

if not (
    NB22E_V4_READ_ONLY is True
    and NB22E_V4_TRANSFER.get("status") == "TRANSFER_LOADER_PASS"
    and NB22E_V4_BOUNDARY_PROPOSAL.get("status") == "DRAFT_RELOAD_PASS"
    and NB22E_V4_RUNTIME_PREFLIGHT.get("status")
    == "LIMITED_READ_ONLY_PREFLIGHT_COMPLETE"
    and NB22E_V4_RUNTIME_PREFLIGHT.get("index_metadata_files_verified") == 4
    and NB22E_V4_RUNTIME_PREFLIGHT.get("model_file_status")
    == "ONNX_FILE_HASH_MATCHED"
    and NB22E_V4_RUNTIME_PREFLIGHT.get("onnxruntime_version_matches") is True
):
    raise RuntimeError("C005_REQUIRES_SUCCESSFUL_C002_C003_C004")

c005_root = c002_root.resolve()
c005_out = c003_out.resolve()
c005_source = c004_index_path.resolve()

if (
    c005_out.is_relative_to(c005_root)
    or not c005_source.is_relative_to(c005_root)
    or not c005_source.is_dir()
):
    raise RuntimeError("C005_UNSAFE_REPOSITORY_BOUNDARY")

# ============================================================
# 2. Bind the previously verified release and handoff
# ============================================================

c005_release = c004_docs["release"]
c005_handoff = c004_docs["handoff"]

c005_backend = c005_release.get("backend")

if not isinstance(c005_backend, dict):
    raise RuntimeError("C005_RELEASE_BACKEND_SCHEMA")

c005_collection = c005_backend.get("collection_name")
c005_storage_relative = c005_backend.get("storage_path")

c005_logical_sha = (
    "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4"
)

if not (
    isinstance(c005_collection, str)
    and bool(c005_collection)
    and c005_collection == c005_handoff.get("collection_name")
    and c005_storage_relative == c005_handoff.get("storage_path")
    and c005_storage_relative == c004_expected_index_relative
    and c005_release.get("logical_index_sha256") == c005_logical_sha
    and c005_handoff.get("logical_index_sha256") == c005_logical_sha
    and c005_release.get("status") == "FROZEN_FOR_DOWNSTREAM_RETRIEVAL"
    and c005_release.get("final_status") == "PASS_COMPLETE_FROZEN"
    and c005_handoff.get("status") == "READY_FOR_NB16"
):
    raise RuntimeError("C005_INDEX_RELEASE_HANDOFF_CONFLICT")

# ============================================================
# 3. Load the exact local tokenizer without loading the model
# ============================================================

c005_tokenizer_result = {
    "status": "NOT_ATTEMPTED",
    "snapshot": str(c004_snapshot),
    "local_files_only": True,
    "trust_remote_code": False,
    "model_loaded": False,
    "inference_calls": 0,
}

try:
    c005_auto_tokenizer = nb22e_v4_require_imports(
        "AutoTokenizer"
    )["AutoTokenizer"]

    c005_tokenizer = c005_auto_tokenizer.from_pretrained(
        str(c004_snapshot),
        local_files_only=True,
        trust_remote_code=False,
    )

    # Structural smoke input, not a Test-dataset evaluation.
    c005_encoded = c005_tokenizer(
        "Local runtime check",
        "Local context check",
        truncation=True,
        max_length=512,
        return_attention_mask=True,
    )

    c005_input_ids = c005_encoded.get("input_ids")
    c005_attention = c005_encoded.get("attention_mask")

    if not (
        isinstance(c005_input_ids, list)
        and isinstance(c005_attention, list)
        and len(c005_input_ids) > 0
        and len(c005_input_ids) == len(c005_attention)
        and len(c005_input_ids) <= 512
    ):
        raise RuntimeError("C005_TOKENIZER_ENCODING_CONFLICT")

    c005_tokenizer_result.update({
        "status": "LOCAL_TOKENIZER_LOAD_AND_ENCODE_PASS",
        "tokenizer_class": type(c005_tokenizer).__name__,
        "is_fast": bool(c005_tokenizer.is_fast),
        "smoke_token_count": len(c005_input_ids),
        "max_length_used": 512,
    })

except Exception as exc:
    c005_tokenizer_result.update({
        "status": "TOKENIZER_RUNTIME_CHECK_FAILED",
        "error_type": type(exc).__name__,
        "error": str(exc)[:500],
    })

# ============================================================
# 4. Inventory only the pinned Qdrant directory
# ============================================================

c005_max_copy_bytes = 512 * 1024 * 1024
c005_max_files = 4000

c005_files = []
c005_total_bytes = 0
c005_inventory_error = None

try:
    for directory, dirs, files in os.walk(
        c005_source,
        topdown=True,
        followlinks=False,
    ):
        directory_path = Path(directory)

        for name in dirs + files:
            entry = directory_path / name

            if entry.is_symlink():
                raise RuntimeError("INDEX_CONTAINS_SYMLINK")

        for name in files:
            source_file = directory_path / name

            if not source_file.is_file():
                raise RuntimeError("INDEX_CONTAINS_NONREGULAR_FILE")

            stat = source_file.stat()
            relative = source_file.relative_to(c005_source)

            c005_files.append({
                "relative": relative,
                "size": stat.st_size,
                "mtime_ns": stat.st_mtime_ns,
            })

            c005_total_bytes += stat.st_size

            if len(c005_files) > c005_max_files:
                raise RuntimeError("INDEX_FILE_COUNT_EXCEEDS_LIMIT")

            if c005_total_bytes > c005_max_copy_bytes:
                raise RuntimeError("INDEX_SIZE_EXCEEDS_COPY_LIMIT")

except Exception as exc:
    c005_inventory_error = {
        "error_type": type(exc).__name__,
        "error": str(exc)[:500],
    }

c005_index_result = {
    "status": "NOT_OPENED",
    "original_index_opened": False,
    "original_index_modified": False,
    "copy_location": "TEMPORARY_OUTSIDE_REPOSITORY",
    "copy_limit_bytes": c005_max_copy_bytes,
    "inventory_files": len(c005_files),
    "inventory_bytes": c005_total_bytes,
    "inventory_error": c005_inventory_error,
    "collection_name": c005_collection,
    "collection_readable": False,
    "exact_count": None,
    "reported_points_count": None,
    "logical_index_hash_reverified": False,
    "qdrant_queries": 0,
    "temporary_copy_removed": True,
}

# ============================================================
# 5. Check disk capacity before creating any temporary copy
# ============================================================

c005_copy_allowed = (
    c005_inventory_error is None
    and len(c005_files) > 0
    and c005_total_bytes > 0
)

if c005_copy_allowed:
    c005_fs = os.statvfs(str(c005_out))
    c005_free_bytes = c005_fs.f_bavail * c005_fs.f_frsize

    c005_required_free = max(
        c005_total_bytes * 2,
        256 * 1024 * 1024,
    )

    c005_index_result["free_bytes_before_copy"] = c005_free_bytes
    c005_index_result["required_free_bytes"] = c005_required_free

    if c005_free_bytes < c005_required_free:
        c005_copy_allowed = False
        c005_index_result["status"] = "INSUFFICIENT_SPACE_FOR_SAFE_COPY"

elif c005_inventory_error is not None:
    c005_index_result["status"] = "INDEX_INVENTORY_BLOCKED"

else:
    c005_index_result["status"] = "INDEX_DIRECTORY_EMPTY"

# ============================================================
# 6. Verify Collection using an isolated temporary working copy
# ============================================================

if c005_copy_allowed:
    c005_client = None

    try:
        with tempfile.TemporaryDirectory(
            prefix="NB22E-v4-C005-qdrant-",
            dir=str(c005_out),
        ) as temporary_root:

            c005_working = Path(temporary_root) / "qdrant"
            c005_working.mkdir()

            print(
                "C005 isolated index copy:",
                len(c005_files),
                "files,",
                c005_total_bytes,
                "bytes"
            )

            # Copy source files only; no Qdrant client touches the source.
            for item in c005_files:
                source_file = c005_source / item["relative"]
                target_file = c005_working / item["relative"]

                stat_before = source_file.stat()

                if (
                    stat_before.st_size != item["size"]
                    or stat_before.st_mtime_ns != item["mtime_ns"]
                    or source_file.is_symlink()
                ):
                    raise RuntimeError(
                        "INDEX_CHANGED_BEFORE_COPY:"
                        + item["relative"].as_posix()
                    )

                target_file.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                copied_bytes = 0
                source_digest = hashlib.sha256()

                with (
                    source_file.open("rb") as source_handle,
                    target_file.open("xb") as target_handle,
                ):
                    while True:
                        block = source_handle.read(1024 * 1024)

                        if not block:
                            break

                        source_digest.update(block)
                        target_handle.write(block)
                        copied_bytes += len(block)

                stat_after = source_file.stat()

                if (
                    copied_bytes != item["size"]
                    or stat_after.st_size != item["size"]
                    or stat_after.st_mtime_ns != item["mtime_ns"]
                    or source_digest.hexdigest()
                    != hashlib.file_digest(
                        target_file.open("rb"),
                        "sha256",
                    ).hexdigest()
                ):
                    raise RuntimeError(
                        "INDEX_COPY_INTEGRITY_CONFLICT:"
                        + item["relative"].as_posix()
                    )

            # The only opened Qdrant storage is the isolated copy.
            c005_qdrant_class = nb22e_v4_require_imports(
                "QdrantClient"
            )["QdrantClient"]

            try:
                c005_client = c005_qdrant_class(
                    path=str(c005_working)
                )

                c005_info = c005_client.get_collection(
                    collection_name=c005_collection
                )

                c005_count = c005_client.count(
                    collection_name=c005_collection,
                    exact=True,
                )

                c005_exact_count = int(c005_count.count)
                c005_reported_count = c005_info.points_count

                c005_count_consistent = (
                    c005_reported_count is None
                    or int(c005_reported_count) == c005_exact_count
                )

                if not c005_count_consistent:
                    raise RuntimeError(
                        "QDRANT_REPORTED_AND_EXACT_COUNT_DIFFER"
                    )

                c005_index_result.update({
                    "status": "ISOLATED_COLLECTION_READ_PASS",
                    "collection_readable": True,
                    "exact_count": c005_exact_count,
                    "reported_points_count": c005_reported_count,
                    "collection_status": str(c005_info.status),
                    "snapshot_files_copied": len(c005_files),
                    "snapshot_bytes_copied": c005_total_bytes,
                    "logical_index_hash_reverified": False,
                })

            finally:
                if c005_client is not None:
                    c005_client.close()
                    c005_client = None

    except Exception as exc:
        c005_index_result.update({
            "status": "ISOLATED_INDEX_CHECK_FAILED",
            "error_type": type(exc).__name__,
            "error": str(exc)[:700],
            "logical_index_hash_reverified": False,
        })

# ============================================================
# 7. Report the exact remaining readiness boundaries
# ============================================================

c005_tokenizer_pass = (
    c005_tokenizer_result["status"]
    == "LOCAL_TOKENIZER_LOAD_AND_ENCODE_PASS"
)

c005_clone_pass = (
    c005_index_result["status"]
    == "ISOLATED_COLLECTION_READ_PASS"
)

NB22E_V4_ISOLATED_RUNTIME_CHECK = {
    "cell": "NB22E-v4:C005",
    "status": (
        "LIMITED_RUNTIME_CHECK_PASS_WITH_VERSION_PENDING"
        if c005_tokenizer_pass and c005_clone_pass
        else "LIMITED_RUNTIME_CHECK_HAS_OPEN_ITEMS"
    ),
    "tokenizer": c005_tokenizer_result,
    "index": c005_index_result,
    "frozen_model_file_hash_verified_in_C004": True,
    "historical_parity_preserved": True,
    "new_population_version_approved": False,
    "live_original_index_opened": False,
    "logical_index_hash_reverified": False,
    "local_pilot_authorized": False,
    "full_population_execution_authorized": False,
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "model_inference_calls": 0,
    "repository_modified": False,
    "next": (
        "VERSION_EQUIVALENCE_AND_BOUNDED_PILOT_DESIGN"
        if c005_tokenizer_pass and c005_clone_pass
        else "REPAIR_ONLY_REPORTED_RUNTIME_BLOCKERS"
    ),
}

print(json.dumps(
    NB22E_V4_ISOLATED_RUNTIME_CHECK,
    ensure_ascii=False,
    indent=2,
))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C005 isolated index copy: 3 files, 46338588 bytes
{
  "cell": "NB22E-v4:C005",
  "status": "LIMITED_RUNTIME_CHECK_PASS_WITH_VERSION_PENDING",
  "tokenizer": {
    "status": "LOCAL_TOKENIZER_LOAD_AND_ENCODE_PASS",
    "snapshot": "/Users/macbookpro/.cache/huggingface/hub/models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1/snapshots/1427fd652930e4ba29e8149678df786c240d8825",
    "local_files_only": true,
    "trust_remote_code": false,
    "model_loaded": false,
    "inference_calls": 0,
    "tokenizer_class": "XLMRobertaTokenizerFast",
    "is_fast": true,
    "smoke_token_count": 11,
    "max_length_used": 512
  },
  "index": {
    "status": "ISOLATED_COLLECTION_READ_PASS",
    "original_index_opened": false,
    "original_index_modified": false,
    "copy_location": "TEMPORARY_OUTSIDE_REPOSITORY",
    "copy_limit_bytes": 536870912,
    "inventory_files": 3,
    "inventory_bytes": 46338588,
    "inventory_error": null,
    "collection_name": "rag_finance_phase2_dense_v1",
    "collecti

<div dir="rtl" align="right">

### NB22E-v4:C006 — R11.2: تثبیت طرح Pilot محدود و مرز نسخه

**نتیجهٔ قبلی:** در C005، بارگذاری Tokenizer و خواندن Collection روی کپی موقت Qdrant موفق بود. تعداد ۳۴۸۸ Point تأیید شد. اعتبار Logical Index Hash و نسخهٔ جمعیتی هنوز نهایی نشده‌اند.

**هدف:** ساخت بستهٔ اجرایی نسخه‌دار برای یک Pilot محدود، بدون شروع محاسبات یا استفاده از Test برای تنظیم الگوریتم.

**ورودی‌ها:**
- اتصال ۱۱۱۶ سؤال از C002.
- پیش‌نویس مرز نسخه از C003.
- قراردادها و هش مدل از C004.
- شواهد کنترل محلی C005.
- تنظیمات فریز‌شدهٔ C118، C119 و مسیر Producer ثبت‌شده در C342.

**عملیات:**

۱. انتخاب مکانیکی چهار شناسهٔ آزمایشی، مستقل از متن پاسخ، امتیاز و معیارهای کیفیت.

۲. تعیین دو مرحلهٔ مشروط Pilot:
- بررسی تطبیق Hybrid100 با نسخهٔ فریز‌شده.
- بررسی حداکثر ۲۰۰ جفت اضافی Local100، فقط پس از احراز شرایط مرحلهٔ اول.

۳. حفظ تنظیمات تاریخی:
- ONNX Runtime روی CPU
- چهار Thread محاسباتی
- Batch Size برابر ۴
- حداکثر طول توالی ۵۱۲

۴. تعیین محدودیت حافظه، مسیر Checkpoint، ثبت خطا و شرایط توقف.

۵. ثبت وابستگی به تأیید مرز نسخه و مجوز Pilot.

۶. نوشتن و Reload یک Artifact برنامه‌ریزی مستقل خارج از مخزن.

**موارد محفوظ:**
- ۲۹ اجرای دقیق تأییدشده.
- دو هویت نامحرز.
- دو خطای ثبت‌شده.
- نسخهٔ تاریخی و تمام فایل‌های فریز‌شده.

**خروجی:** پیش‌نویس `C006.pilot_boundary_plan` با هش معتبر.

**API و مدل:** هیچ اجرایی انجام نمی‌شود.

**تغییر مخزن:** صفر.

**شرط پایان:** ثبت طرح قابل‌بررسی و تعیین پیش‌نیازهای اجرای محدود؛ بدون اعطای هویت جدید یا تصویب Gate.

</div>

In [6]:
# NB22E-v4 | C006 | R11.2
# Prepare a bounded, version-linked pilot plan; no retrieval, inference, API or repository writes.

# ============================================================
# 1. Validate completed checkpoints
# ============================================================

def c006_require(condition, message):
    if not condition:
        raise RuntimeError("C006_" + message)

c006_require(
    NB22E_V4_READ_ONLY is True
    and NB22E_V4_TRANSFER["status"] == "TRANSFER_LOADER_PASS"
    and NB22E_V4_BOUNDARY_PROPOSAL["status"] == "DRAFT_RELOAD_PASS"
    and NB22E_V4_RUNTIME_PREFLIGHT["status"]
    == "LIMITED_READ_ONLY_PREFLIGHT_COMPLETE"
    and NB22E_V4_ISOLATED_RUNTIME_CHECK["status"]
    == "LIMITED_RUNTIME_CHECK_PASS_WITH_VERSION_PENDING",
    "PREVIOUS_CHECKPOINT_NOT_READY",
)

c006_require(
    len(c002_questions) == 1116
    and len(c002_remaining) == 0
    and c005_index_result["status"] == "ISOLATED_COLLECTION_READ_PASS"
    and c005_index_result["exact_count"] == 3488
    and c005_index_result["original_index_opened"] is False
    and c005_tokenizer_result["status"]
    == "LOCAL_TOKENIZER_LOAD_AND_ENCODE_PASS"
    and c004_model_status == "ONNX_FILE_HASH_MATCHED"
    and c004_runtime_version_matches is True,
    "RUNTIME_OR_POPULATION_CONFLICT",
)

c006_vector = c002_summary["query_vector_precheck"]

c006_require(
    c006_vector.get("vector_candidate_id_coverage") == 1116
    and c006_vector.get("cache_hashes_reverified") is True
    and all(
        value.get("status") == "FILE_BYTES_VERIFIED"
        for value in c006_vector["cache_file_checks"].values()
    ),
    "VECTOR_CACHE_RECEIPT_CONFLICT",
)

c006_require(
    all(
        value is True
        for value in c002_summary["producer_contract_checks"].values()
    ),
    "FROZEN_PRODUCER_CONTRACT_CONFLICT",
)

# ============================================================
# 2. Select four IDs without examining evaluation outcomes
# ============================================================

c006_population = sorted(c002_questions)

c006_selection_order = sorted(
    c006_population,
    key=lambda candidate_id: (
        hashlib.sha256(candidate_id.encode("utf-8")).hexdigest(),
        candidate_id,
    ),
)

c006_pilot_ids = c006_selection_order[:4]

c006_require(
    len(c006_pilot_ids) == len(set(c006_pilot_ids)) == 4,
    "PILOT_ID_SELECTION_CONFLICT",
)

c006_selected = [
    {
        "candidate_id": candidate_id,
        "question_sha256": c002_questions[candidate_id][
            "question_sha256"
        ],
    }
    for candidate_id in c006_pilot_ids
]

# Selection uses identifiers only, never scores or outcomes.

# ============================================================
# 3. Bind the verified frozen runtime and input lineage
# ============================================================

c006_expected_model = {
    "model_id": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "model_revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "onnx_sha256": (
        "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"
    ),
    "onnxruntime_version": "1.16.3",
    "batch_size": 4,
    "maximum_sequence_length": 512,
}

c006_require(
    c004_model_actual_sha == c006_expected_model["onnx_sha256"]
    and c004_onnxruntime_version
    == c006_expected_model["onnxruntime_version"]
    and c005_logical_sha
    == c004_docs["release"]["logical_index_sha256"],
    "FROZEN_RUNTIME_IDENTITY_CONFLICT",
)

c006_source_pins = {
    role: {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }
    for role, record in c002_summary["source_checks"].items()
}

c006_input_pins = {
    role: {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }
    for role, record in c002_summary["input_file_checks"].items()
}

# ============================================================
# 4. Freeze the planning boundary, not a target release
# ============================================================

c006_core = {
    "schema": "NB22E_V4_R11_2_BOUNDED_PILOT_PLAN_V1",
    "cell": "NB22E-v4:C006",
    "subtask": "R11.2",
    "status": "PILOT_PLAN_PREPARED_APPROVAL_PENDING",

    "population": {
        "total_test_questions": 1149,
        "new_candidate_count": 1116,
        "pilot_candidate_count": 4,
        "selection_method": (
            "FIRST_FOUR_SHA256_SORTED_CANDIDATE_IDS"
        ),
        "pilot_candidates": c006_selected,
        "selection_uses_test_scores": False,
        "selection_uses_test_answers": False,
        "question_binding_sha256": c002_specs["questions"]["sha"],
        "manifest_sha256": c002_specs["manifest"]["sha"],
    },

    "version_boundary": {
        "historical_target_identity": (
            c002_summary["target_identity_sha256_original"]
        ),
        "historical_rank_mode": (
            c002_summary["original_rank_input_mode"]
        ),
        "boundary_proposal_file_sha256": (
            NB22E_V4_BOUNDARY_PROPOSAL["file_bytes_sha256"]
        ),
        "boundary_proposal_content_sha256": (
            NB22E_V4_BOUNDARY_PROPOSAL["proposal_content_sha256"]
        ),
        "new_population_identity": "UNASSIGNED",
        "version_approval": "PENDING",
        "historical_hybrid100_equivalence": "NOT_YET_PROVEN",
        "pool50_version_membership": "NOT_YET_PROVEN",
        "new_rank_receipts": "NOT_YET_PRODUCED",
        "new_context_receipts": "NOT_YET_PRODUCED",
        "target_generation_authorized": False,
    },

    "verified_inputs": {
        "source_pins": c006_source_pins,
        "input_pins": c006_input_pins,
        "vector_cache_pointers": (
            c006_vector["cache_file_pointers"]
        ),
        "producer_contract_checks": (
            c002_summary["producer_contract_checks"]
        ),
    },

    "index": {
        "collection": c005_collection,
        "frozen_logical_index_sha256": c005_logical_sha,
        "verified_clone_point_count": 3488,
        "verified_clone_file_count": (
            c005_index_result["inventory_files"]
        ),
        "verified_clone_inventory_bytes": (
            c005_index_result["inventory_bytes"]
        ),
        "original_index_opened": False,
        "logical_index_hash_reverified": False,
        "pilot_index_policy": (
            "ISOLATED_WORKING_COPY_OUTSIDE_REPOSITORY"
        ),
    },

    "local_runtime": {
        **c006_expected_model,
        "provider": "CPUExecutionProvider",
        "intra_op_num_threads": 4,
        "inter_op_num_threads": 1,
        "execution_mode": "ORT_SEQUENTIAL",
        "candidate_parallelism": 1,
        "tokenizer_local_only": True,
        "model_download_allowed": False,
        "historical_four_pair_parity_preserved": True,
    },

    "pilot": {
        "stage_1": {
            "name": "VERIFY_FROZEN_HYBRID100_EQUIVALENCE",
            "candidate_count_max": 4,
            "retrieval_queries_max": 4,
            "historical_order_reuse_unconditional": False,
            "requires_version_approval": True,
            "status": "NOT_STARTED",
        },
        "stage_2": {
            "name": "LOCAL100_REMAINING_50_PAIR_CHECK",
            "candidate_count_max": 4,
            "additional_pairs_max": 200,
            "onnx_batch_size": 4,
            "requires_exact_hybrid100_membership": True,
            "requires_pool50_version_membership": True,
            "requires_stage_1_success": True,
            "status": "NOT_STARTED",
        },
        "after_pilot": {
            "rank_producer": "PINNED_NB22C_AND_NB22D",
            "context_boundary": "PINNED_C139_AND_PIPELINE",
            "configuration_tuning_on_test": False,
            "full_population_execution": "NOT_AUTHORIZED",
        },
    },

    "resource_controls": {
        "host_memory_gib_reported": 16,
        "proposed_pilot_rss_stop_threshold_gib": 6,
        "candidate_processing": "SEQUENTIAL",
        "checkpoint_frequency": "AFTER_EACH_COMPLETED_CANDIDATE",
        "checkpoint_write_policy": "ATOMIC_OUTSIDE_REPOSITORY",
        "unknown_outcomes_auto_retried": False,
        "pilot_runtime_estimate": "UNMEASURED",
        "pilot_peak_memory_estimate": "UNMEASURED",
        "full_population_runtime_estimate": "NOT_ESTIMATED",
        "pilot_time_and_memory_to_be_measured": True,
    },

    "approval_and_safety": {
        "version_boundary_approval": "PENDING_SUPERVISOR",
        "bounded_local_pilot_approval": "PENDING_SUPERVISOR",
        "execution_allowed_now": False,
        "generation_api_allowed": False,
        "existing_results_preserved": {
            "EXACT_PRESERVED": 29,
            "MODEL_IDENTITY_HELD": 2,
            "RECORDED_ERROR_HELD": 2,
            "OUTSIDE_PACKET": 1116,
        },
        "repository_read_only": True,
        "historical_artifacts_immutable": True,
        "target_release_gate": "BLOCKED",
    },
}

# ============================================================
# 5. Content-bound fingerprint
# ============================================================

c006_canonical_bytes = json.dumps(
    c006_core,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

c006_content_sha = hashlib.sha256(
    c006_canonical_bytes
).hexdigest()

c006_document = {
    **c006_core,
    "plan_content_sha256": c006_content_sha,
    "fingerprint_kind": "CANONICAL_JSON_PLAN_CONTENT",
}

c006_bytes = json.dumps(
    c006_document,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8") + b"\n"

c006_file_sha = hashlib.sha256(c006_bytes).hexdigest()

# ============================================================
# 6. Atomic, immutable output outside the repository
# ============================================================

c006_output_dir = c003_out.resolve()
c006_root = c002_root.resolve()

c006_require(
    c006_output_dir.is_dir()
    and not c006_output_dir.is_relative_to(c006_root),
    "OUTPUT_DIRECTORY_UNSAFE",
)

c006_target = c006_output_dir / (
    "C006.pilot_boundary_plan."
    + c006_file_sha
    + ".json"
)

c006_temp = None

if not c006_target.exists():
    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=c006_output_dir,
        prefix=".C006.pilot.",
        suffix=".tmp",
        delete=False,
    ) as handle:
        c006_temp = Path(handle.name)
        handle.write(c006_bytes)
        handle.flush()
        os.fsync(handle.fileno())

    try:
        c006_require(
            c006_temp.read_bytes() == c006_bytes,
            "TEMP_RELOAD_CONFLICT",
        )

        try:
            os.link(c006_temp, c006_target)
        except FileExistsError:
            pass
    finally:
        c006_temp.unlink(missing_ok=True)

# ============================================================
# 7. Independent output reload and report
# ============================================================

c006_reloaded_bytes = c006_target.read_bytes()

c006_require(
    c006_reloaded_bytes == c006_bytes
    and hashlib.sha256(c006_reloaded_bytes).hexdigest()
    == c006_file_sha,
    "FINAL_FILE_BYTES_CONFLICT",
)

c006_reloaded = json.loads(c006_reloaded_bytes)

c006_reloaded_core = {
    key: value
    for key, value in c006_reloaded.items()
    if key not in ("plan_content_sha256", "fingerprint_kind")
}

c006_reloaded_core_bytes = json.dumps(
    c006_reloaded_core,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8")

c006_require(
    hashlib.sha256(c006_reloaded_core_bytes).hexdigest()
    == c006_reloaded["plan_content_sha256"]
    == c006_content_sha
    and c006_reloaded["pilot"]["stage_1"]["status"]
    == "NOT_STARTED"
    and c006_reloaded["pilot"]["stage_2"]["status"]
    == "NOT_STARTED"
    and c006_reloaded["approval_and_safety"]["execution_allowed_now"]
    is False,
    "PLAN_RELOAD_VALIDATION_CONFLICT",
)

NB22E_V4_PILOT_PLAN = {
    "status": "PILOT_PLAN_RELOAD_PASS",
    "path": str(c006_target),
    "file_bytes_sha256": c006_file_sha,
    "plan_content_sha256": c006_content_sha,
    "candidate_ids": c006_pilot_ids,
    "pilot_authorized": False,
}

print(json.dumps({
    "cell": "NB22E-v4:C006",
    "status": "PILOT_PLAN_RELOAD_PASS",
    "population": 1116,
    "pilot_candidate_count": 4,
    "pilot_candidate_ids": c006_pilot_ids,
    "hybrid100_queries_max_if_approved": 4,
    "local100_additional_pairs_max_if_approved": 200,
    "onnx_batch_size": 4,
    "onnx_max_sequence_length": 512,
    "onnx_intra_op_threads": 4,
    "candidate_parallelism": 1,
    "model_file_verified": True,
    "tokenizer_verified": True,
    "isolated_qdrant_collection_verified": True,
    "logical_index_hash_reverified": False,
    "version_approval": "PENDING",
    "pilot_execution_authorized": False,
    "output_receipt": {
        "path": str(c006_target),
        "file_bytes_sha256": c006_file_sha,
        "size_bytes": len(c006_bytes),
    },
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "retrieval_runs": 0,
    "rank_runs": 0,
    "model_inference_calls": 0,
    "repository_writes_by_cell": 0,
    "next": "SUPERVISOR_VERSION_AND_BOUNDED_PILOT_DECISION",
}, ensure_ascii=False, indent=2))

{
  "cell": "NB22E-v4:C006",
  "status": "PILOT_PLAN_RELOAD_PASS",
  "population": 1116,
  "pilot_candidate_count": 4,
  "pilot_candidate_ids": [
    "golden_v2_candidate_ca5362d68c692d667fad3006",
    "golden_v2_candidate_15cb48c40c3d1e292f647ff9",
    "golden_v2_candidate_4032db31c53d0465590cf0ab",
    "golden_v2_candidate_ba8ca208df074fc579e48eb5"
  ],
  "hybrid100_queries_max_if_approved": 4,
  "local100_additional_pairs_max_if_approved": 200,
  "onnx_batch_size": 4,
  "onnx_max_sequence_length": 512,
  "onnx_intra_op_threads": 4,
  "candidate_parallelism": 1,
  "model_file_verified": true,
  "tokenizer_verified": true,
  "isolated_qdrant_collection_verified": true,
  "logical_index_hash_reverified": false,
  "version_approval": "PENDING",
  "pilot_execution_authorized": false,
  "output_receipt": {
    "path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/_RAG_finance_NB22E_v4_outputs/C006.pilot_boundary_plan.fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d5

<div dir="rtl" align="right">

### NB22E-v4:C007 — R11.2: تطبیق Logical Index Hash با قرارداد فریز‌شده

**هدف:** احراز همسانی واقعی محتوای Index با نسخهٔ فریز‌شده، پیش از هر Query آزمایشی.

**قرارداد مرجع:** `NB14 / Code Cell 10`

**روش محاسبه:**
- ترتیب Pointها از ترتیب اصلی رکوردهای فایل Chunk گرفته می‌شود.
- شناسهٔ هر Point با UUIDv5 و Namespace فریز‌شده ساخته می‌شود.
- برای هر Point، شناسهٔ UTF-8، کل Payload به‌شکل JSON کانونی و بردار little-endian float32 مصرف می‌شوند.
- پیش از هر جزء، طول آن به‌صورت unsigned big-endian هشت‌بایتی به SHA-256 افزوده می‌شود.
- نتیجه با هش فریز‌شدهٔ Release، Handoff و Manifest مقایسه می‌شود.

**ایمنی:** Qdrant اصلی باز نمی‌شود. محاسبه روی کپی موقت خارج مخزن، با Batch برابر ۱۲۸ Point انجام می‌شود.

**خروجی:** Checkpoint مستقل با وضعیت `LOGICAL_INDEX_HASH_MATCH` یا `BLOCKED`.

**شرط ادامه:** فقط تطبیق دقیق هش و Reload موفق Checkpoint اجازهٔ ورود به C008 می‌دهد.

**API، مدل و Retrieval:** صفر.

</div>

In [7]:
# NB22E-v4 | C007 | R11.2
# Verify frozen NB14 logical Index hash on an isolated Qdrant copy.

if not (
    NB22E_V4_READ_ONLY is True
    and NB22E_V4_PILOT_PLAN["status"] == "PILOT_PLAN_RELOAD_PASS"
    and NB22E_V4_RUNTIME_PREFLIGHT["index_metadata_files_verified"] == 4
    and NB22E_V4_ISOLATED_RUNTIME_CHECK["index"]["status"]
    == "ISOLATED_COLLECTION_READ_PASS"
):
    raise RuntimeError("C007_PRIOR_GATE_NOT_PASSED")

c007_plan_sha = (
    "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213"
)

c007_plan_path = Path(NB22E_V4_PILOT_PLAN["path"]).resolve()

if not (
    c007_plan_path.parent == c003_out.resolve()
    and NB22E_V4_PILOT_PLAN["file_bytes_sha256"] == c007_plan_sha
    and hashlib.sha256(c007_plan_path.read_bytes()).hexdigest()
    == c007_plan_sha
):
    raise RuntimeError("C007_SUPERVISOR_APPROVED_PLAN_HASH_CONFLICT")

c007_root = c002_root.resolve()
c007_out = c003_out.resolve()
c007_original = c004_index_path.resolve()

c007_release = c004_docs["release"]
c007_handoff = c004_docs["handoff"]
c007_manifest = c004_docs["manifest"]

c007_expected = (
    "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4"
)

c007_chunks_sha = c007_release["inputs"]["chunk_sha256"]

c007_chunks = (
    c007_root / c007_release["inputs"]["chunk_path"]
).resolve()

c007_source_nb14 = (
    c007_root / "notebooks/phase2/14_dense_index.ipynb"
).resolve()

if not (
    c007_out.is_dir()
    and not c007_out.is_relative_to(c007_root)
    and c007_original.is_relative_to(c007_root)
    and c007_original.is_dir()
    and c007_chunks.is_relative_to(c007_root)
    and c007_chunks.is_file()
    and c007_source_nb14.is_file()
    and c007_source_nb14.is_relative_to(c007_root)
    and c007_release["logical_index_sha256"] == c007_expected
    and c007_handoff["logical_index_sha256"] == c007_expected
    and c007_manifest["index_content"]["logical_index_sha256"]
    == c007_expected
    and c007_release["point_count"] == 3488
    and c007_manifest["index_content"]["point_count"] == 3488
    and c007_chunks_sha
    == c002_summary["input_file_checks"]["CHUNKS"]["file_bytes_sha256"]
):
    raise RuntimeError("C007_FROZEN_REFERENCE_OR_PATH_CONFLICT")

# Verify the original NB14 producer Git blob.
c007_nb14_bytes = c007_source_nb14.read_bytes()

c007_nb14_blob = hashlib.sha1(
    b"blob "
    + str(len(c007_nb14_bytes)).encode("ascii")
    + b"\0"
    + c007_nb14_bytes
).hexdigest()

c007_nb14_expected_blob = (
    "1f8a8a82f2f1fbadc8e05977d8b129572381f2bf"
)

if c007_nb14_blob != c007_nb14_expected_blob:
    raise RuntimeError("C007_NB14_FROZEN_PRODUCER_GIT_BLOB_CONFLICT")

c007_started = time.perf_counter()

c007_result = {
    "cell": "NB22E-v4:C007",
    "status": "BLOCKED",
    "producer": "NB14_CODE_CELL_10",
    "producer_git_blob_sha1": c007_nb14_blob,
    "expected_logical_index_sha256": c007_expected,
    "observed_logical_index_sha256": None,
    "collection": c007_release["backend"]["collection_name"],
    "point_count": 0,
    "batches": 0,
    "original_qdrant_opened": False,
    "repository_writes": 0,
    "retrieval_queries": 0,
    "model_inference_calls": 0,
    "api_calls": 0,
    "pilot_started": False,
    "version_identity_final": False,
}

try:
    # Preserve exact source order; do not sort chunk IDs.
    c007_ids = []
    c007_chunk_ids = []
    c007_chunk_digest = hashlib.sha256()

    c007_namespace = uuid.UUID(
        c007_manifest["point_identity"]["namespace"]
    )

    if not (
        c007_namespace.hex == "4dd4798997bb5e13bc0ffae8fa884248"
        and c007_manifest["point_identity"]["algorithm"] == "UUIDv5"
    ):
        raise RuntimeError("POINT_ID_CONTRACT_MISMATCH")

    with c007_chunks.open("rb") as handle:
        for line_number, line in enumerate(handle, 1):
            if (
                line_number > 3488
                or not line.strip()
                or len(line) > 2 * 1024 * 1024
            ):
                raise RuntimeError("CHUNK_JSONL_BOUNDS_OR_FORMAT")

            c007_chunk_digest.update(line)
            chunk_id = json.loads(line)["chunk_id"]

            if not isinstance(chunk_id, str) or not chunk_id:
                raise RuntimeError("INVALID_CHUNK_ID")

            c007_chunk_ids.append(chunk_id)
            c007_ids.append(
                str(uuid.uuid5(c007_namespace, chunk_id))
            )

    if not (
        len(c007_ids) == len(set(c007_ids)) == 3488
        and c007_chunk_digest.hexdigest() == c007_chunks_sha
    ):
        raise RuntimeError("CHUNK_ORDER_COUNT_OR_SHA256_CONFLICT")

    # Bounded inventory of the existing Index files.
    c007_files = []
    c007_total_bytes = 0

    for directory, dirs, files in os.walk(
        c007_original,
        followlinks=False,
    ):
        if any(
            (Path(directory) / name).is_symlink()
            for name in dirs + files
        ):
            raise RuntimeError("ORIGINAL_INDEX_HAS_SYMLINK")

        for filename in files:
            source = Path(directory) / filename

            if not source.is_file():
                raise RuntimeError("ORIGINAL_INDEX_HAS_NONREGULAR_FILE")

            stat = source.stat()

            c007_files.append((
                source.relative_to(c007_original),
                stat.st_size,
                stat.st_mtime_ns,
            ))

            c007_total_bytes += stat.st_size

            if (
                len(c007_files) > 4000
                or c007_total_bytes > 512 * 1024 * 1024
            ):
                raise RuntimeError("INDEX_COPY_SIZE_LIMIT")

    if not (
        len(c007_files) == c005_index_result["inventory_files"]
        and c007_total_bytes == c005_index_result["inventory_bytes"]
    ):
        raise RuntimeError("INDEX_INVENTORY_CHANGED_SINCE_C005")

    c007_fs = os.statvfs(str(c007_out))
    c007_free = c007_fs.f_bavail * c007_fs.f_frsize

    if c007_free < max(
        256 * 1024 * 1024,
        c007_total_bytes * 2,
    ):
        raise RuntimeError("INSUFFICIENT_FREE_DISK")

    c007_np = nb22e_v4_require_imports("np")["np"]
    c007_qdrant_class = nb22e_v4_require_imports(
        "QdrantClient"
    )["QdrantClient"]

    with tempfile.TemporaryDirectory(
        prefix="NB22E-v4-C007-index-",
        dir=str(c007_out),
    ) as working_directory:

        clone = Path(working_directory) / "qdrant"
        clone.mkdir()

        # Copy and verify bytes; never instantiate Qdrant on original.
        for relative, size, modified in c007_files:
            source = c007_original / relative
            target = clone / relative

            before = source.stat()

            if (
                source.is_symlink()
                or before.st_size != size
                or before.st_mtime_ns != modified
            ):
                raise RuntimeError(
                    "ORIGINAL_CHANGED_BEFORE_COPY:" + str(relative)
                )

            target.parent.mkdir(parents=True, exist_ok=True)
            source_digest = hashlib.sha256()

            with (
                source.open("rb") as source_handle,
                target.open("xb") as target_handle,
            ):
                for block in iter(
                    lambda: source_handle.read(1024 * 1024),
                    b"",
                ):
                    source_digest.update(block)
                    target_handle.write(block)

            after = source.stat()

            if (
                after.st_size != size
                or after.st_mtime_ns != modified
            ):
                raise RuntimeError(
                    "ORIGINAL_CHANGED_DURING_COPY:" + str(relative)
                )

            target_digest = hashlib.sha256()

            with target.open("rb") as verify_handle:
                for block in iter(
                    lambda: verify_handle.read(1024 * 1024),
                    b"",
                ):
                    target_digest.update(block)

            if target_digest.digest() != source_digest.digest():
                raise RuntimeError(
                    "INDEX_COPY_BYTES_MISMATCH:" + str(relative)
                )

        client = None

        try:
            client = c007_qdrant_class(path=str(clone))
            collection = c007_result["collection"]

            if client.count(
                collection_name=collection,
                exact=True,
            ).count != 3488:
                raise RuntimeError("QDRANT_COUNT_MISMATCH")

            config = client.get_collection(
                collection_name=collection
            ).config.params.vectors

            if not (
                getattr(config, "size", None) == 384
                and str(
                    getattr(config, "distance", "")
                ).upper().split(".")[-1] == "COSINE"
            ):
                raise RuntimeError("QDRANT_VECTOR_CONFIG_MISMATCH")

            logical_digest = hashlib.sha256()

            # NB14 frozen algorithm: batches of 128, original chunk order.
            for start in range(0, 3488, 128):
                batch_ids = c007_ids[start:start + 128]

                points = client.retrieve(
                    collection_name=collection,
                    ids=batch_ids,
                    with_payload=True,
                    with_vectors=True,
                )

                by_id = {
                    str(point.id): point
                    for point in points
                }

                if (
                    len(by_id) != len(batch_ids)
                    or set(by_id) != set(batch_ids)
                ):
                    raise RuntimeError("QDRANT_POINT_ID_PARITY_CONFLICT")

                for point_id in batch_ids:
                    point = by_id[point_id]

                    if not isinstance(point.payload, dict):
                        raise RuntimeError("MISSING_POINT_PAYLOAD")

                    payload_bytes = json.dumps(
                        point.payload,
                        ensure_ascii=False,
                        sort_keys=True,
                        separators=(",", ":"),
                    ).encode("utf-8")

                    vector = c007_np.asarray(
                        point.vector,
                        dtype="<f4",
                    )

                    if not (
                        vector.shape == (384,)
                        and bool(c007_np.isfinite(vector).all())
                    ):
                        raise RuntimeError("INVALID_STORED_VECTOR")

                    vector_bytes = vector.tobytes(order="C")

                    for component in (
                        point_id.encode("utf-8"),
                        payload_bytes,
                        vector_bytes,
                    ):
                        logical_digest.update(
                            len(component).to_bytes(
                                8,
                                byteorder="big",
                                signed=False,
                            )
                        )
                        logical_digest.update(component)

                    c007_result["point_count"] += 1

                c007_result["batches"] += 1

                if (
                    c007_result["batches"] % 7 == 0
                    or c007_result["point_count"] == 3488
                ):
                    print(
                        "C007 logical hash progress:",
                        c007_result["point_count"],
                        "/ 3488",
                        flush=True,
                    )

            c007_result["observed_logical_index_sha256"] = (
                logical_digest.hexdigest()
            )

            if (
                c007_result["observed_logical_index_sha256"]
                != c007_expected
            ):
                raise RuntimeError(
                    "FROZEN_LOGICAL_INDEX_SHA256_MISMATCH"
                )

            c007_result["status"] = "LOGICAL_INDEX_HASH_MATCH"

        finally:
            if client is not None:
                client.close()

except Exception as error:
    c007_result["status"] = "BLOCKED"
    c007_result["blocker"] = {
        "type": type(error).__name__,
        "detail": str(error)[:600],
    }

c007_result["elapsed_seconds"] = round(
    time.perf_counter() - c007_started,
    3,
)

c007_result["pilot_stage_1_authorized_if_pass"] = (
    c007_result["status"] == "LOGICAL_INDEX_HASH_MATCH"
)

c007_result["pilot_stage_2_authorized"] = False
c007_result["target_release_gate"] = "BLOCKED"

# Immutable external checkpoint.
c007_bytes = json.dumps(
    c007_result,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode("utf-8") + b"\n"

c007_sha = hashlib.sha256(c007_bytes).hexdigest()

c007_target = c007_out / (
    "C007.logical_index_check."
    + c007_sha
    + ".json"
)

if not c007_target.is_file():
    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=c007_out,
        prefix=".C007-",
        suffix=".tmp",
        delete=False,
    ) as stream:
        c007_temp = Path(stream.name)
        stream.write(c007_bytes)
        stream.flush()
        os.fsync(stream.fileno())

    try:
        if c007_temp.read_bytes() != c007_bytes:
            raise RuntimeError("C007_TEMP_RELOAD_CONFLICT")

        try:
            os.link(c007_temp, c007_target)
        except FileExistsError:
            pass
    finally:
        c007_temp.unlink(missing_ok=True)

if (
    c007_target.read_bytes() != c007_bytes
    or hashlib.sha256(c007_target.read_bytes()).hexdigest()
    != c007_sha
):
    raise RuntimeError("C007_CHECKPOINT_RELOAD_CONFLICT")

NB22E_V4_LOGICAL_INDEX_CHECK = {
    **c007_result,
    "checkpoint_path": str(c007_target),
    "checkpoint_sha256": c007_sha,
}

print(json.dumps(
    NB22E_V4_LOGICAL_INDEX_CHECK,
    ensure_ascii=False,
    indent=2,
))

C007 logical hash progress: 896 / 3488
C007 logical hash progress: 1792 / 3488
C007 logical hash progress: 2688 / 3488
C007 logical hash progress: 3488 / 3488
{
  "cell": "NB22E-v4:C007",
  "status": "LOGICAL_INDEX_HASH_MATCH",
  "producer": "NB14_CODE_CELL_10",
  "producer_git_blob_sha1": "1f8a8a82f2f1fbadc8e05977d8b129572381f2bf",
  "expected_logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
  "observed_logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
  "collection": "rag_finance_phase2_dense_v1",
  "point_count": 3488,
  "batches": 28,
  "original_qdrant_opened": false,
  "repository_writes": 0,
  "retrieval_queries": 0,
  "model_inference_calls": 0,
  "api_calls": 0,
  "pilot_started": false,
  "version_identity_final": false,
  "elapsed_seconds": 4.472,
  "pilot_stage_1_authorized_if_pass": true,
  "pilot_stage_2_authorized": false,
  "target_release_gate": "BLOCKED",
  "checkpoint_path": "/Users/ma

<div dir="rtl" align="right">

### NB22E-v4:C008 — R11.2: Pilot مرحلهٔ اول، تطبیق Hybrid100

**پیش‌نیاز اجباری:** `C007.status = LOGICAL_INDEX_HASH_MATCH`

**هدف:** اجرای حداکثر چهار Query با همان چهار شناسهٔ C006 و مقایسهٔ دقیق ترتیب Hybrid100 با خروجی تاریخی.

**روش:**
- خواندن Cache بردار Test بدون تولید embedding.
- مصرف Sparse Index و Analyzer فریز‌شده.
- محاسبهٔ BM25 و Weighted Fusion با تنظیمات ثابت NB16.
- اجرای Dense Retrieval صرفاً روی کپی موقت Index خارج از مخزن.
- مقایسهٔ تمام ۱۰۰ رتبه و عضویت Pool50.
- ذخیرهٔ نتیجهٔ هر سؤال پیش از سؤال بعدی.

**خروجی:** Checkpointهای هر سؤال و Summary مرحلهٔ اول.

**حداکثر Query:** چهار.

**شرط ورود به Stage 2:** هر چهار ترتیب Hybrid100 دقیقاً برابر باشند، عضویت Pool50 تطبیق کند و Summary معتبر ذخیره شود.

**محدودیت:** موفقیت این مرحله به معنی تأیید کیفیت، تولید پاسخ، مجوز Full Run یا پذیرش R11.2 نیست.

</div>

In [8]:
# NB22E-v4 | C008 | R11.2
# Run approved Stage 1: up to four frozen Hybrid100 queries on an isolated Index copy.

if not (
    NB22E_V4_LOGICAL_INDEX_CHECK.get("status")
    == "LOGICAL_INDEX_HASH_MATCH"
    and NB22E_V4_LOGICAL_INDEX_CHECK.get("checkpoint_sha256")
    and NB22E_V4_PILOT_PLAN["file_bytes_sha256"]
    == "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213"
):
    raise RuntimeError("C008_BLOCKED_UNTIL_C007_HASH_MATCH")

c008_root = c002_root.resolve()
c008_out = c003_out.resolve()
c008_ids = NB22E_V4_PILOT_PLAN["candidate_ids"]

if len(c008_ids) != 4 or len(set(c008_ids)) != 4:
    raise RuntimeError("C008_C006_FOUR_IDS_REQUIRED")

c008_np = nb22e_v4_require_imports("np")["np"]
c008_qdrant_class = nb22e_v4_require_imports(
    "QdrantClient"
)["QdrantClient"]

c008_started = time.perf_counter()
c008_attempts = 0
c008_results = []

def c008_require(condition, code):
    if not condition:
        raise RuntimeError("C008_" + code)

def c008_sha_file(path, expected):
    c008_require(
        path.is_file() and path.resolve().is_relative_to(c008_root),
        "UNSAFE_OR_MISSING_INPUT",
    )
    before = path.stat()
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    after = path.stat()

    c008_require(
        (before.st_size, before.st_mtime_ns)
        == (after.st_size, after.st_mtime_ns)
        and digest.hexdigest() == expected,
        "INPUT_SHA256_MISMATCH:" + path.name,
    )
    return path

def c008_read_file_pin(receipt):
    path = (c008_root / receipt["path"]).resolve()
    return c008_sha_file(path, receipt["file_bytes_sha256"])

def c008_write_once(label, value):
    raw = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8") + b"\n"

    digest = hashlib.sha256(raw).hexdigest()
    target = c008_out / (label + "." + digest + ".json")

    if not target.exists():
        with tempfile.NamedTemporaryFile(
            "wb",
            dir=c008_out,
            prefix=".C008-",
            suffix=".tmp",
            delete=False,
        ) as stream:
            tmp = Path(stream.name)
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())

        try:
            c008_require(
                tmp.read_bytes() == raw,
                "TEMP_RELOAD_MISMATCH",
            )
            try:
                os.link(tmp, target)
            except FileExistsError:
                pass
        finally:
            tmp.unlink(missing_ok=True)

    c008_require(
        target.read_bytes() == raw,
        "IMMUTABLE_FILE_CONFLICT",
    )

    return {
        "path": str(target),
        "file_bytes_sha256": digest,
        "size_bytes": len(raw),
    }

# Load only the pinned vector cache.
c008_vectors = c002_summary[
    "query_vector_precheck"
]["cache_file_pointers"]

c008_metadata_path = c008_read_file_pin(
    c008_vectors["metadata"]
)
c008_array_path = c008_read_file_pin(
    c008_vectors["array"]
)

c008_metadata = json.loads(
    c008_metadata_path.read_bytes()
)

c008_candidate_order = c008_metadata["candidate_ids"]

c008_require(
    len(c008_candidate_order)
    == len(set(c008_candidate_order))
    == 1149
    and set(c008_ids).issubset(c008_candidate_order),
    "VECTOR_CACHE_CANDIDATES",
)

c008_cache_array = c008_np.load(
    c008_array_path,
    mmap_mode="r",
    allow_pickle=False,
)

c008_require(
    c008_cache_array.shape == (1149, 384)
    and c008_cache_array.dtype == c008_np.float32,
    "VECTOR_CACHE_SHAPE",
)

c008_vector_positions = {
    cid: index
    for index, cid in enumerate(c008_candidate_order)
    if cid in c008_ids
}

# Pinned NB15 sparse contracts.
c008_sparse_handoff_ref = {
    "path": "artifacts/phase2/sparse_index/nb16_sparse_index_handoff_v1.json",
    "file_bytes_sha256": (
        "e4070684ba1662de35e2fbdf2737560aee1b7c67a694e6dab3bd98be2f578439"
    ),
}

c008_sparse_release_ref = {
    "path": "artifacts/phase2/sparse_index/sparse_index_release_v1.json",
    "file_bytes_sha256": (
        "879f52850a71629f02437c1ab1173e3f9e6c55ec5441ad4649784be63935c51b"
    ),
}

c008_handoff = json.loads(
    c008_read_file_pin(c008_sparse_handoff_ref).read_bytes()
)
c008_sparse_release = json.loads(
    c008_read_file_pin(c008_sparse_release_ref).read_bytes()
)

c008_require(
    c008_handoff["dense_pairing_contract"][
        "dense_logical_index_sha256"
    ] == NB22E_V4_LOGICAL_INDEX_CHECK[
        "observed_logical_index_sha256"
    ]
    and c008_handoff["sparse_release"]["sha256"]
    == c008_sparse_release_ref["file_bytes_sha256"]
    and c008_sparse_release["logical_index_sha256"]
    == c008_handoff["logical_index_sha256"]
    and c008_handoff["loader_contract"]["analyzer_id"]
    == "fa_join_zwnj_stopwords_v1",
    "SPARSE_VERSION_PAIRING_CONFLICT",
)

c008_loader = c008_handoff["loader_contract"]

c008_cfg_ref = {
    "path": c008_loader["configuration_path"],
    "file_bytes_sha256": c008_loader["configuration_file_sha256"],
}

c008_cfg = json.loads(
    c008_read_file_pin(c008_cfg_ref).read_bytes()
)

c008_require(
    c008_cfg["configuration_sha256"]
    == "96d66676d20bacd764ec9a163995dd48266c90c2acea1c8a5f77d95d9efb6ecc"
    and c008_cfg["configuration_status"]
    == "FROZEN_BEFORE_TEST_ACCESS"
    and float(c008_cfg["bm25"]["k1"]) == 1.2
    and float(c008_cfg["bm25"]["b"]) == 0.3,
    "SPARSE_CONFIG_CONFLICT",
)

c008_analyzer = c008_cfg["analyzer"]["definition"]
c008_stopwords = frozenset(
    c008_cfg["analyzer"]["stopwords"]
)

c008_require(
    c008_analyzer["analyzer_id"] == "fa_join_zwnj_stopwords_v1"
    and len(c008_stopwords) == 28,
    "SPARSE_ANALYZER_CONTRACT_CONFLICT",
)

# Frozen NB16 analyzer behavior.
def c008_tokens(text):
    config = c008_analyzer
    text = unicodedata.normalize(config["unicode_form"], text)

    if config["normalize_arabic_letters"]:
        text = text.translate(str.maketrans({
            "ي": "ی", "ى": "ی", "ك": "ک", "ة": "ه",
        }))

    if config["normalize_digits"]:
        text = text.translate(str.maketrans({
            "۰": "0", "۱": "1", "۲": "2", "۳": "3", "۴": "4",
            "۵": "5", "۶": "6", "۷": "7", "۸": "8", "۹": "9",
            "٠": "0", "١": "1", "٢": "2", "٣": "3", "٤": "4",
            "٥": "5", "٦": "6", "٧": "7", "٨": "8", "٩": "9",
        }))

    if config["lowercase_latin"]:
        text = text.lower()

    text = re.sub(r"(?<=\d)[,٬](?=\d)", "", text)
    text = re.sub(r"(?<=\d)٫(?=\d)", ".", text)
    text = text.replace("٪", " __percent__ ")
    text = text.replace("%", " __percent__ ")

    if config["zwnj_policy"] == "join":
        text = text.replace("\u200c", "")
    elif config["zwnj_policy"] == "split":
        text = text.replace("\u200c", " ")
    else:
        c008_require(
            config["zwnj_policy"] == "preserve",
            "UNKNOWN_ZWNJ_POLICY",
        )

    characters = []

    for character in text:
        if (
            config["remove_diacritics"]
            and unicodedata.category(character).startswith("M")
        ):
            continue

        if (
            character == "\u200c"
            or character == "."
            or character == "_"
            or character.isalpha()
            or character.isdigit()
        ):
            characters.append(character)
        else:
            characters.append(" ")

    words = [
        word.strip("._")
        for word in "".join(characters).split()
    ]

    return [
        word
        for word in words
        if word
        and not (
            config["remove_stopwords"]
            and word in c008_stopwords
        )
    ]

# Preserve the three frozen analyzer unit tests.
c008_require(
    c008_tokens("مي‌خواهم كسب‌وكار را بررسی کنم.")
    == ["میخواهم", "کسبوکار", "بررسی", "کنم"],
    "FROZEN_ANALYZER_UNIT_TEST_1",
)

c008_require(
    c008_tokens("بازدهی ۱۲٫۵٪ و مبلغ 1,250 تومان")
    == ["بازدهی", "12.5", "percent", "مبلغ", "1250", "تومان"],
    "FROZEN_ANALYZER_UNIT_TEST_2",
)

c008_require(
    c008_tokens("ETF، P/E و بهرهٔ مرکب چیست؟")
    == ["etf", "p", "e", "بهره", "مرکب", "چیست"],
    "FROZEN_ANALYZER_UNIT_TEST_3",
)

c008_query_tokens = {
    cid: c008_tokens(c002_questions[cid]["question"].strip())
    for cid in c008_ids
}

c008_relevant_tokens = set().union(
    *(set(tokens) for tokens in c008_query_tokens.values())
)

# Load only frozen sparse documents and relevant postings.
c008_document_ref = {
    "path": c008_loader["documents_path"],
    "file_bytes_sha256": c008_loader["documents_sha256"],
}
c008_postings_ref = {
    "path": c008_loader["postings_path"],
    "file_bytes_sha256": c008_loader["postings_sha256"],
}

c008_doc_path = c008_read_file_pin(c008_document_ref)
c008_postings_path = c008_read_file_pin(c008_postings_ref)

c008_docs = []

with c008_doc_path.open("rb") as handle:
    for line in handle:
        row = json.loads(line)
        c008_require(
            row["document_index"] == len(c008_docs),
            "SPARSE_DOCUMENT_ORDER",
        )
        c008_docs.append((
            row["document_id"],
            float(row["document_length"]),
        ))

c008_require(
    len(c008_docs) == 3488
    and [item[0] for item in c008_docs] == c007_chunk_ids,
    "SPARSE_DOCUMENTS_CHUNK_PARITY",
)

c008_doc_ids = [item[0] for item in c008_docs]

c008_doc_lengths = c008_np.asarray(
    [item[1] for item in c008_docs],
    dtype=c008_np.float64,
)

c008_average_doc_length = float(
    c008_cfg["index_statistics"]["average_document_length"]
)

c008_postings = {}

with c008_postings_path.open("rb") as handle:
    for line in handle:
        row = json.loads(line)

        if row["token"] not in c008_relevant_tokens:
            continue

        indexes = c008_np.asarray(
            row["document_indexes"],
            dtype=c008_np.int64,
        )

        frequencies = c008_np.asarray(
            row["term_frequencies"],
            dtype=c008_np.float64,
        )

        c008_require(
            len(indexes)
            == len(frequencies)
            == row["document_frequency"]
            and bool((indexes[:-1] < indexes[1:]).all()),
            "POSTINGS_DATA_CONFLICT",
        )

        c008_postings[row["token"]] = (
            indexes,
            frequencies,
        )

def c008_sparse_retrieve(candidate_id):
    scores = c008_np.zeros(
        3488,
        dtype=c008_np.float64,
    )

    for token, query_frequency in collections.Counter(
        c008_query_tokens[candidate_id]
    ).items():
        posting = c008_postings.get(token)

        if posting is None:
            continue

        indexes, frequencies = posting
        df = len(indexes)

        idf = math.log(
            1.0 + (3488 - df + 0.5) / (df + 0.5)
        )

        normalizer = (
            1.0 - 0.3
            + 0.3 * (
                c008_doc_lengths[indexes]
                / c008_average_doc_length
            )
        )

        scores[indexes] += (
            idf
            * (frequencies * 2.2)
            / (frequencies + 1.2 * normalizer)
        ) * float(query_frequency)

    positive = c008_np.flatnonzero(scores > 0)

    selected = positive[
        c008_np.lexsort((
            positive,
            -scores[positive],
        ))
    ][:100]

    return (
        [c008_doc_ids[i] for i in selected],
        [float(scores[i]) for i in selected],
    )

def c008_minmax(scores):
    arr = c008_np.asarray(
        scores,
        dtype=c008_np.float64,
    )

    if not arr.size:
        return arr

    if c008_np.isclose(
        float(arr.max()),
        float(arr.min()),
    ):
        return c008_np.ones_like(
            arr,
            dtype=c008_np.float64,
        )

    return (
        (arr - float(arr.min()))
        / (float(arr.max()) - float(arr.min()))
    )

def c008_fuse(dense_ids, dense_scores, sparse_ids, sparse_scores):
    combined = collections.defaultdict(float)

    for cid, value in zip(
        dense_ids[:100],
        c008_minmax(dense_scores[:100]),
    ):
        combined[str(cid)] += 0.4 * float(value)

    for cid, value in zip(
        sparse_ids[:100],
        c008_minmax(sparse_scores[:100]),
    ):
        combined[str(cid)] += 0.6 * float(value)

    return [
        cid
        for cid, _ in sorted(
            combined.items(),
            key=lambda item: (-item[1], item[0]),
        )
    ][:100]

# Bind historical rows without modifying them.
c008_hybrid_ref = c002_summary["input_file_checks"]["HYBRID"]
c008_reranker_ref = c002_summary["input_file_checks"]["RERANKER"]

def c008_select_rows(receipt, depth):
    path = c008_read_file_pin(receipt)
    selected = {}

    with path.open("rb") as handle:
        for line in handle:
            row = json.loads(line)
            candidate_id = row.get("candidate_id")

            if candidate_id not in c008_ids:
                continue

            valid_split = (
                row.get("split") == "test"
                if depth == 100
                else row.get("split") in (None, "test")
            )

            c008_require(
                candidate_id not in selected and valid_split,
                "HISTORICAL_CANDIDATE_DUPLICATE_OR_SPLIT",
            )

            entries = row["candidates"]

            c008_require(
                len(entries) == depth
                and [entry["rank"] for entry in entries]
                == list(range(1, depth + 1)),
                "HISTORICAL_RANK_SEQUENCE",
            )

            selected[candidate_id] = row

    c008_require(
        set(selected) == set(c008_ids),
        "HISTORICAL_FOUR_CANDIDATES_MISSING",
    )

    return selected

c008_historical_hybrid = c008_select_rows(
    c008_hybrid_ref,
    100,
)

c008_historical_reranker = c008_select_rows(
    c008_reranker_ref,
    50,
)

# Repeat the isolated-copy method, not the historical parity audit.
c008_source_files = []

for directory, dirs, files in os.walk(
    c007_original,
    followlinks=False,
):
    c008_require(
        not any(
            (Path(directory) / entry).is_symlink()
            for entry in dirs + files
        ),
        "INDEX_SYMLINK",
    )

    for name in files:
        source = Path(directory) / name
        stat = source.stat()

        c008_source_files.append((
            source.relative_to(c007_original),
            stat.st_size,
            stat.st_mtime_ns,
        ))

c008_require(
    len(c008_source_files) == c005_index_result["inventory_files"]
    and sum(item[1] for item in c008_source_files)
    == c005_index_result["inventory_bytes"],
    "INDEX_INVENTORY_CHANGED",
)

c008_stage1_status = "BLOCKED"
c008_error = None

try:
    with tempfile.TemporaryDirectory(
        prefix="NB22E-v4-C008-pilot-",
        dir=str(c008_out),
    ) as working_directory:

        clone = Path(working_directory) / "qdrant"
        clone.mkdir()

        for relative, size, modified in c008_source_files:
            source = c007_original / relative
            target = clone / relative

            before = source.stat()

            c008_require(
                not source.is_symlink()
                and (before.st_size, before.st_mtime_ns)
                == (size, modified),
                "INDEX_CHANGED",
            )

            target.parent.mkdir(parents=True, exist_ok=True)

            with (
                source.open("rb") as source_handle,
                target.open("xb") as target_handle,
            ):
                for block in iter(
                    lambda: source_handle.read(1024 * 1024),
                    b"",
                ):
                    target_handle.write(block)

            after = source.stat()

            c008_require(
                (after.st_size, after.st_mtime_ns)
                == (size, modified),
                "INDEX_CHANGED_DURING_COPY",
            )

        client = None

        try:
            client = c008_qdrant_class(path=str(clone))

            c008_require(
                client.count(
                    collection_name=c005_collection,
                    exact=True,
                ).count == 3488,
                "INDEX_CLONE_COUNT",
            )

            for candidate_id in c008_ids:
                started = time.perf_counter()

                vector = c008_np.asarray(
                    c008_cache_array[
                        c008_vector_positions[candidate_id]
                    ],
                    dtype=c008_np.float32,
                )

                c008_require(
                    vector.shape == (384,)
                    and bool(c008_np.isfinite(vector).all())
                    and bool(
                        c008_np.isclose(
                            c008_np.linalg.norm(vector),
                            1.0,
                        )
                    ),
                    "VECTOR_INVALID",
                )

                sparse_ids, sparse_scores = (
                    c008_sparse_retrieve(candidate_id)
                )

                c008_attempts += 1

                dense_result = client.query_points(
                    collection_name=c005_collection,
                    query=vector.tolist(),
                    limit=100,
                    with_payload=True,
                    with_vectors=False,
                )

                dense_points = getattr(
                    dense_result,
                    "points",
                    dense_result,
                )

                dense_ids = [
                    str(point.payload["chunk_id"])
                    for point in dense_points
                ]

                dense_scores = [
                    float(point.score)
                    for point in dense_points
                ]

                observed = c008_fuse(
                    dense_ids,
                    dense_scores,
                    sparse_ids,
                    sparse_scores,
                )

                expected = [
                    row["chunk_id"]
                    for row in c008_historical_hybrid[
                        candidate_id
                    ]["candidates"]
                ]

                pool = [
                    row["chunk_id"]
                    for row in c008_historical_reranker[
                        candidate_id
                    ]["candidates"]
                ]

                order_matches = observed == expected
                pool_matches = set(observed[:50]) == set(pool)

                result = {
                    "candidate_id": candidate_id,
                    "question_sha256": (
                        c002_questions[candidate_id][
                            "question_sha256"
                        ]
                    ),
                    "observed_hybrid100_order": observed,
                    "expected_hybrid100_order_sha256": (
                        hashlib.sha256(
                            json.dumps(
                                expected,
                                ensure_ascii=False,
                                separators=(",", ":"),
                            ).encode("utf-8")
                        ).hexdigest()
                    ),
                    "dense_retrieved": len(dense_ids),
                    "sparse_retrieved": len(sparse_ids),
                    "observed_depth": len(observed),
                    "full_order_matches": order_matches,
                    "pool50_membership_matches": pool_matches,
                    "first_mismatch_rank": next(
                        (
                            index + 1
                            for index, (left, right) in enumerate(
                                zip(observed, expected)
                            )
                            if left != right
                        ),
                        None,
                    ),
                    "elapsed_seconds": round(
                        time.perf_counter() - started,
                        3,
                    ),
                    "stage1_status": (
                        "PASS"
                        if (
                            order_matches
                            and pool_matches
                            and len(observed) == 100
                        )
                        else "BLOCKED"
                    ),
                    "qdrant_queries": 1,
                    "api_calls": 0,
                    "model_inference_calls": 0,
                }

                result["checkpoint"] = c008_write_once(
                    "C008.stage1_candidate",
                    result,
                )

                c008_results.append(result)

                print(
                    "C008 Stage 1:",
                    len(c008_results),
                    "/ 4",
                    result["stage1_status"],
                    flush=True,
                )

                if result["stage1_status"] != "PASS":
                    raise RuntimeError(
                        "STAGE1_HYBRID100_OR_POOL50_VERSION_CONFLICT:"
                        + candidate_id
                    )

        finally:
            if client is not None:
                client.close()

    c008_stage1_status = "STAGE1_PASS"

except Exception as error:
    c008_error = {
        "type": type(error).__name__,
        "detail": str(error)[:600],
    }

c008_peak_rss_raw = resource.getrusage(
    resource.RUSAGE_SELF
).ru_maxrss

c008_peak_rss_gib = (
    c008_peak_rss_raw
    / (1024 ** 3 if sys.platform == "darwin" else 1024 ** 2)
)

c008_report = {
    "cell": "NB22E-v4:C008",
    "status": c008_stage1_status,
    "stage": "STAGE1_FROZEN_HYBRID100_AND_POOL50_PARITY",
    "candidate_ids": c008_ids,
    "candidate_count_completed": len(c008_results),
    "results": c008_results,
    "qdrant_queries_attempted": c008_attempts,
    "elapsed_seconds": round(
        time.perf_counter() - c008_started,
        3,
    ),
    "process_peak_rss_gib": round(c008_peak_rss_gib, 3),
    "original_index_opened": False,
    "repository_writes": 0,
    "version_boundary_approved": True,
    "logical_index_hash_match_checkpoint_sha256": (
        NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"]
    ),
    "stage2_authorized": c008_stage1_status == "STAGE1_PASS",
    "stage2_started": False,
    "target_release_gate": "BLOCKED",
    "api_calls": 0,
    "model_inference_calls": 0,
    "blocker": c008_error,
}

c008_report["checkpoint"] = c008_write_once(
    "C008.stage1_summary",
    c008_report,
)

NB22E_V4_STAGE1_RESULT = c008_report

print(json.dumps(
    {
        key: value
        for key, value in c008_report.items()
        if key != "results"
    },
    ensure_ascii=False,
    indent=2,
))

C008 Stage 1: 1 / 4 PASS
C008 Stage 1: 2 / 4 PASS
C008 Stage 1: 3 / 4 PASS
C008 Stage 1: 4 / 4 PASS
{
  "cell": "NB22E-v4:C008",
  "status": "STAGE1_PASS",
  "stage": "STAGE1_FROZEN_HYBRID100_AND_POOL50_PARITY",
  "candidate_ids": [
    "golden_v2_candidate_ca5362d68c692d667fad3006",
    "golden_v2_candidate_15cb48c40c3d1e292f647ff9",
    "golden_v2_candidate_4032db31c53d0465590cf0ab",
    "golden_v2_candidate_ba8ca208df074fc579e48eb5"
  ],
  "candidate_count_completed": 4,
  "qdrant_queries_attempted": 4,
  "elapsed_seconds": 3.66,
  "process_peak_rss_gib": 0.851,
  "original_index_opened": false,
  "repository_writes": 0,
  "version_boundary_approved": true,
  "logical_index_hash_match_checkpoint_sha256": "d52e91db8f136f5b2145715f54d6aecc9e5bd5a2869cb1da719c5008c76a49b8",
  "stage2_authorized": true,
  "stage2_started": false,
  "target_release_gate": "BLOCKED",
  "api_calls": 0,
  "model_inference_calls": 0,
  "blocker": null,
  "checkpoint": {
    "path": "/Users/macbookpro/AMIR_DA

<div dir="rtl" align="right">

### NB22E-v4:C009 — R11.2: اجرای محدود Stage 2 / Local100

**پیش‌نیازهای تأییدشده:**
- C007: تطبیق کامل Logical Index Hash.
- C008: موفقیت Stage 1 برای چهار سؤال.
- مجوز ناظر: حداکثر ۲۰۰ جفت اضافی ONNX.

**روش اجرا:**
۱. حفظ همان چهار شناسهٔ C006 و ترتیب Hybrid100 تأییدشده در C008.
۲. استفادهٔ مجدد از ۵۰ امتیاز تاریخی Pool50 برای هر سؤال.
۳. محاسبهٔ فقط ۵۰ جفت باقی‌مانده برای هر سؤال، در Batchهای حداکثر چهارتایی.
۴. اجرای ONNX صرفاً روی CPU با چهار Thread و `max_length=512`.
۵. تشکیل رتبه‌بندی Local100 با امتیاز نزولی و در تساوی، `chunk_id` صعودی.
۶. ذخیرهٔ Checkpoint پیش از هر Batch، پس از تکمیل آن و پس از هر سؤال.
۷. بازیابی Batchهای تکمیل‌شده بدون استنتاج تکراری؛ توقف در صورت وجود Batch شروع‌شده با نتیجهٔ نامعلوم.
۸. ثبت زمان، حافظهٔ اوج پردازش، تعداد جفت‌ها، تعداد فراخوانی‌های مدل و خطاها.

**محدودیت‌ها:**
- بدون Qdrant Query تازه، API، تولید پاسخ یا تغییر مخزن.
- بدون تغییر تنظیمات براساس نتایج Test.
- بدون اجرای ۱۱۱۶ سؤال.
- موفقیت Stage 2، معادل تأیید کیفیت یا پذیرش نهایی R11.2 نیست.
- هویت نهایی نسخهٔ پاسخ همچنان جدا از هویت تاریخی تعیین خواهد شد.

**خروجی:** گزارش‌ها و Checkpointهای تغییرناپذیر در پوشهٔ مستقل `C009_stage2` خارج از مخزن.

**شرط پایان:** چهار Checkpoint معتبر با مجموع دقیق ۲۰۰ امتیاز جدید و Summary موفق؛ در غیر این صورت `STAGE2_BLOCKED`.

</div>

In [9]:
# NB22E-v4 | C009 | R11.2
# Stage 2 repair: preserve Hugging Face snapshot aliases while verifying actual ONNX file bytes.

def c009_require(condition, code):
    if not condition:
        raise RuntimeError("C009_" + code)

# ============================================================
# 1. Bind the approved plan and completed checkpoints
# ============================================================

c009_plan_sha = "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213"
c009_stage1_sha = "ba56c485f014a2eb149b63bad85eec6c790d2eb8ebf36f4f8348a9768f7955c6"
c009_index_sha = "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4"
c009_model_sha = "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"

c009_root = c002_root.resolve()
c009_out = c003_out.resolve()

c009_require(
    NB22E_V4_READ_ONLY is True
    and c009_out.is_dir()
    and not c009_out.is_relative_to(c009_root)
    and NB22E_V4_PILOT_PLAN.get("status") == "PILOT_PLAN_RELOAD_PASS"
    and NB22E_V4_LOGICAL_INDEX_CHECK.get("status") == "LOGICAL_INDEX_HASH_MATCH"
    and NB22E_V4_STAGE1_RESULT.get("status") == "STAGE1_PASS"
    and NB22E_V4_STAGE1_RESULT.get("stage2_authorized") is True
    and NB22E_V4_STAGE1_RESULT.get("qdrant_queries_attempted") == 4,
    "PREREQUISITE_GATE",
)

def c009_verify_checkpoint(path_value, expected_sha):
    path = Path(path_value).resolve()
    c009_require(
        path.parent == c009_out and path.is_file(),
        "CHECKPOINT_LOCATION",
    )
    raw = path.read_bytes()
    c009_require(
        hashlib.sha256(raw).hexdigest() == expected_sha,
        "CHECKPOINT_FILE_BYTES",
    )
    return json.loads(raw)

c009_require(
    NB22E_V4_PILOT_PLAN["file_bytes_sha256"] == c009_plan_sha
    and NB22E_V4_STAGE1_RESULT["checkpoint"]["file_bytes_sha256"]
    == c009_stage1_sha
    and NB22E_V4_LOGICAL_INDEX_CHECK["observed_logical_index_sha256"]
    == c009_index_sha
    and c004_model_actual_sha == c009_model_sha
    and c004_onnxruntime_version == "1.16.3",
    "FROZEN_PIN_CONFLICT",
)

c009_plan = c009_verify_checkpoint(
    NB22E_V4_PILOT_PLAN["path"], c009_plan_sha
)
c009_stage1 = c009_verify_checkpoint(
    NB22E_V4_STAGE1_RESULT["checkpoint"]["path"],
    c009_stage1_sha,
)
c009_index = c009_verify_checkpoint(
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_path"],
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"],
)

c009_ids = list(NB22E_V4_PILOT_PLAN["candidate_ids"])

c009_require(
    len(c009_ids) == len(set(c009_ids)) == 4
    and [row["candidate_id"] for row in c009_plan["population"]["pilot_candidates"]]
    == c009_ids
    and c009_stage1["candidate_ids"] == c009_ids
    and c009_stage1["status"] == "STAGE1_PASS"
    and c009_stage1["stage2_authorized"] is True
    and c009_stage1["logical_index_hash_match_checkpoint_sha256"]
    == NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"]
    and c009_index["status"] == "LOGICAL_INDEX_HASH_MATCH"
    and c009_index["observed_logical_index_sha256"] == c009_index_sha,
    "APPROVED_SCOPE_CONFLICT",
)

c009_stage1_rows = c009_stage1["results"]

c009_require(
    len(c009_stage1_rows) == 4
    and [row["candidate_id"] for row in c009_stage1_rows] == c009_ids
    and all(
        row["stage1_status"] == "PASS"
        and row["full_order_matches"] is True
        and row["pool50_membership_matches"] is True
        and row["observed_depth"] == 100
        and row["qdrant_queries"] == 1
        for row in c009_stage1_rows
    ),
    "STAGE1_DETAIL_CONFLICT",
)

for row in c009_stage1_rows:
    receipt = row["checkpoint"]
    saved = c009_verify_checkpoint(
        receipt["path"], receipt["file_bytes_sha256"]
    )
    c009_require(
        saved["candidate_id"] == row["candidate_id"]
        and saved["stage1_status"] == "PASS"
        and saved["observed_hybrid100_order"]
        == row["observed_hybrid100_order"],
        "STAGE1_CANDIDATE_CHECKPOINT",
    )

# ============================================================
# 2. Bind 50 historical scores and 50 missing pairs per question
# ============================================================

c009_inputs = {}
c009_needed_chunk_ids = set()

for candidate_id, stage1_row in zip(c009_ids, c009_stage1_rows):
    hybrid_rows = c008_historical_hybrid[candidate_id]["candidates"]
    pool_rows = c008_historical_reranker[candidate_id]["candidates"]

    hybrid_ids = [row["chunk_id"] for row in hybrid_rows]
    pool_ids = [row["chunk_id"] for row in pool_rows]

    c009_require(
        len(hybrid_ids) == len(set(hybrid_ids)) == 100
        and len(pool_ids) == len(set(pool_ids)) == 50
        and [row["rank"] for row in hybrid_rows] == list(range(1, 101))
        and [row["rank"] for row in pool_rows] == list(range(1, 51))
        and hybrid_ids == stage1_row["observed_hybrid100_order"]
        and set(pool_ids) == set(hybrid_ids[:50]),
        "HISTORICAL_POOL_OR_RANK_CONFLICT:" + candidate_id,
    )

    saved_scores = {
        row["chunk_id"]: float(row["reranker_score"])
        for row in pool_rows
    }

    c009_require(
        len(saved_scores) == 50
        and all(math.isfinite(score) for score in saved_scores.values()),
        "HISTORICAL_SCORE_INVALID:" + candidate_id,
    )

    missing_ids = [
        chunk_id for chunk_id in hybrid_ids
        if chunk_id not in saved_scores
    ]

    c009_require(
        len(missing_ids) == len(set(missing_ids)) == 50,
        "MISSING50_IDENTITY_CONFLICT:" + candidate_id,
    )

    raw_question = c002_questions[candidate_id]["question"]
    question_sha = hashlib.sha256(
        raw_question.encode("utf-8")
    ).hexdigest()

    c009_require(
        bool(raw_question.strip())
        and question_sha == c002_questions[candidate_id]["question_sha256"]
        and question_sha == stage1_row["question_sha256"],
        "QUESTION_SHA_CONFLICT:" + candidate_id,
    )

    c009_inputs[candidate_id] = {
        "question": raw_question.strip(),
        "question_sha256": question_sha,
        "hybrid100": hybrid_ids,
        "pool50": pool_ids,
        "saved_scores": saved_scores,
        "missing50": missing_ids,
    }

    c009_needed_chunk_ids.update(missing_ids)

c009_require(
    sum(len(item["missing50"]) for item in c009_inputs.values()) == 200,
    "PAIR_BUDGET",
)

# ============================================================
# 3. Load exact chunk texts; repair the ONNX symlink check
# ============================================================

c009_chunks_receipt = c002_summary["input_file_checks"]["CHUNKS"]
c009_chunks_path = c008_read_file_pin(c009_chunks_receipt)

c009_contents = {}

with c009_chunks_path.open("rb") as stream:
    for line in stream:
        row = json.loads(line)
        chunk_id = row["chunk_id"]

        if chunk_id not in c009_needed_chunk_ids:
            continue

        c009_require(
            chunk_id not in c009_contents
            and isinstance(row.get("content"), str),
            "CHUNK_TEXT_INVALID_OR_DUPLICATE",
        )
        c009_contents[chunk_id] = row["content"]

c009_require(
    set(c009_contents) == c009_needed_chunk_ids,
    "CHUNK_TEXT_COVERAGE",
)

# Important: do not dereference the snapshot alias before checking
# its parent directory. Hugging Face model files may link to blobs/.
c009_model_path = Path(c004_model_path).expanduser()
c009_snapshot_alias = Path(c004_snapshot).expanduser()

c009_model_alias_parent = Path(
    os.path.abspath(str(c009_model_path.parent.parent))
)
c009_expected_snapshot_alias = Path(
    os.path.abspath(str(c009_snapshot_alias))
)

c009_model_cache_root = c009_snapshot_alias.parent.parent.resolve()
c009_model_target = c009_model_path.resolve()

c009_require(
    c009_model_path.is_file()
    and c009_model_alias_parent == c009_expected_snapshot_alias
    and c009_model_target.is_file()
    and c009_model_target.is_relative_to(c009_model_cache_root),
    "ONNX_SNAPSHOT_ALIAS_OR_CACHE_BOUNDARY_CONFLICT",
)

print(json.dumps({
    "cell": "NB22E-v4:C009",
    "model_alias": str(c009_model_path),
    "model_resolved_target": str(c009_model_target),
    "snapshot_alias_verified": True,
    "model_sha256_required_at_execution": c009_model_sha,
    "candidate_count": 4,
    "additional_pairs_max": 200,
    "qdrant_queries": 0,
    "api_calls": 0,
}, ensure_ascii=False, indent=2))

# ============================================================
# 4. Freeze Stage 2 identity
# ============================================================

def c009_canonical(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

c009_scope = {
    "schema": "NB22E_V4_R11_2_STAGE2_INPUTS_V1",
    "approved_plan_sha256": c009_plan_sha,
    "verified_stage1_sha256": c009_stage1_sha,
    "verified_c007_sha256": (
        NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"]
    ),
    "logical_index_sha256": c009_index_sha,
    "chunks_sha256": c009_chunks_receipt["file_bytes_sha256"],
    "hybrid_sha256": (
        c002_summary["input_file_checks"]["HYBRID"]["file_bytes_sha256"]
    ),
    "reranker_sha256": (
        c002_summary["input_file_checks"]["RERANKER"]["file_bytes_sha256"]
    ),
    "model_sha256": c009_model_sha,
    "model_revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "onnxruntime_version": "1.16.3",
    "provider": "CPUExecutionProvider",
    "threads": 4,
    "batch_size": 4,
    "max_length": 512,
    "candidate_ids": c009_ids,
    "new_pairs_max": 200,
}

c009_scope_sha = hashlib.sha256(
    c009_canonical(c009_scope)
).hexdigest()

c009_dir = (c009_out / "C009_stage2").resolve()

c009_require(
    c009_dir.parent == c009_out,
    "STAGE2_DIRECTORY_UNSAFE",
)

c009_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# 5. Immutable checkpoint helpers
# ============================================================

def c009_record_bytes(core):
    core_sha = hashlib.sha256(
        c009_canonical(core)
    ).hexdigest()

    return c009_canonical({
        **core,
        "record_core_sha256": core_sha,
    }) + b"\n"

def c009_put_once(path, core):
    c009_require(
        path.parent.resolve() == c009_dir,
        "OUTPUT_PATH_UNSAFE",
    )

    raw = c009_record_bytes(core)

    if not path.exists():
        temporary = None

        try:
            with tempfile.NamedTemporaryFile(
                "wb",
                dir=c009_dir,
                prefix=".C009-",
                suffix=".tmp",
                delete=False,
            ) as stream:
                temporary = Path(stream.name)
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())

            try:
                os.link(temporary, path)
            except FileExistsError:
                pass

        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    c009_require(
        path.read_bytes() == raw,
        "IMMUTABLE_CHECKPOINT_CONFLICT:" + path.name,
    )

    return hashlib.sha256(raw).hexdigest()

def c009_read_record(path):
    c009_require(
        path.parent.resolve() == c009_dir and path.is_file(),
        "CHECKPOINT_MISSING_OR_UNSAFE",
    )

    raw = path.read_bytes()
    record = json.loads(raw)
    recorded_core_sha = record.pop("record_core_sha256", None)

    c009_require(
        recorded_core_sha == hashlib.sha256(
            c009_canonical(record)
        ).hexdigest(),
        "CHECKPOINT_CORE_SHA_CONFLICT:" + path.name,
    )

    return record, hashlib.sha256(raw).hexdigest()

def c009_rss_gib():
    peak = resource.getrusage(
        resource.RUSAGE_SELF
    ).ru_maxrss

    divisor = (
        1024 ** 3 if sys.platform == "darwin"
        else 1024 ** 2
    )

    return round(peak / divisor, 3)

# ============================================================
# 6. Execute only missing batches; preserve completed batches
# ============================================================

c009_started = time.perf_counter()
c009_pairs_attempted_this_run = 0
c009_pairs_completed_this_run = 0
c009_pairs_reused_this_run = 0
c009_model_calls_this_run = 0

c009_candidate_results = []
c009_blocker = None
c009_current_candidate = None

c009_session = None
c009_tokenizer = c005_tokenizer

try:
    for candidate_id in c009_ids:
        c009_current_candidate = candidate_id
        candidate_started = time.perf_counter()
        item = c009_inputs[candidate_id]

        candidate_key = hashlib.sha256(
            candidate_id.encode("utf-8")
        ).hexdigest()[:24]

        candidate_identity = {
            "scope_sha256": c009_scope_sha,
            "candidate_id": candidate_id,
            "question_sha256": item["question_sha256"],
            "hybrid100": item["hybrid100"],
            "pool50": item["pool50"],
            "saved_scores": item["saved_scores"],
            "missing50": item["missing50"],
            "chunk_text_sha256": {
                chunk_id: hashlib.sha256(
                    c009_contents[chunk_id].encode("utf-8")
                ).hexdigest()
                for chunk_id in item["missing50"]
            },
        }

        candidate_input_sha = hashlib.sha256(
            c009_canonical(candidate_identity)
        ).hexdigest()

        candidate_path = c009_dir / (
            f"C009.candidate.{candidate_key}."
            f"{c009_scope_sha[:16]}.json"
        )

        def c009_batch_paths(start):
            key = (
                f"{candidate_key}.{start:02d}."
                f"{c009_scope_sha[:16]}"
            )

            return (
                c009_dir / f"C009.started.{key}.json",
                c009_dir / f"C009.completed.{key}.json",
            )

        # Reuse an already validated complete candidate.
        if candidate_path.is_file():
            saved, candidate_file_sha = c009_read_record(
                candidate_path
            )

            c009_require(
                saved["status"] == "CANDIDATE_PASS"
                and saved["scope_sha256"] == c009_scope_sha
                and saved["input_sha256"] == candidate_input_sha
                and saved["candidate_id"] == candidate_id
                and saved["original_hybrid100_order"]
                == item["hybrid100"]
                and saved["stored_pool50_order"] == item["pool50"]
                and saved["stored_pool50_scores"] == item["saved_scores"]
                and set(saved["new_scores_by_chunk_id"])
                == set(item["missing50"])
                and len(saved["batch_receipts"]) == 13,
                "EXISTING_CANDIDATE_IDENTITY_CONFLICT",
            )

            full_scores = {
                **item["saved_scores"],
                **saved["new_scores_by_chunk_id"],
            }

            c009_require(
                len(full_scores) == 100
                and all(math.isfinite(v) for v in full_scores.values())
                and saved["local100_order"] == sorted(
                    item["hybrid100"],
                    key=lambda chunk_id: (
                        -full_scores[chunk_id],
                        chunk_id,
                    ),
                ),
                "EXISTING_LOCAL100_ORDER_CONFLICT",
            )

            for index, receipt in enumerate(saved["batch_receipts"]):
                start = index * 4
                batch_ids = item["missing50"][start:start + 4]
                started_path, completed_path = c009_batch_paths(start)

                c009_require(
                    receipt["path"] == str(completed_path)
                    and receipt["batch_start"] == start
                    and started_path.is_file(),
                    "CANDIDATE_BATCH_RECEIPT_PATH",
                )

                started_row, _ = c009_read_record(started_path)
                completed_row, completed_sha = c009_read_record(
                    completed_path
                )

                c009_require(
                    started_row["status"] == "STARTED_NOT_VERIFIED"
                    and started_row["input_sha256"] == candidate_input_sha
                    and started_row["chunk_ids"] == batch_ids
                    and completed_sha == receipt["file_bytes_sha256"]
                    and completed_row["status"] == "BATCH_PASS"
                    and completed_row["input_sha256"] == candidate_input_sha
                    and completed_row["chunk_ids"] == batch_ids
                    and dict(zip(
                        completed_row["chunk_ids"],
                        completed_row["scores"],
                    )) == {
                        chunk_id: saved["new_scores_by_chunk_id"][chunk_id]
                        for chunk_id in batch_ids
                    },
                    "EXISTING_BATCH_RECEIPT_CONFLICT",
                )

            c009_pairs_reused_this_run += 50

            c009_candidate_results.append({
                "candidate_id": candidate_id,
                "status": "CANDIDATE_PASS_REUSED",
                "pairs_in_receipt": 50,
                "new_pairs_this_run": 0,
                "path": str(candidate_path),
                "file_bytes_sha256": candidate_file_sha,
            })

            print(
                "C009 candidate:",
                len(c009_candidate_results),
                "/4 REUSED",
                flush=True,
            )

            continue

        new_scores = {}
        batch_receipts = []
        candidate_new_calls = 0
        candidate_new_pairs = 0

        for start in range(0, 50, 4):
            batch_ids = item["missing50"][start:start + 4]
            started_path, completed_path = c009_batch_paths(start)

            attempt_identity = {
                "cell": "NB22E-v4:C009",
                "status": "STARTED_NOT_VERIFIED",
                "scope_sha256": c009_scope_sha,
                "input_sha256": candidate_input_sha,
                "candidate_id": candidate_id,
                "batch_start": start,
                "chunk_ids": batch_ids,
                "score_count_expected": len(batch_ids),
                "manual_resolution_required_if_completion_missing": True,
            }

            if completed_path.is_file():
                c009_require(
                    started_path.is_file(),
                    "COMPLETED_WITHOUT_STARTED",
                )

                started_record, _ = c009_read_record(started_path)
                completed_record, completed_sha = c009_read_record(
                    completed_path
                )

                c009_require(
                    started_record == attempt_identity
                    and completed_record["status"] == "BATCH_PASS"
                    and completed_record["scope_sha256"] == c009_scope_sha
                    and completed_record["input_sha256"]
                    == candidate_input_sha
                    and completed_record["candidate_id"] == candidate_id
                    and completed_record["chunk_ids"] == batch_ids,
                    "REUSED_BATCH_IDENTITY_CONFLICT",
                )

                values = completed_record["scores"]

                c009_require(
                    len(values) == len(batch_ids)
                    and all(math.isfinite(v) for v in values),
                    "REUSED_BATCH_SCORE_CONFLICT",
                )

                c009_pairs_reused_this_run += len(batch_ids)

            else:
                c009_require(
                    not started_path.exists(),
                    "UNRESOLVED_STARTED_BATCH_NO_AUTO_RETRY:"
                    + candidate_id + ":" + str(start),
                )

                c009_require(
                    c009_pairs_attempted_this_run + len(batch_ids) <= 200,
                    "AUTHORIZED_PAIR_LIMIT",
                )

                if c009_session is None:
                    c009_np = nb22e_v4_require_imports("np")["np"]
                    c009_ort = nb22e_v4_require_imports("ort")["ort"]

                    c009_require(
                        c009_ort.__version__ == "1.16.3",
                        "ONNX_RUNTIME_VERSION_CONFLICT",
                    )

                    # Hash the resolved file contents, not the symlink text.
                    before = c009_model_path.stat()
                    digest = hashlib.sha256()

                    with c009_model_path.open("rb") as stream:
                        for block in iter(
                            lambda: stream.read(1024 * 1024),
                            b"",
                        ):
                            digest.update(block)

                    after = c009_model_path.stat()

                    c009_require(
                        (before.st_size, before.st_mtime_ns)
                        == (after.st_size, after.st_mtime_ns)
                        and digest.hexdigest() == c009_model_sha,
                        "MODEL_FILE_SHA256_CONFLICT",
                    )

                    options = c009_ort.SessionOptions()
                    options.intra_op_num_threads = 4
                    options.inter_op_num_threads = 1
                    options.execution_mode = (
                        c009_ort.ExecutionMode.ORT_SEQUENTIAL
                    )
                    options.graph_optimization_level = (
                        c009_ort.GraphOptimizationLevel.ORT_ENABLE_ALL
                    )

                    c009_session = c009_ort.InferenceSession(
                        str(c009_model_path),
                        sess_options=options,
                        providers=["CPUExecutionProvider"],
                    )

                    c009_input_names = {
                        field.name
                        for field in c009_session.get_inputs()
                    }

                    c009_require(
                        c009_session.get_providers()
                        == ["CPUExecutionProvider"]
                        and {"input_ids", "attention_mask"}
                        <= c009_input_names,
                        "ONNX_CPU_PROVIDER_OR_INPUT_CONFLICT",
                    )

                c009_require(
                    c009_rss_gib() <= 6.0,
                    "PROCESS_RSS_OVER_LIMIT",
                )

                encoded = c009_tokenizer(
                    [item["question"]] * len(batch_ids),
                    [
                        c009_contents[chunk_id]
                        for chunk_id in batch_ids
                    ],
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="np",
                )

                c009_require(
                    c009_input_names <= set(encoded),
                    "TOKENIZER_INPUT_CONFLICT",
                )

                feed = {
                    name: c009_np.asarray(
                        encoded[name],
                        dtype=c009_np.int64,
                    )
                    for name in c009_input_names
                }

                c009_require(
                    all(
                        value.shape[0] == len(batch_ids)
                        and value.shape[1] <= 512
                        for value in feed.values()
                    ),
                    "TOKENIZER_BATCH_SHAPE",
                )

                # Durable intent is written before inference.
                c009_put_once(
                    started_path,
                    attempt_identity,
                )

                batch_started = time.perf_counter()

                c009_pairs_attempted_this_run += len(batch_ids)
                c009_model_calls_this_run += 1
                candidate_new_calls += 1

                outputs = c009_session.run(None, feed)

                scores = c009_np.asarray(
                    outputs[0],
                    dtype=c009_np.float64,
                ).reshape(-1)

                c009_require(
                    scores.shape == (len(batch_ids),)
                    and bool(c009_np.isfinite(scores).all()),
                    "ONNX_OUTPUT_SCORE_CONFLICT",
                )

                values = [float(score) for score in scores]

                completed_sha = c009_put_once(
                    completed_path,
                    {
                        "cell": "NB22E-v4:C009",
                        "status": "BATCH_PASS",
                        "scope_sha256": c009_scope_sha,
                        "input_sha256": candidate_input_sha,
                        "candidate_id": candidate_id,
                        "chunk_ids": batch_ids,
                        "scores": values,
                        "elapsed_seconds": round(
                            time.perf_counter() - batch_started,
                            4,
                        ),
                        "process_peak_rss_gib": c009_rss_gib(),
                        "model_inference_calls": 1,
                        "qdrant_queries": 0,
                        "api_calls": 0,
                    },
                )

                c009_pairs_completed_this_run += len(batch_ids)
                candidate_new_pairs += len(batch_ids)

            c009_require(
                not set(batch_ids).intersection(new_scores),
                "DUPLICATE_BATCH_IDS",
            )

            new_scores.update(dict(zip(batch_ids, values)))

            batch_receipts.append({
                "batch_start": start,
                "count": len(batch_ids),
                "path": str(completed_path),
                "file_bytes_sha256": completed_sha,
            })

        # ====================================================
        # 7. Finish this candidate before the next one
        # ====================================================

        c009_require(
            len(new_scores) == 50
            and set(new_scores) == set(item["missing50"]),
            "CANDIDATE_INCOMPLETE",
        )

        all_scores = {
            **item["saved_scores"],
            **new_scores,
        }

        c009_require(
            len(all_scores) == 100
            and set(all_scores) == set(item["hybrid100"])
            and all(math.isfinite(v) for v in all_scores.values()),
            "LOCAL100_SCORE_SET_CONFLICT",
        )

        local100_order = sorted(
            item["hybrid100"],
            key=lambda chunk_id: (
                -all_scores[chunk_id],
                chunk_id,
            ),
        )

        c009_require(
            len(local100_order) == len(set(local100_order)) == 100,
            "LOCAL100_RANK_CONFLICT",
        )

        candidate_report = {
            "cell": "NB22E-v4:C009",
            "status": "CANDIDATE_PASS",
            "candidate_id": candidate_id,
            "scope_sha256": c009_scope_sha,
            "input_sha256": candidate_input_sha,
            "question_sha256": item["question_sha256"],
            "original_hybrid100_order": item["hybrid100"],
            "stored_pool50_order": item["pool50"],
            "stored_pool50_scores": item["saved_scores"],
            "new_scores_by_chunk_id": new_scores,
            "all100_scores_by_chunk_id": all_scores,
            "local100_order": local100_order,
            "local100_order_sha256": hashlib.sha256(
                c009_canonical(local100_order)
            ).hexdigest(),
            "new_pair_count": 50,
            "reused_pool50_pair_count": 50,
            "new_model_calls_this_execution": candidate_new_calls,
            "new_pairs_this_execution": candidate_new_pairs,
            "batch_receipts": batch_receipts,
            "elapsed_seconds_this_execution": round(
                time.perf_counter() - candidate_started,
                3,
            ),
            "process_peak_rss_gib_since_kernel_start": c009_rss_gib(),
            "historical_local100_rank_equivalence": "NOT_ASSERTED",
            "target_release_gate": "BLOCKED",
            "qdrant_queries": 0,
            "api_calls": 0,
        }

        candidate_sha = c009_put_once(
            candidate_path,
            candidate_report,
        )

        c009_candidate_results.append({
            "candidate_id": candidate_id,
            "status": "CANDIDATE_PASS",
            "pairs_in_receipt": 50,
            "new_pairs_this_run": candidate_new_pairs,
            "path": str(candidate_path),
            "file_bytes_sha256": candidate_sha,
        })

        print(
            "C009 candidate:",
            len(c009_candidate_results),
            "/4 PASS;",
            "new pairs:",
            candidate_new_pairs,
            "peak RSS GiB:",
            c009_rss_gib(),
            flush=True,
        )

except Exception as error:
    c009_blocker = {
        "type": type(error).__name__,
        "detail": str(error)[:700],
        "candidate_id": c009_current_candidate,
    }

finally:
    c009_session = None

# ============================================================
# 8. Persist final Stage 2 status
# ============================================================

c009_success = (
    c009_blocker is None
    and len(c009_candidate_results) == 4
    and all(
        row["status"] in (
            "CANDIDATE_PASS",
            "CANDIDATE_PASS_REUSED",
        )
        for row in c009_candidate_results
    )
    and sum(
        row["pairs_in_receipt"]
        for row in c009_candidate_results
    ) == 200
)

c009_summary = {
    "cell": "NB22E-v4:C009",
    "status": (
        "STAGE2_PASS" if c009_success else "STAGE2_BLOCKED"
    ),
    "scope_sha256": c009_scope_sha,
    "approved_pilot_plan_file_sha256": c009_plan_sha,
    "verified_stage1_file_sha256": c009_stage1_sha,
    "logical_index_sha256": c009_index_sha,
    "candidate_ids": c009_ids,
    "candidate_results": c009_candidate_results,
    "candidates_completed": len(c009_candidate_results),
    "new_pairs_max_authorized": 200,
    "new_pairs_attempted_this_execution": (
        c009_pairs_attempted_this_run
    ),
    "new_pairs_completed_this_execution": (
        c009_pairs_completed_this_run
    ),
    "pairs_reused_from_stage2_checkpoints": (
        c009_pairs_reused_this_run
    ),
    "complete_stage2_new_pair_receipts": sum(
        row["pairs_in_receipt"]
        for row in c009_candidate_results
    ),
    "historical_pool50_scores_reused_if_all_complete": (
        200 if c009_success else None
    ),
    "model_inference_calls_this_execution": (
        c009_model_calls_this_run
    ),
    "qdrant_queries_this_execution": 0,
    "api_calls": 0,
    "elapsed_seconds_this_execution": round(
        time.perf_counter() - c009_started,
        3,
    ),
    "process_peak_rss_gib_since_kernel_start": c009_rss_gib(),
    "blocker": c009_blocker,
    "repository_modified": False,
    "version_identity_final": False,
    "target_release_gate": "BLOCKED",
    "full_population_authorized": False,
    "next": (
        "PILOT_EVIDENCE_REVIEW_AND_MEASURED_CONTINUATION_ESTIMATE"
        if c009_success
        else "REPAIR_REPORTED_BLOCKER_NO_UNVERIFIED_RETRIES"
    ),
}

summary_bytes = c009_canonical(c009_summary) + b"\n"
summary_sha = hashlib.sha256(summary_bytes).hexdigest()

summary_path = c009_dir / (
    "C009.stage2_summary." + summary_sha + ".json"
)

if not summary_path.exists():
    temporary = None

    try:
        with tempfile.NamedTemporaryFile(
            "wb",
            dir=c009_dir,
            prefix=".C009-summary-",
            suffix=".tmp",
            delete=False,
        ) as stream:
            temporary = Path(stream.name)
            stream.write(summary_bytes)
            stream.flush()
            os.fsync(stream.fileno())

        try:
            os.link(temporary, summary_path)
        except FileExistsError:
            pass

    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

c009_require(
    summary_path.read_bytes() == summary_bytes,
    "SUMMARY_RELOAD_CONFLICT",
)

NB22E_V4_STAGE2_RESULT = {
    **c009_summary,
    "checkpoint": {
        "path": str(summary_path),
        "file_bytes_sha256": summary_sha,
        "size_bytes": len(summary_bytes),
    },
}

print(json.dumps(
    {
        key: value
        for key, value in NB22E_V4_STAGE2_RESULT.items()
        if key != "candidate_results"
    },
    ensure_ascii=False,
    indent=2,
))

RuntimeError: C009_FROZEN_PIN_CONFLICT

<div dir="rtl" align="right">

### NB22E-v4:C010 — R11.2: ممیزی Pilot و برآورد ادامه

**هدف:** اعتبارسنجی مستقل خروجی‌های موفق C007 تا C009، بدون اجرای مجدد Retrieval یا مدل.

**کنترل‌ها:**
۱. Reload و تطبیق SHA-256 طرح مصوب C006، کنترل Index در C007 و نتایج دو Stage.
۲. بررسی چهار Checkpoint نهایی سؤال‌ها.
۳. اعتبارسنجی ۵۲ رسید شروع و ۵۲ رسید تکمیل Batch.
۴. تطبیق ۲۰۰ امتیاز جدید ONNX با خروجی‌های ذخیره‌شده.
۵. تأیید حفظ ۲۰۰ امتیاز تاریخی Pool50.
۶. بازسازی ترتیب Local100 از امتیازها و کنترل قانون مرتب‌سازی.
۷. استخراج زمان و حافظهٔ اندازه‌گیری‌شده و تهیهٔ برآورد مشروط برای ۱۱۱۶ سؤال.
۸. ثبت گزارش تغییرناپذیر و تأیید Reload آن.

**محدودیت:** هیچ Query، مدل، API یا اجرای جمعیتی انجام نمی‌شود. هویت نسخهٔ جدید نهایی نمی‌شود و Gate پذیرش R11.2 مسدود باقی می‌ماند.

**خروجی:** `C010.pilot_audit.<SHA256>.json` خارج از مخزن.

</div>

In [ ]:
# NB22E-v4 | C010 | R11.2
# Audit approved Pilot receipts; no retrieval, ONNX inference, API or repository writes.

def c010_require(ok, code):
    if not ok:
        raise RuntimeError("C010_" + code)

c010_root = c002_root.resolve()
c010_out = c003_out.resolve()
c010_stage2_dir = (c010_out / "C009_stage2").resolve()

c010_require(
    NB22E_V4_READ_ONLY is True
    and c010_out.is_dir()
    and not c010_out.is_relative_to(c010_root)
    and c010_stage2_dir.parent == c010_out
    and c010_stage2_dir.is_dir(),
    "EXTERNAL_OUTPUT_BOUNDARY",
)

c010_pins = {
    "plan": "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213",
    "stage1": "ba56c485f014a2eb149b63bad85eec6c790d2eb8ebf36f4f8348a9768f7955c6",
    "stage2": "933a0bfdbe38991f9d6715d97e8b5517f7815a6f67a3e7562c4df5d67ecf8174",
    "scope": "fce1de6a63a9614fa05fd23014f8d7353f0767724d9351ea23ea30199f2cc6d6",
    "index": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
}

# ============================================================
# 1. Independently reload pinned stage artifacts
# ============================================================

def c010_read(path_value, expected_sha, allowed_parent):
    path = Path(path_value).resolve()
    c010_require(
        path.parent == allowed_parent and path.is_file(),
        "UNSAFE_RECEIPT",
    )

    before = path.stat()
    raw = path.read_bytes()
    after = path.stat()

    c010_require(
        before.st_size == after.st_size == len(raw)
        and before.st_mtime_ns == after.st_mtime_ns
        and hashlib.sha256(raw).hexdigest() == expected_sha,
        "RECEIPT_BYTES_OR_SHA_CONFLICT:" + path.name,
    )
    return json.loads(raw)

c010_require(
    NB22E_V4_PILOT_PLAN["file_bytes_sha256"] == c010_pins["plan"]
    and NB22E_V4_STAGE1_RESULT["checkpoint"]["file_bytes_sha256"]
    == c010_pins["stage1"]
    and NB22E_V4_STAGE2_RESULT["checkpoint"]["file_bytes_sha256"]
    == c010_pins["stage2"],
    "SUPERVISOR_OR_STAGE_PIN_CONFLICT",
)

c010_plan = c010_read(
    NB22E_V4_PILOT_PLAN["path"],
    c010_pins["plan"],
    c010_out,
)
c010_s1 = c010_read(
    NB22E_V4_STAGE1_RESULT["checkpoint"]["path"],
    c010_pins["stage1"],
    c010_out,
)
c010_s2 = c010_read(
    NB22E_V4_STAGE2_RESULT["checkpoint"]["path"],
    c010_pins["stage2"],
    c010_stage2_dir,
)
c010_index = c010_read(
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_path"],
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"],
    c010_out,
)

c010_ids = list(NB22E_V4_PILOT_PLAN["candidate_ids"])

c010_require(
    c010_s2 == {
        key: value
        for key, value in NB22E_V4_STAGE2_RESULT.items()
        if key != "checkpoint"
    }
    and len(c010_ids) == len(set(c010_ids)) == 4
    and c010_plan["population"]["new_candidate_count"] == 1116
    and [
        item["candidate_id"]
        for item in c010_plan["population"]["pilot_candidates"]
    ] == c010_ids
    and c010_index["status"] == "LOGICAL_INDEX_HASH_MATCH"
    and c010_index["observed_logical_index_sha256"]
    == c010_pins["index"]
    and c010_index["point_count"] == 3488
    and c010_s1["status"] == "STAGE1_PASS"
    and c010_s1["candidate_ids"] == c010_ids
    and c010_s1["qdrant_queries_attempted"] == 4
    and c010_s1["stage2_authorized"] is True
    and c010_s1["logical_index_hash_match_checkpoint_sha256"]
    == NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_sha256"]
    and c010_s2["status"] == "STAGE2_PASS"
    and c010_s2["candidate_ids"] == c010_ids
    and c010_s2["scope_sha256"] == c010_pins["scope"]
    and c010_s2["approved_pilot_plan_file_sha256"]
    == c010_pins["plan"]
    and c010_s2["verified_stage1_file_sha256"]
    == c010_pins["stage1"]
    and c010_s2["logical_index_sha256"]
    == c010_pins["index"]
    and c010_s2["target_release_gate"] == "BLOCKED"
    and c010_s2["full_population_authorized"] is False
    and c010_s2["version_identity_final"] is False
    and c010_s2["blocker"] is None
    and c010_s2["repository_modified"] is False
    and c010_s2["api_calls"] == 0
    and c010_s2["qdrant_queries_this_execution"] == 0
    and c010_s2["new_pairs_attempted_this_execution"] == 200
    and c010_s2["new_pairs_completed_this_execution"] == 200
    and c010_s2["pairs_reused_from_stage2_checkpoints"] == 0
    and c010_s2["complete_stage2_new_pair_receipts"] == 200
    and c010_s2["historical_pool50_scores_reused_if_all_complete"] == 200
    and c010_s2["model_inference_calls_this_execution"] == 52
    and c010_s2["candidates_completed"] == 4,
    "CHECKPOINT_CHAIN_OR_AGGREGATE_CONFLICT",
)

# ============================================================
# 2. Verify each candidate and all 52 batch receipts
# ============================================================

c010_verified = []
c010_batch_seconds = []
c010_expected_started = set()
c010_expected_completed = set()
c010_expected_candidates = set()
c010_total_new_pairs = 0
c010_total_old_scores = 0

for pos, receipt in enumerate(c010_s2["candidate_results"]):
    candidate_id = c010_ids[pos]
    candidate_path = Path(receipt["path"]).resolve()

    c010_require(
        receipt["candidate_id"] == candidate_id
        and receipt["status"] == "CANDIDATE_PASS"
        and receipt["pairs_in_receipt"] == 50
        and receipt["new_pairs_this_run"] == 50,
        "CANDIDATE_SUMMARY_CONFLICT",
    )

    candidate, observed_sha = c009_read_record(candidate_path)
    c010_expected_candidates.add(candidate_path)

    c010_require(
        observed_sha == receipt["file_bytes_sha256"]
        and candidate["status"] == "CANDIDATE_PASS"
        and candidate["scope_sha256"] == c010_pins["scope"]
        and candidate["candidate_id"] == candidate_id
        and candidate["question_sha256"]
        == c002_questions[candidate_id]["question_sha256"]
        and candidate["original_hybrid100_order"]
        == c010_s1["results"][pos]["observed_hybrid100_order"]
        and c010_s1["results"][pos]["stage1_status"] == "PASS"
        and c010_s1["results"][pos]["full_order_matches"] is True
        and c010_s1["results"][pos]["pool50_membership_matches"] is True
        and candidate["new_pair_count"] == 50
        and candidate["reused_pool50_pair_count"] == 50
        and candidate["new_model_calls_this_execution"] == 13
        and candidate["new_pairs_this_execution"] == 50
        and candidate["historical_local100_rank_equivalence"]
        == "NOT_ASSERTED"
        and len(candidate["batch_receipts"]) == 13,
        "CANDIDATE_VERSION_OR_COUNT_CONFLICT:" + candidate_id,
    )

    hybrid = candidate["original_hybrid100_order"]
    old_order = candidate["stored_pool50_order"]
    old_scores = candidate["stored_pool50_scores"]
    new_scores = candidate["new_scores_by_chunk_id"]
    all_scores = candidate["all100_scores_by_chunk_id"]

    c010_require(
        len(hybrid) == len(set(hybrid)) == 100
        and len(old_order) == len(set(old_order)) == 50
        and set(old_order) == set(hybrid[:50]) == set(old_scores)
        and set(new_scores) == set(hybrid[50:])
        and len(new_scores) == 50
        and all_scores == {**old_scores, **new_scores}
        and all(math.isfinite(score) for score in all_scores.values())
        and candidate["local100_order"] == sorted(
            hybrid,
            key=lambda chunk_id: (-all_scores[chunk_id], chunk_id),
        )
        and candidate["local100_order_sha256"] == hashlib.sha256(
            c009_canonical(candidate["local100_order"])
        ).hexdigest(),
        "LOCAL100_SCORE_OR_RANK_CONFLICT:" + candidate_id,
    )

    reassembled = {}
    candidate_batch_seconds = 0.0

    for batch_pos, batch_receipt in enumerate(
        candidate["batch_receipts"]
    ):
        start = batch_pos * 4
        expected_ids = hybrid[50:][start:start + 4]
        completed_path = Path(batch_receipt["path"]).resolve()
        started_path = completed_path.with_name(
            completed_path.name.replace(
                "C009.completed.",
                "C009.started.",
                1,
            )
        )

        c010_require(
            completed_path.name.startswith("C009.completed.")
            and batch_receipt["batch_start"] == start
            and batch_receipt["count"] == len(expected_ids)
            and completed_path not in c010_expected_completed
            and started_path not in c010_expected_started,
            "BATCH_PATH_OR_OFFSET_CONFLICT",
        )

        started, _ = c009_read_record(started_path)
        completed, actual_sha = c009_read_record(completed_path)

        c010_expected_started.add(started_path)
        c010_expected_completed.add(completed_path)

        c010_require(
            actual_sha == batch_receipt["file_bytes_sha256"]
            and started["status"] == "STARTED_NOT_VERIFIED"
            and started["scope_sha256"] == c010_pins["scope"]
            and started["input_sha256"] == candidate["input_sha256"]
            and started["candidate_id"] == candidate_id
            and started["batch_start"] == start
            and started["chunk_ids"] == expected_ids
            and started["score_count_expected"] == len(expected_ids)
            and completed["status"] == "BATCH_PASS"
            and completed["scope_sha256"] == c010_pins["scope"]
            and completed["input_sha256"] == candidate["input_sha256"]
            and completed["candidate_id"] == candidate_id
            and completed["chunk_ids"] == expected_ids
            and completed["model_inference_calls"] == 1
            and completed["qdrant_queries"] == 0
            and completed["api_calls"] == 0,
            "BATCH_RECEIPT_CHAIN_CONFLICT",
        )

        scores = completed["scores"]
        seconds = float(completed["elapsed_seconds"])

        c010_require(
            len(scores) == len(expected_ids)
            and all(math.isfinite(float(value)) for value in scores)
            and math.isfinite(seconds)
            and seconds >= 0,
            "BATCH_SCORE_OR_TIMING_CONFLICT",
        )

        reassembled.update(zip(expected_ids, scores))
        c010_batch_seconds.append(seconds)
        candidate_batch_seconds += seconds
        c010_total_new_pairs += len(scores)

    c010_require(
        reassembled == new_scores
        and len(reassembled) == 50,
        "BATCH_SCORE_REASSEMBLY_CONFLICT",
    )

    c010_total_old_scores += len(old_scores)

    c010_verified.append({
        "candidate_id": candidate_id,
        "status": "VERIFIED",
        "hybrid100_count": 100,
        "reused_pool50_scores": 50,
        "new_onnx_scores": 50,
        "verified_batches": 13,
        "local100_rank_count": 100,
        "local100_order_sha256": (
            candidate["local100_order_sha256"]
        ),
        "candidate_receipt_sha256": observed_sha,
        "batch_model_seconds_sum": round(
            candidate_batch_seconds, 4
        ),
    })

# ============================================================
# 3. Confirm exact scoped checkpoint coverage
# ============================================================

c010_short_scope = c010_pins["scope"][:16]

def c010_scope_files(prefix):
    return {
        path.resolve()
        for path in c010_stage2_dir.glob(
            prefix + "*." + c010_short_scope + ".json"
        )
    }

c010_require(
    len(c010_verified) == 4
    and len(c010_batch_seconds) == 52
    and c010_total_new_pairs == 200
    and c010_total_old_scores == 200
    and c010_scope_files("C009.started.")
    == c010_expected_started
    and c010_scope_files("C009.completed.")
    == c010_expected_completed
    and c010_scope_files("C009.candidate.")
    == c010_expected_candidates,
    "INCOMPLETE_OR_EXTRA_SCOPED_RECEIPTS",
)

# ============================================================
# 4. Measured workload and conditional projection
# ============================================================

c010_stage1_seconds = float(c010_s1["elapsed_seconds"])
c010_stage2_seconds = float(
    c010_s2["elapsed_seconds_this_execution"]
)
c010_batch_seconds_sum = sum(c010_batch_seconds)
c010_peak_rss = float(
    c010_s2["process_peak_rss_gib_since_kernel_start"]
)

c010_require(
    0 < c010_stage1_seconds
    and 0 < c010_stage2_seconds
    and 0 < c010_peak_rss
    and c010_batch_seconds_sum <= c010_stage2_seconds + 0.5,
    "PILOT_MEASUREMENT_CONFLICT",
)

c010_population = 1116

c010_projection = {
    "method": "CONDITIONAL_LINEAR_PROJECTION_FROM_4_QUESTIONS",
    "population_if_approved": c010_population,
    "stage1_queries_if_same_contract_holds": c010_population,
    "additional_onnx_pairs_if_all_have_pool50": (
        c010_population * 50
    ),
    "additional_onnx_batches_if_all_have_pool50": (
        c010_population * 13
    ),
    "stage1_linear_hours": round(
        (c010_stage1_seconds / 4 * c010_population) / 3600,
        3,
    ),
    "stage2_model_run_only_linear_hours": round(
        (
            c010_batch_seconds_sum
            / 52
            * c010_population
            * 13
        ) / 3600,
        3,
    ),
    "stage2_end_to_end_linear_hours": round(
        (c010_stage2_seconds / 4 * c010_population) / 3600,
        3,
    ),
    "both_stages_linear_hours": round(
        (
            (c010_stage1_seconds + c010_stage2_seconds)
            / 4
            * c010_population
        ) / 3600,
        3,
    ),
    "not_a_runtime_guarantee": True,
    "excludes_generation_and_all_other_R11_2_work": True,
    "not_authorized_to_run_full_population": True,
}

# ============================================================
# 5. Save immutable audit outside the repository
# ============================================================

c010_report = {
    "cell": "NB22E-v4:C010",
    "status": "PILOT_EVIDENCE_AUDIT_PASS",
    "approved_plan_sha256": c010_pins["plan"],
    "index_logical_sha256": c010_pins["index"],
    "stage1_summary_sha256": c010_pins["stage1"],
    "stage2_summary_sha256": c010_pins["stage2"],
    "stage2_scope_sha256": c010_pins["scope"],
    "candidates_verified": c010_verified,
    "started_batch_receipts_verified": 52,
    "completed_batch_receipts_verified": 52,
    "new_onnx_pair_receipts_verified": c010_total_new_pairs,
    "historical_pool50_scores_preserved": c010_total_old_scores,
    "stage1_queries_observed": 4,
    "stage2_onnx_calls_observed": 52,
    "stage1_elapsed_seconds_observed": c010_stage1_seconds,
    "stage2_elapsed_seconds_observed": c010_stage2_seconds,
    "stage2_batch_model_seconds_total_observed": round(
        c010_batch_seconds_sum, 4
    ),
    "stage2_batch_model_seconds_median_observed": round(
        statistics.median(c010_batch_seconds), 4
    ),
    "process_peak_rss_gib_since_kernel_start": c010_peak_rss,
    "peak_rss_not_incremental_stage2_only": True,
    "continuation_projection": c010_projection,
    "historical_target_identity_kept_separate": (
        "8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065"
    ),
    "new_population_identity_finalized": False,
    "R11_2_acceptance": "BLOCKED",
    "target_release_gate": "BLOCKED",
    "full_population_run_authorized": False,
    "repository_writes": 0,
    "qdrant_queries_by_C010": 0,
    "model_inference_calls_by_C010": 0,
    "api_calls_by_C010": 0,
    "next": "SUPERVISOR_VERSIONED_POPULATION_AND_CONTINUATION_DECISION",
}

c010_bytes = c009_canonical(c010_report) + b"\n"
c010_file_sha = hashlib.sha256(c010_bytes).hexdigest()

c010_target = c010_out / (
    "C010.pilot_audit." + c010_file_sha + ".json"
)

if not c010_target.is_file():
    c010_temp = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="wb",
            dir=c010_out,
            prefix=".C010-",
            suffix=".tmp",
            delete=False,
        ) as stream:
            c010_temp = Path(stream.name)
            stream.write(c010_bytes)
            stream.flush()
            os.fsync(stream.fileno())

        try:
            os.link(c010_temp, c010_target)
        except FileExistsError:
            pass

    finally:
        if c010_temp is not None:
            c010_temp.unlink(missing_ok=True)

c010_require(
    c010_target.read_bytes() == c010_bytes,
    "AUDIT_ARTIFACT_RELOAD_CONFLICT",
)

NB22E_V4_PILOT_AUDIT = {
    **c010_report,
    "checkpoint": {
        "path": str(c010_target),
        "file_bytes_sha256": c010_file_sha,
        "size_bytes": len(c010_bytes),
    },
}

print(json.dumps({
    "cell": NB22E_V4_PILOT_AUDIT["cell"],
    "status": NB22E_V4_PILOT_AUDIT["status"],
    "candidate_count": 4,
    "stage1_queries": 4,
    "stage2_new_pairs": c010_total_new_pairs,
    "historical_pool50_scores": c010_total_old_scores,
    "batch_receipts_verified": 52,
    "inference_calls_observed": 52,
    "stage1_seconds": c010_stage1_seconds,
    "stage2_seconds": c010_stage2_seconds,
    "process_peak_rss_gib_since_kernel_start": c010_peak_rss,
    "continuation_projection": c010_projection,
    "new_population_identity_finalized": False,
    "target_release_gate": "BLOCKED",
    "checkpoint": NB22E_V4_PILOT_AUDIT["checkpoint"],
}, ensure_ascii=False, indent=2))

{
  "cell": "NB22E-v4:C010",
  "status": "PILOT_EVIDENCE_AUDIT_PASS",
  "candidate_count": 4,
  "stage1_queries": 4,
  "stage2_new_pairs": 200,
  "historical_pool50_scores": 200,
  "batch_receipts_verified": 52,
  "inference_calls_observed": 52,
  "stage1_seconds": 3.513,
  "stage2_seconds": 32.2,
  "process_peak_rss_gib_since_kernel_start": 1.019,
  "continuation_projection": {
    "method": "CONDITIONAL_LINEAR_PROJECTION_FROM_4_QUESTIONS",
    "population_if_approved": 1116,
    "stage1_queries_if_same_contract_holds": 1116,
    "additional_onnx_pairs_if_all_have_pool50": 55800,
    "additional_onnx_batches_if_all_have_pool50": 14508,
    "stage1_linear_hours": 0.272,
    "stage2_model_run_only_linear_hours": 2.214,
    "stage2_end_to_end_linear_hours": 2.496,
    "both_stages_linear_hours": 2.768,
    "not_a_runtime_guarantee": true,
    "excludes_generation_and_all_other_R11_2_work": true,
    "not_authorized_to_run_full_population": true
  },
  "new_population_identity_finalized":

<div dir="rtl" align="right">

### NB22E-v4:C011 — R11.2: ثبت هویت مستقل جمعیتی و Manifest اجرایی

**هدف:** ثبت هویت جدید برای جمعیت دقیق ۱۱۱۶ سؤال و تثبیت مرز اجرای مرحله‌ای با مجوز ناظر.

**کنترل‌ها:**

۱. تطبیق SHA-256 بایتی Artifact ممیزی C010 و زنجیرهٔ رسیدهای C006 تا C009.

۲. ثبت همهٔ ۱۱۱۶ شناسه همراه با هش متن سؤال و ترتیب قطعی عضویت.

۳. حفظ چهار سؤال Pilot به‌عنوان نتایج موفق قابل‌استفادهٔ مجدد، بدون قرار دادن آن‌ها در صف اجرای تازه.

۴. تعریف ۴۵ دستهٔ حداکثر ۲۵سؤالی برای ۱۱۱۲ سؤال باقی‌مانده.

۵. اتصال هویت مستقل به ورودی‌ها، قراردادهای Producer، Index فریز‌شده، بردارهای Query، ONNX، فایل‌های Tokenizer، نسخه‌های Runtime و تنظیمات واقعی.

۶. ایجاد SHA-256 مستقل برای قرارداد مرز اجرای جمعیتی؛ جدا از شناسهٔ تاریخی و هش Artifactهای رتبهٔ آینده.

۷. نوشتن Manifest تغییرناپذیر خارج مخزن و کنترل Reload آن.

**محدودیت:** هیچ Query، استنتاج مدل، API یا اجرای جمعیتی در این سل انجام نمی‌شود.

**وضعیت پذیرش:** R11.2 همچنان BLOCKED است. مجوز اجرای Stage 1/2، به‌معنای مجوز مراحل بعدی Rank، Context یا Generation نیست.

</div>

In [ ]:
# NB22E-v4 | C011 | R11.2
# Register immutable 1116-member population boundary; no model, retrieval, API or repository writes.

def c011_require(ok, code):
    if not ok:
        raise RuntimeError("C011_" + code)

c011_root = c002_root.resolve()
c011_out = c003_out.resolve()
c011_pilot_dir = (c011_out / "C009_stage2").resolve()

c011_require(
    NB22E_V4_READ_ONLY is True
    and c011_root.name == "RAG_finance"
    and c011_out.is_dir()
    and not c011_out.is_relative_to(c011_root)
    and c011_pilot_dir.is_dir()
    and c011_pilot_dir.parent == c011_out,
    "UNSAFE_BOUNDARY",
)

c011_expected = {
    "plan": "fbfa555c8d3ab8e9f54e05dee701349ee63bc3c2a5aaee41d80e6d568e1c6213",
    "index_receipt": "6e61431607fac16faa1295455670d016da88a3a9cf5e8951f242353232326d79",
    "stage1": "ba56c485f014a2eb149b63bad85eec6c790d2eb8ebf36f4f8348a9768f7955c6",
    "stage2": "933a0bfdbe38991f9d6715d97e8b5517f7815a6f67a3e7562c4df5d67ecf8174",
    "audit": "ae2360b309dc5ed6fe44c1735eecdb0407f96324ca394776739bd6fddbccc6a0",
    "logical_index": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
    "onnx": "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b",
    "historical": "8d5875bfee0cfe5f2dc0ce9bdf67e5f14301191b60891047a630c3fe65442065",
}

def c011_canon(obj):
    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")

def c011_sha_file(path, size_limit=None):
    path = Path(path)
    c011_require(path.is_file(), "FILE_MISSING:" + path.name)
    before = path.stat()

    if size_limit is not None:
        c011_require(
            before.st_size <= size_limit,
            "FILE_OVERSIZE:" + path.name,
        )

    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    after = path.stat()

    c011_require(
        (before.st_size, before.st_mtime_ns)
        == (after.st_size, after.st_mtime_ns),
        "CHANGED_DURING_HASH:" + path.name,
    )

    return digest.hexdigest(), before.st_size

def c011_load_receipt(path_value, expected_sha, parent):
    path = Path(path_value).resolve()

    c011_require(
        path.parent == parent and path.is_file(),
        "RECEIPT_PATH_UNSAFE",
    )

    actual_sha, size = c011_sha_file(
        path,
        16 * 1024 * 1024,
    )

    c011_require(
        actual_sha == expected_sha,
        "RECEIPT_SHA_CONFLICT:" + path.name,
    )

    return json.loads(path.read_bytes()), {
        "path": str(path),
        "file_bytes_sha256": actual_sha,
        "size_bytes": size,
    }

# ============================================================
# 1. Independently verify C006 through C010
# ============================================================

c011_plan, c011_plan_pin = c011_load_receipt(
    NB22E_V4_PILOT_PLAN["path"],
    c011_expected["plan"],
    c011_out,
)

c011_index, c011_index_pin = c011_load_receipt(
    NB22E_V4_LOGICAL_INDEX_CHECK["checkpoint_path"],
    c011_expected["index_receipt"],
    c011_out,
)

c011_stage1, c011_stage1_pin = c011_load_receipt(
    NB22E_V4_STAGE1_RESULT["checkpoint"]["path"],
    c011_expected["stage1"],
    c011_out,
)

c011_stage2, c011_stage2_pin = c011_load_receipt(
    NB22E_V4_STAGE2_RESULT["checkpoint"]["path"],
    c011_expected["stage2"],
    c011_pilot_dir,
)

c011_audit, c011_audit_pin = c011_load_receipt(
    NB22E_V4_PILOT_AUDIT["checkpoint"]["path"],
    c011_expected["audit"],
    c011_out,
)

c011_pilot_ids = [
    item["candidate_id"]
    for item in c011_plan["population"]["pilot_candidates"]
]

c011_population_ids = sorted(c002_questions)

c011_require(
    c011_audit["status"] == "PILOT_EVIDENCE_AUDIT_PASS"
    and c011_audit["stage2_summary_sha256"]
    == c011_expected["stage2"]
    and c011_audit["stage1_summary_sha256"]
    == c011_expected["stage1"]
    and c011_audit["approved_plan_sha256"]
    == c011_expected["plan"]
    and c011_audit["started_batch_receipts_verified"] == 52
    and c011_audit["completed_batch_receipts_verified"] == 52
    and c011_audit["new_onnx_pair_receipts_verified"] == 200
    and c011_audit["historical_pool50_scores_preserved"] == 200
    and c011_index["status"] == "LOGICAL_INDEX_HASH_MATCH"
    and c011_index["observed_logical_index_sha256"]
    == c011_expected["logical_index"]
    and c011_stage1["status"] == "STAGE1_PASS"
    and c011_stage1["candidate_ids"] == c011_pilot_ids
    and c011_stage1["stage2_authorized"] is True
    and c011_stage2["status"] == "STAGE2_PASS"
    and c011_stage2["candidate_ids"] == c011_pilot_ids
    and c011_stage2["candidates_completed"] == 4
    and c011_stage2["complete_stage2_new_pair_receipts"] == 200
    and c011_plan["population"]["new_candidate_count"] == 1116
    and len(c011_population_ids) == 1116
    and len(set(c011_population_ids)) == 1116
    and len(c011_pilot_ids) == len(set(c011_pilot_ids)) == 4
    and set(c011_pilot_ids).issubset(c011_population_ids),
    "PILOT_AUDIT_OR_MEMBERSHIP_MISMATCH",
)

# ============================================================
# 2. Exact membership and exclusion of completed Pilot work
# ============================================================

c011_membership = []

for cid in c011_population_ids:
    record = c002_questions[cid]
    question = record["question"]
    recorded_sha = record["question_sha256"]

    c011_require(
        isinstance(cid, str)
        and bool(cid)
        and isinstance(question, str)
        and bool(question.strip())
        and hashlib.sha256(
            question.encode("utf-8")
        ).hexdigest() == recorded_sha,
        "QUESTION_BYTES_OR_SHA_CONFLICT:" + cid,
    )

    c011_membership.append({
        "candidate_id": cid,
        "question_sha256": recorded_sha,
    })

c011_membership_sha = hashlib.sha256(
    c011_canon(c011_membership)
).hexdigest()

c011_pilot_id_set = set(c011_pilot_ids)

c011_remaining = [
    cid
    for cid in c011_population_ids
    if cid not in c011_pilot_id_set
]

c011_batches = [
    {
        "batch_number": offset // 25 + 1,
        "candidate_ids": c011_remaining[offset:offset + 25],
    }
    for offset in range(0, len(c011_remaining), 25)
]

c011_require(
    len(c011_remaining) == 1112
    and sum(
        len(batch["candidate_ids"])
        for batch in c011_batches
    ) == 1112
    and len(c011_batches) == 45
    and all(
        1 <= len(batch["candidate_ids"]) <= 25
        for batch in c011_batches
    ),
    "BATCH_MEMBERSHIP_CONFLICT",
)

# ============================================================
# 3. Bind and recheck frozen source and vector file bytes
# ============================================================

c011_source_pins = {}

for label, record in sorted(
    c002_summary["source_checks"].items()
):
    c011_require(
        record.get("status") == "FILE_BYTES_VERIFIED",
        "SOURCE_NOT_PINNED:" + label,
    )

    c011_source_pins[label] = {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }

c011_input_pins = {}

for label, record in sorted(
    c002_summary["input_file_checks"].items()
):
    c011_require(
        record.get("status") == "FILE_BYTES_VERIFIED",
        "INPUT_NOT_PINNED:" + label,
    )

    c011_input_pins[label] = {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }

c011_vector_pins = {}

for label, record in sorted(
    c002_summary["query_vector_precheck"][
        "cache_file_pointers"
    ].items()
):
    c011_require(
        isinstance(record, dict)
        and isinstance(record.get("path"), str)
        and isinstance(record.get("file_bytes_sha256"), str),
        "VECTOR_PIN_INVALID:" + label,
    )

    c011_vector_pins[label] = {
        "path": record["path"],
        "file_bytes_sha256": record["file_bytes_sha256"],
    }

c011_all_pins = {
    **c011_source_pins,
    **c011_input_pins,
    **{
        "VECTOR_" + key: value
        for key, value in c011_vector_pins.items()
    },
}

for label, record in c011_all_pins.items():
    relative = Path(record["path"])
    path = (c011_root / relative).resolve()

    c011_require(
        not relative.is_absolute()
        and path.is_relative_to(c011_root),
        "INPUT_PATH_ESCAPE:" + label,
    )

    actual_sha, _ = c011_sha_file(path)

    c011_require(
        actual_sha == record["file_bytes_sha256"],
        "INPUT_CHANGED:" + label,
    )

c011_require(
    c002_summary["target_identity_sha256_original"]
    == c011_expected["historical"]
    and c002_summary["producer_contract_checks"]
    and all(
        value is True
        for value in c002_summary[
            "producer_contract_checks"
        ].values()
    )
    and c002_summary["query_vector_precheck"][
        "vector_candidate_id_coverage"
    ] == 1116,
    "FROZEN_LINEAGE_OR_CONTRACT_CONFLICT",
)

# ============================================================
# 4. Actual ONNX, Tokenizer and Runtime identities
# ============================================================

c011_snapshot = Path(c004_snapshot).expanduser()
c011_model_alias = Path(c004_model_path).expanduser()

c011_require(
    c011_model_alias.is_file()
    and Path(
        os.path.abspath(
            str(c011_model_alias.parent.parent)
        )
    ) == Path(
        os.path.abspath(str(c011_snapshot))
    )
    and c011_snapshot.name
    == "1427fd652930e4ba29e8149678df786c240d8825",
    "ONNX_SNAPSHOT_CONTRACT",
)

c011_model_actual, c011_model_bytes = c011_sha_file(
    c011_model_alias
)

c011_require(
    c011_model_actual == c011_expected["onnx"],
    "ONNX_HASH_CHANGED",
)

c011_tokenizer_files = {}

for filename in (
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
):
    file_path = c011_snapshot / filename

    file_sha, file_bytes = c011_sha_file(
        file_path,
        64 * 1024 * 1024,
    )

    c011_tokenizer_files[filename] = {
        "sha256": file_sha,
        "size_bytes": file_bytes,
    }

c011_tokenizer_identity = {
    "model_snapshot_revision": c011_snapshot.name,
    "tokenizer_class": c005_tokenizer_result[
        "tokenizer_class"
    ],
    "is_fast": c005_tokenizer_result["is_fast"],
    "files": c011_tokenizer_files,
    "local_files_only": True,
    "trust_remote_code": False,
}

c011_tokenizer_sha = hashlib.sha256(
    c011_canon(c011_tokenizer_identity)
).hexdigest()

c011_versions = {
    "python": platform.python_version(),
    "numpy": importlib.metadata.version("numpy"),
    "onnxruntime": importlib.metadata.version("onnxruntime"),
    "qdrant-client": importlib.metadata.version("qdrant-client"),
    "transformers": importlib.metadata.version("transformers"),
    "tokenizers": importlib.metadata.version("tokenizers"),
}

c011_require(
    c011_versions["onnxruntime"] == "1.16.3"
    and c004_docs["release"]["backend"][
        "qdrant_client_version"
    ] == c011_versions["qdrant-client"]
    and c004_docs["release"]["logical_index_sha256"]
    == c011_expected["logical_index"]
    and c011_index["point_count"] == 3488,
    "RUNTIME_OR_INDEX_VERSION_CONFLICT",
)

# ============================================================
# 5. Independent population execution identity
# ============================================================

c011_execution_contract = {
    "schema": "NB22E_V4_R11_2_POPULATION_EXECUTION_BOUNDARY_V1",
    "membership_order": "CANDIDATE_ID_ASCENDING",
    "member_count": 1116,
    "members": c011_membership,
    "membership_sha256": c011_membership_sha,
    "historical_lineage_identity_sha256": (
        c011_expected["historical"]
    ),
    "historical_identity_inherited": False,
    "question_bindings_sha256": (
        c011_plan["population"]["question_binding_sha256"]
    ),
    "transfer_manifest_sha256": (
        c011_plan["population"]["manifest_sha256"]
    ),
    "source_producer_pins": c011_source_pins,
    "input_pins": c011_input_pins,
    "query_vector_cache_pins": c011_vector_pins,
    "producer_contract_checks": (
        c002_summary["producer_contract_checks"]
    ),
    "dense_index": {
        "release_file_sha256": (
            c004_file_checks["release"]["file_bytes_sha256"]
        ),
        "manifest_file_sha256": (
            c004_file_checks["manifest"]["file_bytes_sha256"]
        ),
        "handoff_file_sha256": (
            c004_file_checks["handoff"]["file_bytes_sha256"]
        ),
        "lifecycle_file_sha256": (
            c004_file_checks["lifecycle"]["file_bytes_sha256"]
        ),
        "collection": c005_collection,
        "logical_index_sha256": (
            c011_expected["logical_index"]
        ),
        "point_count": 3488,
        "point_dimension": 384,
        "hash_method_producer": "NB14_CODE_CELL_10",
        "producer_git_blob_sha1": (
            c011_index["producer_git_blob_sha1"]
        ),
    },
    "onnx": {
        "model_id": (
            "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
        ),
        "revision": c011_snapshot.name,
        "model_alias_relative_inside_snapshot": (
            "onnx/model_quint8_avx2.onnx"
        ),
        "model_sha256": c011_model_actual,
        "model_size_bytes": c011_model_bytes,
        "tokenizer_identity_sha256": c011_tokenizer_sha,
        "tokenizer": c011_tokenizer_identity,
    },
    "runtime_versions": c011_versions,
    "frozen_parameters": {
        "dense_candidate_depth": 100,
        "hybrid_fusion": "WEIGHTED_SCORE_FUSION",
        "hybrid_normalization": "minmax",
        "dense_weight": 0.4,
        "sparse_weight": 0.6,
        "sparse_analyzer": "fa_join_zwnj_stopwords_v1",
        "sparse_k1": 1.2,
        "sparse_b": 0.3,
        "local100_onnx_provider": "CPUExecutionProvider",
        "onnx_intra_op_threads": 4,
        "onnx_inter_op_threads": 1,
        "onnx_batch_size": 4,
        "onnx_max_length": 512,
        "candidate_parallelism": 1,
        "rank_tiebreak": "(-score,chunk_id)",
    },
}

c011_boundary_sha = hashlib.sha256(
    c011_canon(c011_execution_contract)
).hexdigest()

c011_require(
    c011_boundary_sha != c011_expected["historical"],
    "IDENTITY_REUSE_FORBIDDEN",
)

# ============================================================
# 6. Assemble the approved execution queue
# ============================================================

c011_document = {
    "cell": "NB22E-v4:C011",
    "status": "POPULATION_BOUNDARY_REGISTERED_APPROVAL_SCOPED",
    "execution_boundary_sha256": c011_boundary_sha,
    "execution_contract": c011_execution_contract,
    "approved_pilot_provenance": {
        "C006": c011_plan_pin,
        "C007": c011_index_pin,
        "C008": c011_stage1_pin,
        "C009": c011_stage2_pin,
        "C010": c011_audit_pin,
        "pilot_ids": c011_pilot_ids,
        "pilot_outcome": (
            "STAGE1_STAGE2_PASS_C010_AUDIT_PASS"
        ),
        "pilot_is_subset_of_population": True,
        "pilot_will_not_be_rerun": True,
    },
    "execution_queue": {
        "remaining_count": 1112,
        "pilot_reused_count": 4,
        "max_batch_size": 25,
        "batch_count": len(c011_batches),
        "batches": c011_batches,
        "fresh_stage1_queries_max": 1112,
        "fresh_stage2_onnx_pairs_max": 55600,
    },
    "status_policy": {
        "no_automatic_retry_of_uncertain_batches": True,
        "no_download_or_install": True,
        "no_new_model_or_hybrid_configuration": True,
        "stage1_must_pass_before_stage2_per_candidate": True,
        "stop_on_first_input_version_rank_score_or_receipt_conflict": True,
        "upstream_repo_read_only": True,
        "rank_artifact_identity_distinct_from_execution_boundary": True,
        "generation_api_allowed": False,
        "derived_order_quota_char_rrf_authorized": False,
        "full_R11_2_quality_gate": "BLOCKED",
        "response_generation_gate": "BLOCKED",
    },
}

# ============================================================
# 7. Immutable atomic write and independent reload
# ============================================================

c011_raw = c011_canon(c011_document) + b"\n"
c011_file_sha = hashlib.sha256(c011_raw).hexdigest()

c011_target = c011_out / (
    "C011.population_boundary."
    + c011_file_sha
    + ".json"
)

if not c011_target.exists():
    c011_tmp = None

    try:
        with tempfile.NamedTemporaryFile(
            "wb",
            dir=c011_out,
            prefix=".C011-",
            suffix=".tmp",
            delete=False,
        ) as stream:
            c011_tmp = Path(stream.name)
            stream.write(c011_raw)
            stream.flush()
            os.fsync(stream.fileno())

        try:
            os.link(c011_tmp, c011_target)
        except FileExistsError:
            pass

    finally:
        if c011_tmp is not None:
            c011_tmp.unlink(missing_ok=True)

c011_require(
    c011_target.read_bytes() == c011_raw,
    "IMMUTABLE_MANIFEST_RELOAD",
)

c011_reloaded = json.loads(c011_target.read_bytes())

c011_require(
    c011_reloaded["execution_boundary_sha256"]
    == c011_boundary_sha
    and hashlib.sha256(
        c011_canon(c011_reloaded["execution_contract"])
    ).hexdigest() == c011_boundary_sha
    and len(
        c011_reloaded["execution_contract"]["members"]
    ) == 1116
    and c011_reloaded["execution_queue"]["remaining_count"]
    == 1112
    and c011_reloaded["approved_pilot_provenance"]["C010"][
        "file_bytes_sha256"
    ] == c011_expected["audit"],
    "MANIFEST_IDENTITY_RELOAD_CONFLICT",
)

NB22E_V4_POPULATION_BOUNDARY = {
    "status": "POPULATION_BOUNDARY_RELOAD_PASS",
    "execution_boundary_sha256": c011_boundary_sha,
    "manifest_path": str(c011_target),
    "manifest_file_sha256": c011_file_sha,
    "population_count": 1116,
    "pilot_reuse_count": 4,
    "remaining_count": 1112,
    "batch_count": len(c011_batches),
}

print(json.dumps(
    {
        "cell": "NB22E-v4:C011",
        **NB22E_V4_POPULATION_BOUNDARY,
        "runtime_versions": c011_versions,
        "tokenizer_identity_sha256": c011_tokenizer_sha,
        "new_target_quality_accepted": False,
        "repository_modified": False,
        "model_runs": 0,
        "retrieval_queries": 0,
        "api_calls": 0,
        "next": "C012_BATCHED_STAGE1_STAGE2_RESUMABLE_EXECUTION",
    },
    ensure_ascii=False,
    indent=2,
))

{
  "cell": "NB22E-v4:C011",
  "status": "POPULATION_BOUNDARY_RELOAD_PASS",
  "execution_boundary_sha256": "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc",
  "manifest_path": "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/_RAG_finance_NB22E_v4_outputs/C011.population_boundary.a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80.json",
  "manifest_file_sha256": "a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80",
  "population_count": 1116,
  "pilot_reuse_count": 4,
  "remaining_count": 1112,
  "batch_count": 45,
  "runtime_versions": {
    "python": "3.11.3",
    "numpy": "1.24.3",
    "onnxruntime": "1.16.3",
    "qdrant-client": "1.18.0",
    "transformers": "4.53.2",
    "tokenizers": "0.21.2"
  },
  "tokenizer_identity_sha256": "34ab0082bf9db105fafe2b0e581339e820c2a3748d4d826a2f8de6b17def9325",
  "new_target_quality_accepted": false,
  "repository_modified": false,
  "model_runs": 0,
  "retrieval_queries": 0,
  "api_call

<div dir="rtl" align="right">

### NB22E-v4:C012 — اصلاح کنترل مسیر ONNX پس از توقف ایمن

**علت توقف:** استفاده از `resolve()` روی مسیر ارجاعی ONNX، رابطهٔ ظاهری فایل با Snapshot را تغییر داده و کنترل `ONNX_ALIAS` را نامعتبر کرده است.

**اصلاح:** کنترل مسیر مطابق روش موفق C011 انجام می‌شود؛ مسیر نسبی `onnx/model_quint8_avx2.onnx` و Revision با قرارداد فریز‌شده تطبیق می‌یابند. کنترل SHA-256 کامل مدل همچنان برقرار است.

**نتایج محفوظ:** چهار Pilot، سه سؤال کامل، ۱۷۰ جفت ONNX موفق و رسید تاریخی Batch نامعلوم. اجرای ناموفق اخیر هیچ Query یا ONNX تازه‌ای انجام نداده است.

**دامنهٔ اجرای بعد:** یک Retry دستی مصوب برای چهار جفت نامعلوم و ادامهٔ حداکثر پنج سؤال تکمیل‌نشده، با همان چهار Thread و مکث حفاظتی.

**معیار بررسی:** رسیدهای معتبر، سقف تجمعی ۵۵٬۶۰۴ pair-attempt، زمان‌بندی تفکیکی، مصرف حافظه و گزارش گلوگاه واقعی.

**محدودیت:** بدون API، تغییر مدل، GPU، تغییر مخزن یا تکرار نتایج موفق.

</div>

In [15]:
# NB22E-v4 | C012 | R11.2
# One approved manual retry; then at most five unfinished candidates with timing instrumentation.

c012_execute_new_work = True
c012_max_new_candidates = 5
c012_onnx_pause_seconds = 0.8
c012_process_rss_limit_gib = 4.0
c012_query_slow_limit_seconds = 45.0
c012_onnx_slow_limit_seconds = 25.0

c012_retry_id = "golden_v2_candidate_0184f90d79f3dfc4744b6e29"
c012_retry_start = 20
c012_retry_original_start_sha = (
    "4745577680cbd38d3a212e27ec67e3af98a42088cd2f3aace077fca563e4daf2"
)
c012_boundary_sha = (
    "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc"
)
c012_manifest_sha = (
    "a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80"
)
c012_c010_sha = (
    "ae2360b309dc5ed6fe44c1735eecdb0407f96324ca394776739bd6fddbccc6a0"
)
c012_model_sha = (
    "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"
)

c012_run_started = time.perf_counter()

c012_times = {
    key: 0.0
    for key in (
        "source_sha",
        "index_sha",
        "input_read",
        "qdrant_setup",
        "model_setup",
        "query",
        "tokenization",
        "onnx",
        "pause",
        "checkpoint",
    )
}

c012_work = {
    "query_attempts": 0,
    "onnx_attempted_pairs": 0,
    "onnx_done_pairs": 0,
    "onnx_calls": 0,
    "new_completed": 0,
    "manual_retry_pairs": 0,
}

c012_stop = None
c012_active = None


def c012_check(ok, label):
    if not ok:
        raise RuntimeError("C012_" + label)


def c012_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")


def c012_stamp(path):
    stat = path.stat()
    return (
        stat.st_dev,
        stat.st_ino,
        stat.st_size,
        stat.st_mtime_ns,
    )


def c012_sha(path):
    before = c012_stamp(path)
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for data in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(data)

    c012_check(
        c012_stamp(path) == before,
        "HASHED_FILE_CHANGED:" + path.name,
    )
    return digest.hexdigest()


def c012_memory():
    current = None

    try:
        result = subprocess.run(
            [
                "/bin/ps",
                "-o",
                "rss=",
                "-p",
                str(os.getpid()),
            ],
            capture_output=True,
            text=True,
            timeout=3,
            check=True,
        )
        current = round(
            int(result.stdout.strip()) / (1024 ** 2),
            3,
        )
    except (
        OSError,
        ValueError,
        subprocess.SubprocessError,
    ):
        pass

    peak = resource.getrusage(
        resource.RUSAGE_SELF
    ).ru_maxrss

    divisor = (
        1024 ** 3
        if sys.platform == "darwin"
        else 1024 ** 2
    )

    return {
        "process_current_rss_gib": current,
        "process_peak_rss_gib": round(
            peak / divisor,
            3,
        ),
        "system_memory_pressure": "NOT_MEASURED",
    }


# ============================================================
# 1. Independent population boundary
# ============================================================

c012_root = c002_root.resolve()
c012_out = c003_out.resolve()

c012_manifest_path = (
    c012_out
    / (
        "C011.population_boundary."
        + c012_manifest_sha
        + ".json"
    )
).resolve()

c012_check(
    NB22E_V4_READ_ONLY is True
    and c012_out.is_dir()
    and not c012_out.is_relative_to(c012_root)
    and c012_manifest_path.parent == c012_out
    and c012_sha(c012_manifest_path) == c012_manifest_sha,
    "MANIFEST_BYTES",
)

c012_manifest = json.loads(
    c012_manifest_path.read_bytes()
)

c012_contract = c012_manifest["execution_contract"]
c012_batches = c012_manifest["execution_queue"]["batches"]

c012_ids = [
    cid
    for batch in c012_batches
    for cid in batch["candidate_ids"]
]

c012_members = {
    item["candidate_id"]: item["question_sha256"]
    for item in c012_contract["members"]
}

c012_check(
    c012_manifest["execution_boundary_sha256"]
    == c012_boundary_sha
    and hashlib.sha256(
        c012_bytes(c012_contract)
    ).hexdigest() == c012_boundary_sha
    and c012_contract["historical_identity_inherited"] is False
    and len(c012_ids) == len(set(c012_ids)) == 1112
    and len(c012_members) == 1116
    and len(c012_batches) == 45
    and len(
        c012_manifest[
            "approved_pilot_provenance"
        ]["pilot_ids"]
    ) == 4
    and c012_contract["onnx"]["model_sha256"]
    == c012_model_sha
    and c012_contract["frozen_parameters"][
        "onnx_intra_op_threads"
    ] == 4
    and c012_contract["frozen_parameters"][
        "onnx_batch_size"
    ] == 4
    and c012_contract["frozen_parameters"][
        "onnx_max_length"
    ] == 512
    and c012_contract["frozen_parameters"][
        "candidate_parallelism"
    ] == 1
    and 1 <= c012_max_new_candidates <= 5
    and c012_onnx_pause_seconds >= 0,
    "BOUNDARY_OR_FROZEN_PROFILE",
)

c012_audit_pin = c012_manifest[
    "approved_pilot_provenance"
]["C010"]

c012_audit_path = Path(
    c012_audit_pin["path"]
).resolve()

c012_check(
    c012_audit_pin["file_bytes_sha256"]
    == c012_c010_sha
    and c012_audit_path.parent == c012_out
    and c012_sha(c012_audit_path) == c012_c010_sha
    and json.loads(
        c012_audit_path.read_bytes()
    )["status"] == "PILOT_EVIDENCE_AUDIT_PASS",
    "PILOT_AUDIT_BYTES",
)

c012_dir = (
    c012_out
    / ("C012_population_" + c012_boundary_sha)
).resolve()

c012_check(
    c012_dir.parent == c012_out
    and c012_dir.is_dir()
    and not c012_dir.is_symlink(),
    "EXISTING_C012_RECEIPTS_MISSING",
)


# ============================================================
# 2. Existing receipt format and immutable writes
# ============================================================

def c012_path(cid, suffix):
    key = hashlib.sha256(
        cid.encode("utf-8")
    ).hexdigest()

    return c012_dir / (
        "C012." + key + "." + suffix + ".json"
    )


def c012_read(path):
    started = time.perf_counter()

    c012_check(
        path.parent.resolve() == c012_dir
        and path.is_file()
        and not path.is_symlink(),
        "RECEIPT_MISSING:" + path.name,
    )

    raw = path.read_bytes()
    row = json.loads(raw)
    signature = row.pop("core_sha256", None)

    c012_check(
        hashlib.sha256(
            c012_bytes(row)
        ).hexdigest() == signature,
        "RECEIPT_SELF_HASH:" + path.name,
    )

    c012_times["checkpoint"] += (
        time.perf_counter() - started
    )

    return row, hashlib.sha256(raw).hexdigest()


def c012_put(path, core):
    started = time.perf_counter()

    c012_check(
        path.parent.resolve() == c012_dir
        and not path.is_symlink(),
        "UNSAFE_OUTPUT",
    )

    raw = c012_bytes({
        **core,
        "core_sha256": hashlib.sha256(
            c012_bytes(core)
        ).hexdigest(),
    }) + b"\n"

    if not path.exists():
        temporary = None

        try:
            with tempfile.NamedTemporaryFile(
                "wb",
                dir=c012_dir,
                prefix=".C012-",
                suffix=".tmp",
                delete=False,
            ) as stream:
                temporary = Path(stream.name)
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())

            c012_check(
                temporary.read_bytes() == raw,
                "TEMP_RELOAD",
            )

            try:
                os.link(temporary, path)
            except FileExistsError:
                pass

        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    c012_check(
        path.read_bytes() == raw,
        "IMMUTABLE_CONFLICT:" + path.name,
    )

    c012_times["checkpoint"] += (
        time.perf_counter() - started
    )

    return hashlib.sha256(raw).hexdigest()


# ============================================================
# 3. Receipt audit, including the one authorized retry
# ============================================================

def c012_audit():
    counts = {
        "complete": 0,
        "partial": 0,
        "not_started": 0,
        "uncertain": 0,
        "blocked": 0,
    }

    states = {}
    unknown = []

    query_attempts = 0
    pair_attempts = 0
    pair_done = 0

    for cid in c012_ids:
        s1 = c012_path(cid, "stage1_started")
        d1 = c012_path(cid, "stage1_done")
        d2 = c012_path(cid, "stage2_done")

        s1_data = None
        d1_data = None

        if s1.is_file():
            s1_data, _ = c012_read(s1)

            c012_check(
                s1_data["status"] == "STARTED_UNCONFIRMED"
                and s1_data["boundary"] == c012_boundary_sha
                and s1_data["candidate_id"] == cid,
                "S1_START:" + cid,
            )

            query_attempts += 1

        if d1.is_file():
            c012_check(
                s1_data is not None,
                "S1_WITHOUT_START:" + cid,
            )

            d1_data, _ = c012_read(d1)

            c012_check(
                d1_data["boundary"] == c012_boundary_sha
                and d1_data["candidate_id"] == cid
                and d1_data["input_sha256"]
                == s1_data["input_sha256"]
                and d1_data["status"] in (
                    "STAGE1_PASS",
                    "STAGE1_BLOCKED",
                ),
                "S1_DONE:" + cid,
            )

        has_done = {}
        unknown_batches = []

        for start in range(0, 50, 4):
            begun = c012_path(
                cid,
                "stage2_batch_%02d_started" % start,
            )
            ended = c012_path(
                cid,
                "stage2_batch_%02d_done" % start,
            )

            retry_begun = (
                c012_path(
                    cid,
                    "stage2_batch_20_retry1_started",
                )
                if cid == c012_retry_id and start == 20
                else None
            )

            retry_ended = (
                c012_path(
                    cid,
                    "stage2_batch_20_retry1_done",
                )
                if retry_begun is not None
                else None
            )

            original = None
            original_sha = None

            if begun.is_file():
                original, original_sha = c012_read(begun)

                c012_check(
                    original["status"] == "STARTED_UNCONFIRMED"
                    and original["candidate_id"] == cid
                    and original["boundary"] == c012_boundary_sha
                    and original["batch_start"] == start
                    and len(original["chunk_ids"])
                    == min(4, 50 - start),
                    "S2_START:" + cid,
                )

                pair_attempts += len(
                    original["chunk_ids"]
                )

            c012_check(
                not ended.is_file()
                or original is not None,
                "S2_DONE_WITHOUT_START:" + cid,
            )

            retry_intent = None

            if (
                retry_begun is not None
                and retry_begun.is_file()
            ):
                c012_check(
                    original is not None
                    and not ended.is_file()
                    and original_sha
                    == c012_retry_original_start_sha,
                    "RETRY_ORIGINAL_IDENTITY",
                )

                retry_intent, _ = c012_read(
                    retry_begun
                )

                c012_check(
                    retry_intent["status"]
                    == "STARTED_UNCONFIRMED"
                    and retry_intent["original_start_sha256"]
                    == c012_retry_original_start_sha
                    and retry_intent["candidate_id"] == cid
                    and retry_intent["input_sha256"]
                    == original["input_sha256"]
                    and retry_intent["chunk_ids"]
                    == original["chunk_ids"]
                    and retry_intent["batch_start"] == start
                    and retry_intent["boundary"]
                    == c012_boundary_sha
                    and retry_intent["manual_attempt"] == 1,
                    "RETRY_STARTED_CONTRACT",
                )

                pair_attempts += len(
                    retry_intent["chunk_ids"]
                )

            active_path = (
                retry_ended
                if retry_ended is not None
                and retry_ended.is_file()
                else ended
            )

            if active_path.is_file():
                c012_check(
                    original is not None,
                    "S2_DONE_MISSING_ORIGINAL_START",
                )

                finished, receipt_sha = c012_read(
                    active_path
                )

                c012_check(
                    finished["status"] == "BATCH_PASS"
                    and finished["boundary"]
                    == c012_boundary_sha
                    and finished["candidate_id"] == cid
                    and finished["input_sha256"]
                    == original["input_sha256"]
                    and finished["chunk_ids"]
                    == original["chunk_ids"]
                    and len(finished["scores"])
                    == len(original["chunk_ids"])
                    and all(
                        math.isfinite(value)
                        for value in finished["scores"]
                    )
                    and (
                        active_path == ended
                        or retry_intent is not None
                    ),
                    "S2_DONE_BAD:" + cid,
                )

                has_done[start] = (
                    finished,
                    receipt_sha,
                    active_path,
                )

                pair_done += len(
                    finished["scores"]
                )

            elif original is not None:
                unknown_batches.append({
                    "batch_start": start,
                    "kind": (
                        "RETRY_STARTED_NO_DONE"
                        if retry_intent is not None
                        else "ORIGINAL_STARTED_NO_DONE"
                    ),
                })

        if has_done:
            c012_check(
                d1_data is not None
                and d1_data["status"] == "STAGE1_PASS"
                and all(
                    item[0]["input_sha256"]
                    == d1_data["input_sha256"]
                    for item in has_done.values()
                ),
                "ONNX_WITHOUT_VALID_STAGE1:" + cid,
            )

        if d2.is_file():
            final, _ = c012_read(d2)

            merged = {
                **final["historical_pool50_scores"],
                **final["new_scores"],
            }

            hybrid = final["historical_hybrid100"]

            missing = [
                chunk_id
                for chunk_id in hybrid
                if chunk_id not in final[
                    "historical_pool50_scores"
                ]
            ]

            c012_check(
                final["status"] == "STAGE2_PASS"
                and final["boundary"] == c012_boundary_sha
                and final["candidate_id"] == cid
                and d1_data is not None
                and d1_data["status"] == "STAGE1_PASS"
                and final["input_sha256"]
                == d1_data["input_sha256"]
                and d1_data["observed_hybrid100"] == hybrid
                and len(hybrid)
                == len(set(hybrid))
                == len(merged)
                == 100
                and len(missing)
                == len(final["new_scores"])
                == 50
                and all(
                    math.isfinite(value)
                    for value in merged.values()
                )
                and final["local100_order"] == sorted(
                    hybrid,
                    key=lambda chunk_id: (
                        -merged[chunk_id],
                        chunk_id,
                    ),
                )
                and final[
                    "rank_artifact_content_sha256"
                ] == hashlib.sha256(
                    c012_bytes({
                        "boundary": c012_boundary_sha,
                        "candidate_id": cid,
                        "scores": merged,
                        "order": final["local100_order"],
                    })
                ).hexdigest()
                and len(final["batch_receipts"])
                == len(has_done)
                == 13,
                "FINAL_SCORE_CONFLICT:" + cid,
            )

            rebuilt = {}

            for index, receipt in enumerate(
                final["batch_receipts"]
            ):
                begin = index * 4

                c012_check(
                    begin in has_done
                    and receipt["batch_start"] == begin
                    and receipt["path"]
                    == str(has_done[begin][2])
                    and receipt["sha256"]
                    == has_done[begin][1]
                    and has_done[begin][0]["chunk_ids"]
                    == missing[begin:begin + 4],
                    "FINAL_BATCH_LINK:" + cid,
                )

                rebuilt.update(
                    zip(
                        has_done[begin][0]["chunk_ids"],
                        has_done[begin][0]["scores"],
                    )
                )

            c012_check(
                rebuilt == final["new_scores"],
                "FINAL_SCORE_RECONSTRUCT:" + cid,
            )

        if s1_data is not None and d1_data is None:
            status = "UNCERTAIN"
            unknown.append((cid, "QUERY", []))

        elif unknown_batches:
            status = "UNCERTAIN"
            unknown.append((
                cid,
                "ONNX",
                unknown_batches,
            ))

        elif (
            d1_data is not None
            and d1_data["status"] == "STAGE1_BLOCKED"
        ):
            status = "BLOCKED"

        elif d2.is_file():
            status = "COMPLETE"

        elif s1_data is not None or has_done:
            status = "PARTIAL"

        else:
            status = "NOT_STARTED"

        states[cid] = status
        counts[status.lower()] += 1

    c012_check(
        query_attempts <= 1112
        and pair_attempts <= 55604,
        "SUPERVISOR_CUMULATIVE_ATTEMPT_LIMIT",
    )

    return {
        "states": states,
        "counts": counts,
        "unknown": unknown,
        "query_attempts": query_attempts,
        "pair_attempts": pair_attempts,
        "pair_done": pair_done,
    }


# ============================================================
# 4. Audit before any work
# ============================================================

c012_before = c012_audit()

c012_authorized_unknown = [
    (
        c012_retry_id,
        "ONNX",
        [{
            "batch_start": 20,
            "kind": "ORIGINAL_STARTED_NO_DONE",
        }],
    )
]

c012_check(
    c012_before["unknown"] in (
        c012_authorized_unknown,
        [],
    ),
    "UNKNOWN_UNITS_OUTSIDE_APPROVED_RETRY",
)

c012_check(
    c012_before["counts"]["blocked"] == 0,
    "STAGE1_BLOCKED",
)

c012_selected = [
    cid
    for cid in c012_ids
    if c012_before["states"][cid] != "COMPLETE"
][:c012_max_new_candidates]

print(
    json.dumps({
        "cell": "NB22E-v4:C012",
        "phase": "BEFORE_LIMITED_EXECUTION",
        "counts": c012_before["counts"],
        "unknown": c012_before["unknown"],
        "pair_attempts": c012_before["pair_attempts"],
        "selected": c012_selected,
        "max_candidates": 5,
    }, ensure_ascii=False, indent=2),
    flush=True,
)


# ============================================================
# 5. Controlled execution
# ============================================================

def c012_run(selected):
    np = nb22e_v4_require_imports("np")["np"]

    pins = {
        **c012_contract["source_producer_pins"],
        **c012_contract["input_pins"],
        **{
            "VECTOR_" + key: value
            for key, value in c012_contract[
                "query_vector_cache_pins"
            ].items()
        },
    }

    loader = c008_loader

    pins.update({
        "SPARSE_CONFIG": {
            "path": loader["configuration_path"],
            "file_bytes_sha256": loader[
                "configuration_file_sha256"
            ],
        },
        "SPARSE_DOCS": {
            "path": loader["documents_path"],
            "file_bytes_sha256": loader[
                "documents_sha256"
            ],
        },
        "SPARSE_POSTINGS": {
            "path": loader["postings_path"],
            "file_bytes_sha256": loader[
                "postings_sha256"
            ],
        },
    })

    pinned_paths = {}

    for label, pin in pins.items():
        relative = Path(pin["path"])
        path = (c012_root / relative).resolve()

        c012_check(
            not relative.is_absolute()
            and path.is_relative_to(c012_root)
            and path.is_file(),
            "PIN_PATH:" + label,
        )

        if path in pinned_paths:
            c012_check(
                pinned_paths[path]
                == pin["file_bytes_sha256"],
                "PIN_CONFLICT:" + label,
            )

        pinned_paths[path] = pin[
            "file_bytes_sha256"
        ]

        # C012 / R11.2 — ONNX alias contract aligned with C011.
    # Preserve the snapshot alias path; do not resolve the model symlink.

    snapshot = Path(
        os.path.abspath(
            str(Path(c004_snapshot).expanduser())
        )
    )

    model = Path(
        os.path.abspath(
            str(Path(c004_model_path).expanduser())
        )
    )

    c012_check(
        model.is_file()
        and model.parent.parent == snapshot
        and model.relative_to(snapshot).as_posix()
        == c012_contract["onnx"][
            "model_alias_relative_inside_snapshot"
        ]
        and snapshot.name
        == c012_contract["onnx"]["revision"],
        "ONNX_ALIAS",
    )

    pinned_paths[model] = c012_model_sha

    c012_check(
        type(c005_tokenizer).__name__
        == c012_contract["onnx"]["tokenizer"][
            "tokenizer_class"
        ]
        and bool(c005_tokenizer.is_fast)
        == c012_contract["onnx"]["tokenizer"][
            "is_fast"
        ],
        "TOKENIZER_IDENTITY",
    )

    for name, record in c012_contract[
        "onnx"
    ]["tokenizer"]["files"].items():
        pinned_paths[snapshot / name] = record["sha256"]

    for name, version in c012_contract[
        "runtime_versions"
    ].items():
        actual = (
            platform.python_version()
            if name == "python"
            else importlib.metadata.version(name)
        )

        c012_check(
            actual == version,
            "RUNTIME_CHANGED:" + name,
        )

    # Full input and model SHA: exactly once per invocation.
    started = time.perf_counter()
    pin_stamps = {}

    for path, expected in pinned_paths.items():
        c012_check(
            c012_sha(path) == expected,
            "SOURCE_SHA_CHANGED:" + path.name,
        )
        pin_stamps[path] = c012_stamp(path)

    c012_times["source_sha"] += (
        time.perf_counter() - started
    )

    # ========================================================
    # 6. Original Qdrant inventory: one full verification
    # ========================================================

    original = c004_index_path.resolve()

    c012_check(
        original.is_dir()
        and original.is_relative_to(c012_root),
        "QDRANT_ORIGINAL_PATH",
    )

    inventory = c004_docs[
        "manifest"
    ]["storage_inventory"]

    frozen = inventory["files"]

    c012_check(
        inventory["inventory_sha256"]
        == c004_docs["release"][
            "storage_inventory_sha256"
        ]
        == hashlib.sha256(
            c012_bytes(frozen)
        ).hexdigest()
        and inventory["file_count"] == len(frozen),
        "FROZEN_STORAGE_MANIFEST",
    )

    by_path = {
        item["path"]: item
        for item in frozen
    }

    c012_check(
        len(by_path) == len(frozen),
        "FROZEN_STORAGE_DUPLICATE",
    )

    index_stamps = {}
    seen = set()
    size_total = 0

    started = time.perf_counter()

    for folder, dirs, files in os.walk(
        original,
        followlinks=False,
    ):
        c012_check(
            not any(
                (Path(folder) / name).is_symlink()
                for name in dirs + files
            ),
            "INDEX_SYMLINK",
        )

        for name in files:
            file = Path(folder) / name
            relative = file.relative_to(
                original
            ).as_posix()

            c012_check(
                file.is_file(),
                "INDEX_NONFILE",
            )

            size = file.stat().st_size

            if name != ".lock":
                c012_check(
                    relative in by_path
                    and by_path[relative][
                        "size_bytes"
                    ] == size
                    and c012_sha(file)
                    == by_path[relative]["sha256"],
                    "INDEX_SHA_CHANGED:" + relative,
                )

                seen.add(relative)

            index_stamps[file] = c012_stamp(file)
            size_total += size

    c012_times["index_sha"] += (
        time.perf_counter() - started
    )

    c012_check(
        seen == set(by_path)
        and len(index_stamps)
        == c005_index_result["inventory_files"]
        and size_total
        == c005_index_result["inventory_bytes"]
        and size_total <= 512 * 1024 * 1024,
        "INDEX_INVENTORY_DRIFT",
    )

    def guard():
        # Metadata checks only; original full-byte SHA already verified.
        for path, stamp in pin_stamps.items():
            c012_check(
                path.is_file()
                and c012_stamp(path) == stamp,
                "PIN_CHANGED_IN_INVOCATION",
            )

        for path, stamp in index_stamps.items():
            c012_check(
                path.is_file()
                and c012_stamp(path) == stamp,
                "INDEX_CHANGED_IN_INVOCATION",
            )

        # Historical process peak, NOT system memory pressure.
        peak = resource.getrusage(
            resource.RUSAGE_SELF
        ).ru_maxrss

        divisor = (
            1024 ** 3
            if sys.platform == "darwin"
            else 1024 ** 2
        )

        c012_check(
            peak / divisor
            <= c012_process_rss_limit_gib,
            "PROCESS_HISTORICAL_PEAK_LIMIT",
        )

    guard()

    # ========================================================
    # 7. Read data for at most five selected questions
    # ========================================================

    started = time.perf_counter()
    selected_set = set(selected)
    rows = {}

    for key, depth in (
        ("HYBRID", 100),
        ("RERANKER", 50),
    ):
        found = {}

        path = (
            c012_root
            / c012_contract["input_pins"][key]["path"]
        ).resolve()

        with path.open("rb") as source:
            for line in source:
                item = json.loads(line)
                cid = item.get("candidate_id")

                if cid not in selected_set:
                    continue

                sequence = item["candidates"]

                c012_check(
                    cid not in found
                    and len(sequence) == depth
                    and [
                        entry["rank"]
                        for entry in sequence
                    ] == list(range(1, depth + 1))
                    and (
                        item.get("split") == "test"
                        if depth == 100
                        else item.get("split")
                        in ("test", None)
                    ),
                    "HISTORIC_ROW:" + cid,
                )

                found[cid] = sequence

        c012_check(
            set(found) == selected_set,
            "HISTORIC_ROW_COVERAGE:" + key,
        )

        rows[key] = found

    needed = {
        item["chunk_id"]
        for cid in selected
        for item in rows["HYBRID"][cid]
    }

    chunk_text = {}

    chunk_path = (
        c012_root
        / c012_contract["input_pins"]["CHUNKS"]["path"]
    )

    with chunk_path.open("rb") as stream:
        for line in stream:
            item = json.loads(line)
            chunk_id = item["chunk_id"]

            if chunk_id not in needed:
                continue

            c012_check(
                chunk_id not in chunk_text
                and isinstance(
                    item["content"],
                    str,
                ),
                "CHUNK_CONTENT",
            )

            chunk_text[chunk_id] = item["content"]

    c012_check(
        set(chunk_text) == needed,
        "CHUNK_COVERAGE",
    )

    questions = {}

    for cid in selected:
        question = c002_questions[cid]["question"]

        question_sha = hashlib.sha256(
            question.encode("utf-8")
        ).hexdigest()

        c012_check(
            question_sha
            == c002_questions[cid]["question_sha256"]
            == c012_members[cid],
            "QUESTION_PIN:" + cid,
        )

        questions[cid] = question.strip()

    vector_pins = c012_contract[
        "query_vector_cache_pins"
    ]

    metadata = json.loads(
        (
            c012_root
            / vector_pins["metadata"]["path"]
        ).read_bytes()
    )

    vector_order = metadata["candidate_ids"]

    c012_check(
        len(vector_order)
        == len(set(vector_order))
        == 1149
        and selected_set.issubset(vector_order),
        "VECTOR_ID_COVERAGE",
    )

    positions = {
        cid: index
        for index, cid in enumerate(vector_order)
        if cid in selected_set
    }

    vectors = np.load(
        c012_root / vector_pins["array"]["path"],
        mmap_mode="r",
        allow_pickle=False,
    )

    c012_check(
        vectors.shape == (1149, 384)
        and vectors.dtype == np.float32,
        "VECTOR_SHAPE",
    )

    tokens = {
        cid: c008_tokens(questions[cid])
        for cid in selected
    }

    needed_terms = set().union(
        *(set(values) for values in tokens.values())
    )

    postings = {}

    with (
        c012_root / loader["postings_path"]
    ).open("rb") as stream:
        for line in stream:
            item = json.loads(line)

            if item["token"] not in needed_terms:
                continue

            ids = np.asarray(
                item["document_indexes"],
                dtype=np.int64,
            )
            freqs = np.asarray(
                item["term_frequencies"],
                dtype=np.float64,
            )

            c012_check(
                len(ids)
                == len(freqs)
                == item["document_frequency"]
                and bool(
                    (ids[:-1] < ids[1:]).all()
                ),
                "SPARSE_POSTINGS_CONFLICT",
            )

            postings[item["token"]] = (
                ids,
                freqs,
            )

    doc_ids = c008_doc_ids

    lengths = np.asarray(
        c008_doc_lengths,
        dtype=np.float64,
    )

    avg = float(c008_average_doc_length)

    c012_check(
        len(doc_ids)
        == len(lengths)
        == 3488
        and doc_ids == c007_chunk_ids
        and avg > 0,
        "SPARSE_DOCUMENT_CONTRACT",
    )

    c012_times["input_read"] += (
        time.perf_counter() - started
    )

    def sparse(cid):
        scores = np.zeros(
            3488,
            dtype=np.float64,
        )

        for term, qtf in collections.Counter(
            tokens[cid]
        ).items():
            if term not in postings:
                continue

            ids, freqs = postings[term]
            df = len(ids)

            idf = math.log(
                1.0
                + (3488 - df + 0.5) / (df + 0.5)
            )

            normal = (
                1.0 - 0.3
                + 0.3 * lengths[ids] / avg
            )

            scores[ids] += (
                idf
                * freqs
                * 2.2
                / (freqs + 1.2 * normal)
            ) * float(qtf)

        positive = np.flatnonzero(scores > 0)

        ranked = positive[
            np.lexsort((
                positive,
                -scores[positive],
            ))
        ][:100]

        return (
            [doc_ids[index] for index in ranked],
            [float(scores[index]) for index in ranked],
        )

    # ========================================================
    # 8. Lazy model and Qdrant setup
    # ========================================================

    temp_index = None
    client = None
    session = None
    input_names = None
    previous_new_onnx = False

    def model_session():
        nonlocal session, input_names

        if session is not None:
            return

        started = time.perf_counter()

        ort = nb22e_v4_require_imports("ort")["ort"]

        c012_check(
            ort.__version__ == "1.16.3",
            "ORT_VERSION",
        )

        options = ort.SessionOptions()
        options.intra_op_num_threads = 4
        options.inter_op_num_threads = 1
        options.execution_mode = (
            ort.ExecutionMode.ORT_SEQUENTIAL
        )
        options.graph_optimization_level = (
            ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        )

        session = ort.InferenceSession(
            str(model),
            sess_options=options,
            providers=["CPUExecutionProvider"],
        )

        input_names = {
            item.name
            for item in session.get_inputs()
        }

        c012_check(
            session.get_providers()
            == ["CPUExecutionProvider"]
            and {
                "input_ids",
                "attention_mask",
            } <= input_names,
            "ONNX_SESSION_CONTRACT",
        )

        c012_times["model_setup"] += (
            time.perf_counter() - started
        )

    def ensure_client():
        nonlocal client, temp_index

        if client is not None:
            return

        started = time.perf_counter()

        disk = os.statvfs(str(c012_out))

        c012_check(
            disk.f_bavail * disk.f_frsize
            >= max(
                256 * 1024 * 1024,
                2 * size_total,
            ),
            "INSUFFICIENT_CLONE_DISK",
        )

        temp_index = tempfile.TemporaryDirectory(
            prefix="NB22E-C012-",
            dir=str(c012_out),
        )

        clone = Path(temp_index.name) / "qdrant"
        clone.mkdir()

        for source, stamp in index_stamps.items():
            guard()

            target = clone / source.relative_to(original)
            target.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            with (
                source.open("rb") as source_stream,
                target.open("xb") as target_stream,
            ):
                for block in iter(
                    lambda: source_stream.read(
                        1024 * 1024
                    ),
                    b"",
                ):
                    target_stream.write(block)

            c012_check(
                c012_stamp(source) == stamp,
                "SOURCE_CHANGED_DURING_COPY",
            )

        client = nb22e_v4_require_imports(
            "QdrantClient"
        )["QdrantClient"](
            path=str(clone)
        )

        c012_check(
            client.count(
                collection_name=c005_collection,
                exact=True,
            ).count == 3488,
            "CLONE_POINT_COUNT",
        )

        c012_times["qdrant_setup"] += (
            time.perf_counter() - started
        )

    # ========================================================
    # 9. Sequential candidate execution
    # ========================================================

    try:
        for cid in selected:
            global c012_active
            c012_active = cid

            guard()

            ids100 = [
                item["chunk_id"]
                for item in rows["HYBRID"][cid]
            ]

            ids50 = [
                item["chunk_id"]
                for item in rows["RERANKER"][cid]
            ]

            saved = {
                item["chunk_id"]: float(
                    item["reranker_score"]
                )
                for item in rows["RERANKER"][cid]
            }

            missing50 = [
                chunk_id
                for chunk_id in ids100
                if chunk_id not in saved
            ]

            c012_check(
                len(set(ids100)) == 100
                and len(set(ids50))
                == len(saved)
                == 50
                and set(ids50) == set(ids100[:50])
                and len(missing50) == 50
                and all(
                    math.isfinite(value)
                    for value in saved.values()
                ),
                "FROZEN_POOL50:" + cid,
            )

            identity = hashlib.sha256(
                c012_bytes({
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "question_sha256": (
                        c002_questions[cid][
                            "question_sha256"
                        ]
                    ),
                    "historical_hybrid100": ids100,
                    "historical_pool50": ids50,
                    "historical_pool50_scores": saved,
                    "model_sha256": c012_model_sha,
                })
            ).hexdigest()

            s1 = c012_path(
                cid,
                "stage1_started",
            )
            d1 = c012_path(
                cid,
                "stage1_done",
            )
            d2 = c012_path(
                cid,
                "stage2_done",
            )

            s1_core = {
                "status": "STARTED_UNCONFIRMED",
                "boundary": c012_boundary_sha,
                "candidate_id": cid,
                "input_sha256": identity,
                "query_limit": 1,
                "manual_retry_only_if_uncertain": True,
            }

            # ================================================
            # Stage 1: reuse or one new Query
            # ================================================

            if d1.is_file():
                previous, _ = c012_read(d1)
                intent, _ = c012_read(s1)

                c012_check(
                    intent == s1_core
                    and previous["status"] == "STAGE1_PASS"
                    and previous["input_sha256"] == identity
                    and previous["observed_hybrid100"]
                    == ids100
                    and previous["full_order_matches"] is True
                    and previous[
                        "pool50_membership_matches"
                    ] is True,
                    "REUSED_STAGE1_CONFLICT:" + cid,
                )

            else:
                c012_check(
                    not s1.exists(),
                    "UNKNOWN_STAGE1_STARTED_NO_DONE",
                )

                # Only now copy/open Qdrant.
                ensure_client()

                vector = np.asarray(
                    vectors[positions[cid]],
                    dtype=np.float32,
                )

                c012_check(
                    vector.shape == (384,)
                    and bool(
                        np.isfinite(vector).all()
                    )
                    and bool(
                        np.isclose(
                            np.linalg.norm(vector),
                            1.0,
                        )
                    ),
                    "VECTOR_INVALID",
                )

                sparse_ids, sparse_scores = sparse(cid)

                guard()

                c012_check(
                    c012_before["query_attempts"]
                    + c012_work["query_attempts"]
                    + 1 <= 1112,
                    "QUERY_ATTEMPT_BUDGET",
                )

                c012_put(s1, s1_core)

                c012_work["query_attempts"] += 1

                started = time.perf_counter()

                response = client.query_points(
                    collection_name=c005_collection,
                    query=vector.tolist(),
                    limit=100,
                    with_payload=True,
                    with_vectors=False,
                )

                points = getattr(
                    response,
                    "points",
                    response,
                )

                dense_ids = [
                    str(item.payload["chunk_id"])
                    for item in points
                ]

                dense_scores = [
                    float(item.score)
                    for item in points
                ]

                observed = c008_fuse(
                    dense_ids,
                    dense_scores,
                    sparse_ids,
                    sparse_scores,
                )

                elapsed = time.perf_counter() - started
                c012_times["query"] += elapsed

                order_ok = (
                    observed == ids100
                    and len(observed) == 100
                )

                pool_ok = (
                    set(observed[:50]) == set(ids50)
                )

                c012_put(d1, {
                    "status": (
                        "STAGE1_PASS"
                        if order_ok and pool_ok
                        else "STAGE1_BLOCKED"
                    ),
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "input_sha256": identity,
                    "observed_hybrid100": observed,
                    "expected_hybrid100_sha256": (
                        hashlib.sha256(
                            c012_bytes(ids100)
                        ).hexdigest()
                    ),
                    "full_order_matches": order_ok,
                    "pool50_membership_matches": pool_ok,
                    "dense_count": len(dense_ids),
                    "sparse_count": len(sparse_ids),
                    "query_count": 1,
                    "elapsed_seconds": round(elapsed, 4),
                    "first_mismatch_rank": next(
                        (
                            index + 1
                            for index in range(
                                min(len(observed), 100)
                            )
                            if observed[index] != ids100[index]
                        ),
                        None,
                    ),
                })

                c012_check(
                    order_ok and pool_ok,
                    "HYBRID_ORDER_CONFLICT",
                )

                c012_check(
                    elapsed <= c012_query_slow_limit_seconds,
                    "QUERY_SLOW_STOP",
                )

            c012_check(
                not d2.is_file(),
                "COMPLETE_CANDIDATE_WAS_SELECTED",
            )

            # ================================================
            # Stage 2: immutable ONNX batch checkpoints
            # ================================================

            new_scores = {}
            receipts = []

            for start in range(0, 50, 4):
                batch_ids = missing50[
                    start:start + 4
                ]

                begun = c012_path(
                    cid,
                    "stage2_batch_%02d_started" % start,
                )
                done = c012_path(
                    cid,
                    "stage2_batch_%02d_done" % start,
                )

                core = {
                    "status": "STARTED_UNCONFIRMED",
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "input_sha256": identity,
                    "batch_start": start,
                    "chunk_ids": batch_ids,
                    "no_automatic_retry_if_uncertain": True,
                }

                if done.is_file():
                    intent, _ = c012_read(begun)
                    completed, done_sha = c012_read(done)

                    c012_check(
                        intent == core
                        and completed["status"] == "BATCH_PASS"
                        and completed["input_sha256"] == identity
                        and completed["chunk_ids"] == batch_ids
                        and len(completed["scores"])
                        == len(batch_ids)
                        and all(
                            math.isfinite(value)
                            for value in completed["scores"]
                        ),
                        "NORMAL_RECEIPT_REUSE:" + cid,
                    )

                    scores = completed["scores"]
                    selected_done = done

                else:
                    approved_retry = (
                        cid == c012_retry_id
                        and start == c012_retry_start
                    )

                    retry_begun = c012_path(
                        cid,
                        "stage2_batch_20_retry1_started",
                    )
                    retry_done = c012_path(
                        cid,
                        "stage2_batch_20_retry1_done",
                    )

                    if approved_retry:
                        original, original_sha = c012_read(
                            begun
                        )

                        c012_check(
                            original == core
                            and original_sha
                            == c012_retry_original_start_sha,
                            "ORIGINAL_STARTED_RECEIPT_CHANGED",
                        )

                        manual_intent = {
                            **core,
                            "manual_attempt": 1,
                            "original_start_sha256": (
                                c012_retry_original_start_sha
                            ),
                            "authorization": (
                                "SUPERVISOR_SINGLE_MANUAL_RETRY_MAX_55604"
                            ),
                            "total_pair_attempt_ceiling": 55604,
                        }

                        if retry_done.is_file():
                            retry_intent, _ = c012_read(
                                retry_begun
                            )

                            completed, done_sha = c012_read(
                                retry_done
                            )

                            c012_check(
                                retry_intent == manual_intent
                                and completed["status"]
                                == "BATCH_PASS"
                                and completed["input_sha256"]
                                == identity
                                and completed["chunk_ids"]
                                == batch_ids
                                and completed["manual_attempt"] == 1
                                and len(completed["scores"]) == 4
                                and all(
                                    math.isfinite(value)
                                    for value in completed["scores"]
                                ),
                                "RETRY_DONE_CONFLICT",
                            )

                            scores = completed["scores"]
                            selected_done = retry_done

                        else:
                            c012_check(
                                not retry_begun.exists(),
                                "RETRY_STARTED_WITHOUT_DONE_NO_AUTORETRY",
                            )

                    else:
                        c012_check(
                            not begun.exists(),
                            "UNKNOWN_BATCH_NO_AUTORETRY",
                        )

                    if not (
                        approved_retry
                        and retry_done.is_file()
                    ):
                        guard()
                        model_session()

                        c012_check(
                            c012_before["pair_attempts"]
                            + c012_work["onnx_attempted_pairs"]
                            + len(batch_ids)
                            <= 55604,
                            "PAIR_ATTEMPT_BUDGET_55604",
                        )

                        # Measure tokenization separately.
                        started = time.perf_counter()

                        encoded = c005_tokenizer(
                            [questions[cid]] * len(batch_ids),
                            [
                                chunk_text[chunk_id]
                                for chunk_id in batch_ids
                            ],
                            padding=True,
                            truncation=True,
                            max_length=512,
                            return_tensors="np",
                        )

                        c012_times["tokenization"] += (
                            time.perf_counter() - started
                        )

                        c012_check(
                            input_names <= set(encoded),
                            "TOKENIZER_INPUTS",
                        )

                        feed = {
                            name: np.asarray(
                                encoded[name],
                                dtype=np.int64,
                            )
                            for name in input_names
                        }

                        c012_check(
                            all(
                                value.shape[0] == len(batch_ids)
                                and value.shape[1] <= 512
                                for value in feed.values()
                            ),
                            "TOKEN_SHAPE",
                        )

                        # Preserve protective pauses.
                        if previous_new_onnx:
                            started = time.perf_counter()

                            time.sleep(
                                c012_onnx_pause_seconds
                            )

                            c012_times["pause"] += (
                                time.perf_counter() - started
                            )

                            guard()

                        # Historical STARTED stays immutable.
                        # Retry uses a separately named receipt.
                        c012_put(
                            retry_begun if approved_retry else begun,
                            manual_intent if approved_retry else core,
                        )

                        c012_work[
                            "onnx_attempted_pairs"
                        ] += len(batch_ids)

                        c012_work["onnx_calls"] += 1

                        if approved_retry:
                            c012_work[
                                "manual_retry_pairs"
                            ] += len(batch_ids)

                        started = time.perf_counter()

                        outputs = session.run(
                            None,
                            feed,
                        )

                        elapsed = (
                            time.perf_counter() - started
                        )

                        c012_times["onnx"] += elapsed

                        values = np.asarray(
                            outputs[0],
                            dtype=np.float64,
                        ).reshape(-1)

                        c012_check(
                            values.shape == (len(batch_ids),)
                            and bool(
                                np.isfinite(values).all()
                            ),
                            "NONFINITE_ONNX_OUTPUT",
                        )

                        scores = [
                            float(value)
                            for value in values
                        ]

                        selected_done = (
                            retry_done
                            if approved_retry
                            else done
                        )

                        done_core = {
                            "status": "BATCH_PASS",
                            "boundary": c012_boundary_sha,
                            "candidate_id": cid,
                            "input_sha256": identity,
                            "chunk_ids": batch_ids,
                            "scores": scores,
                            "elapsed_seconds": round(
                                elapsed,
                                4,
                            ),
                            "process_peak_rss_gib": (
                                c012_memory()[
                                    "process_peak_rss_gib"
                                ]
                            ),
                            "onnx_calls": 1,
                            "qdrant_queries": 0,
                            "api_calls": 0,
                        }

                        if approved_retry:
                            done_core.update({
                                "manual_attempt": 1,
                                "original_start_sha256": (
                                    c012_retry_original_start_sha
                                ),
                            })

                        done_sha = c012_put(
                            selected_done,
                            done_core,
                        )

                        c012_work[
                            "onnx_done_pairs"
                        ] += len(batch_ids)

                        previous_new_onnx = True

                        c012_check(
                            elapsed
                            <= c012_onnx_slow_limit_seconds,
                            "ONNX_SLOW_STOP",
                        )

                        guard()

                c012_check(
                    not set(batch_ids).intersection(
                        new_scores
                    ),
                    "REUSED_PAIR_DUPLICATE",
                )

                new_scores.update(
                    zip(batch_ids, scores)
                )

                receipts.append({
                    "path": str(selected_done),
                    "sha256": done_sha,
                    "batch_start": start,
                    "count": len(batch_ids),
                })

            # ================================================
            # 10. Complete Local100 without changing scores
            # ================================================

            merged = {
                **saved,
                **new_scores,
            }

            c012_check(
                len(new_scores) == 50
                and len(merged) == 100
                and set(merged) == set(ids100)
                and all(
                    math.isfinite(value)
                    for value in merged.values()
                ),
                "LOCAL100_INCOMPLETE",
            )

            rank = sorted(
                ids100,
                key=lambda chunk_id: (
                    -merged[chunk_id],
                    chunk_id,
                ),
            )

            rank_sha = hashlib.sha256(
                c012_bytes({
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "scores": merged,
                    "order": rank,
                })
            ).hexdigest()

            c012_put(d2, {
                "status": "STAGE2_PASS",
                "boundary": c012_boundary_sha,
                "candidate_id": cid,
                "input_sha256": identity,
                "historical_hybrid100": ids100,
                "historical_pool50_scores": saved,
                "new_scores": new_scores,
                "local100_order": rank,
                "rank_artifact_content_sha256": rank_sha,
                "batch_receipts": receipts,
                "new_pairs": 50,
                "historical_reused_pairs": 50,
                "derived_order_quota_char_rrf_executed": False,
                "context_generated": False,
                "target_gate": "BLOCKED",
            })

            c012_put(
                c012_path(cid, "progress"),
                {
                    "status": "CANDIDATE_CHECKPOINTED",
                    "boundary": c012_boundary_sha,
                    "candidate_id": cid,
                    "stage1_path": str(d1),
                    "stage2_path": str(d2),
                },
            )

            c012_work["new_completed"] += 1

            print(
                "C012 candidate",
                c012_work["new_completed"],
                "/",
                len(selected),
                cid,
                "RSS",
                c012_memory(),
                flush=True,
            )

        # ====================================================
        # 11. Close only fully completed original batches
        # ====================================================

        for batch in c012_batches:
            summary_path = c012_dir / (
                "C012.batch_%03d.done.json"
                % batch["batch_number"]
            )

            if summary_path.is_file():
                continue

            if not all(
                c012_path(
                    cid,
                    "stage2_done",
                ).is_file()
                for cid in batch["candidate_ids"]
            ):
                continue

            receipts = []

            for cid in batch["candidate_ids"]:
                stage1, stage1_sha = c012_read(
                    c012_path(
                        cid,
                        "stage1_done",
                    )
                )

                stage2, stage2_sha = c012_read(
                    c012_path(
                        cid,
                        "stage2_done",
                    )
                )

                c012_check(
                    stage1["status"] == "STAGE1_PASS"
                    and stage2["status"] == "STAGE2_PASS",
                    "BATCH_CLOSURE_NOT_VERIFIED",
                )

                receipts.append({
                    "candidate_id": cid,
                    "stage1_sha256": stage1_sha,
                    "stage2_sha256": stage2_sha,
                })

            c012_put(summary_path, {
                "status": "BATCH_COMPLETE",
                "boundary": c012_boundary_sha,
                "batch_number": batch["batch_number"],
                "candidate_ids": batch["candidate_ids"],
                "completed_in_batch": len(
                    batch["candidate_ids"]
                ),
                "receipts": receipts,
            })

    finally:
        if client is not None:
            client.close()

        session = None

        if temp_index is not None:
            temp_index.cleanup()


# ============================================================
# 12. Execute once, bounded by five unfinished candidates
# ============================================================

if c012_execute_new_work:
    try:
        if c012_selected:
            c012_run(c012_selected)

    except KeyboardInterrupt:
        c012_stop = {
            "kind": "KEYBOARD_INTERRUPT",
            "candidate_id": c012_active,
            "action": (
                "STARTED_WITHOUT_DONE_HELD_NO_AUTO_RETRY"
            ),
        }

    except Exception as error:
        sys.excepthook(type(error), error, error.__traceback__)
        c012_stop = {
            "kind": type(error).__name__,
            "detail": str(error)[:700],
            "candidate_id": c012_active,
        }


# ============================================================
# 13. Final audit, timing table and bounded diagnosis
# ============================================================

c012_after = c012_audit()

c012_elapsed = (
    time.perf_counter() - c012_run_started
)

c012_accounted = sum(
    c012_times.values()
)

c012_times["other_unattributed"] = max(
    0.0,
    c012_elapsed - c012_accounted,
)

c012_share = [
    {
        "part": key,
        "seconds": round(value, 3),
        "percent_of_elapsed": (
            round(
                100 * value / c012_elapsed,
                1,
            )
            if c012_elapsed
            else 0.0
        ),
    }
    for key, value in c012_times.items()
]

c012_share.sort(
    key=lambda row: -row["seconds"]
)

if c012_work["onnx_calls"] == 0:
    c012_diagnosis = (
        "INSUFFICIENT_NEW_ONNX_TIMING"
    )
    c012_next = (
        "CONTINUE_ONE_CONTROLLED_INVOCATION_AFTER_REPAIRING_ANY_STOP"
    )

else:
    dominant = c012_share[0]["part"]

    if dominant == "onnx":
        c012_diagnosis = (
            "ONNX_WALL_TIME_DOMINATES_THIS_INVOCATION;"
            " SYSTEM_PRESSURE_NOT_MEASURED"
        )
        c012_next = (
            "PREPARE_CPU_OR_GPU_NUMERIC_PARITY_FEASIBILITY_REVIEW_ONLY"
        )

    else:
        c012_diagnosis = (
            "LOCAL_OVERHEAD_DOMINANT:" + dominant
        )
        c012_next = (
            "OPTIMIZE_THIS_LOCAL_OVERHEAD_WITHOUT_CHANGING_FROZEN_SCORING"
        )

c012_report = {
    "cell": "NB22E-v4:C012",
    "status": (
        "STOPPED_WITH_CHECKPOINTS"
        if c012_stop
        else (
            "LIMITED_INVOCATION_COMPLETE"
            if c012_execute_new_work
            else "AUDIT_ONLY"
        )
    ),
    "execution_boundary_sha256": c012_boundary_sha,
    "selected_candidate_ids": c012_selected,
    "before": {
        key: value
        for key, value in c012_before.items()
        if key != "states"
    },
    "after": {
        key: value
        for key, value in c012_after.items()
        if key != "states"
    },
    "new_work_this_invocation": c012_work,
    "total_elapsed_seconds": round(
        c012_elapsed,
        3,
    ),
    "timing_table": c012_share,
    "bottleneck_assessment": c012_diagnosis,
    "single_next_action": c012_next,
    "stop_reason": c012_stop,
    "memory": c012_memory(),
    "api_calls": 0,
    "repository_modified": False,
    "four_pilot_reused": 4,
    "retried_historical_started_receipt_unchanged": True,
    "quality_gate": "BLOCKED",
    "GPU_transfer_authorized": False,
}


# ============================================================
# 14. Immutable run report with reload verification
# ============================================================

if c012_execute_new_work:
    raw = c012_bytes(c012_report) + b"\n"

    report_sha = hashlib.sha256(
        raw
    ).hexdigest()

    report_path = c012_dir / (
        "C012.run_summary."
        + report_sha
        + ".json"
    )

    if not report_path.exists():
        temporary = None

        try:
            with tempfile.NamedTemporaryFile(
                "wb",
                dir=c012_dir,
                prefix=".C012-report-",
                suffix=".tmp",
                delete=False,
            ) as stream:
                temporary = Path(stream.name)
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())

            try:
                os.link(temporary, report_path)
            except FileExistsError:
                pass

        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    c012_check(
        report_path.read_bytes() == raw,
        "REPORT_RELOAD_CONFLICT",
    )

    c012_report["report_checkpoint"] = {
        "path": str(report_path),
        "sha256": report_sha,
    }

NB22E_V4_POPULATION_EXECUTION = c012_report

print(
    json.dumps(
        c012_report,
        ensure_ascii=False,
        indent=2,
    ),
    flush=True,
)

{
  "cell": "NB22E-v4:C012",
  "phase": "BEFORE_LIMITED_EXECUTION",
  "counts": {
    "complete": 4,
    "partial": 0,
    "not_started": 1108,
    "uncertain": 0,
    "blocked": 0
  },
  "unknown": [],
  "pair_attempts": 204,
  "selected": [
    "golden_v2_candidate_01b22bd8a25fb93d74033d78",
    "golden_v2_candidate_01bacc2bf4a88b206123a68c",
    "golden_v2_candidate_01d35d353c6e3c5a219fc1f2",
    "golden_v2_candidate_021a702f669446a411c22d0e",
    "golden_v2_candidate_0223fb3898c3681e9fd788a0"
  ],
  "max_candidates": 5
}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

C012 candidate 1 / 5 golden_v2_candidate_01b22bd8a25fb93d74033d78 RSS {'process_current_rss_gib': 0.985, 'process_peak_rss_gib': 0.985, 'system_memory_pressure': 'NOT_MEASURED'}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

C012 candidate 2 / 5 golden_v2_candidate_01bacc2bf4a88b206123a68c RSS {'process_current_rss_gib': 0.901, 'process_peak_rss_gib': 0.998, 'system_memory_pressure': 'NOT_MEASURED'}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

... (4 outputs left out)

C012 candidate 5 / 5 golden_v2_candidate_0223fb3898c3681e9fd788a0 RSS {'process_current_rss_gib': 0.968, 'process_peak_rss_gib': 0.998, 'system_memory_pressure': 'NOT_MEASURED'}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
  "cell": "NB22E-v4:C012",
  "status": "LIMITED_INVOCATION_COMPLETE",
  "execution_boundary_sha256": "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc",
  "selected_candidate_ids": [
    "golden_v2_candidate_01b22bd8a25fb93d74033d78",
    "golden_v2_candidate_01bacc2bf4a88b206123a68c",
    "golden_v2_candidate_01d35d353c6e3c5a219fc1f2",
    "golden_v2_candidate_021a702f669446a411c22d0e",
    "golden_v2_candidate_0223fb3898c3681e9fd788a0"
  ],
  "before": {
    "counts": {
      "complete": 4,
      "partial": 0,
      "not_started": 1108,
      "uncertain": 0,
      "blocked": 0
    },
    "unknown": [],
    "query_attempts": 4,
    "pair_attempts": 204,
    "pair_done": 200
  },
  "after": {
    "counts": {
      "complete": 9,
      "partial": 0,
      "not_started": 1103,
      "uncertain": 0,
      "blocked": 0
    },
    "unknown": [],
    "query_attempts": 9,
    "pair_attempts": 454,
    "pair_done": 450
  },
  "new_work_this_invocation": {
    "query_attempts"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


<div dir="rtl" align="right">

### NB22E-v4 | C012 | R11.2 — ادامهٔ خودکار و محافظت‌شدهٔ Stage 1/2

**هدف:** تکمیل ۱۱۰۳ سؤال باقی‌مانده در مرحله‌های متوالی حداکثر پنج‌سؤالی، بدون تأیید دستی برای هر مرحله و بدون تکرار محاسبات موفق.

**مبنای پذیرفته‌شده:** ۹ سؤال جدید کامل، چهار Pilot محفوظ، ۴۵۰ جفت یکتای موفق، ۴۵۴ pair-attempt و چهار تلاش تاریخیِ تکراریِ مصوب. سقف تجمعی تلاش‌ها ۵۵٬۶۰۴ است.

**حفاظت از نتایج:**
- خواندن و اعتبارسنجی Manifest فریز‌شدهٔ C011 و رسیدهای موجود.
- انتخاب نخستین سؤال ناتمام از ترتیب اصلی جمعیت.
- استفادهٔ مجدد از Stage 1، Batchهای ONNX و امتیازهای معتبر؛ بدون Retry جدید.
- ذخیرهٔ STARTED پیش از کار تازه و DONE فقط پس از خروجی معتبر.
- ذخیره و بازخوانی Checkpoint بعد از هر سؤال.
- حفظ مرز Batchهای اصلی ۲۵سؤالی و عدم تغییر فایل‌های تاریخی.

**کنترل منابع:**
- یک مرتبه هش کامل فایل‌های فریز‌شده در هر فراخوانی؛ سپس کنترل تغییر مشخصات فایل‌ها.
- ساخت یک کپی موقت Qdrant، فقط در صورت نیاز، برای کل فراخوانی.
- یک نشست ONNX و دسترسی انتخابی به داده‌های پنج سؤال جاری.
- حفظ CPU، چهار Thread، Batch=4، max_length=512 و مکث حفاظتی ۰٫۸ ثانیه.
- نمونه‌برداری دوره‌ای از RSS فرایند، شاخص حافظهٔ سیستم و محدودیت سرعت CPU در macOS. دادهٔ ناموجود به‌معنای سلامت تأییدشده نیست.

**گزارش هر مرحله:** تعداد سؤال و جفت‌های تکمیل‌شده، زمان Query، Tokenization، ONNX، مکث، هش و آماده‌سازی، Checkpoint، سربار، مصرف حافظه و برآورد تازهٔ زمان باقی‌مانده.

**شرایط توقف:** Interrupt، STARTED بدون DONE، ناسازگاری SHA یا نسخه، خروجی نامعتبر، شکست Checkpoint، افت شدید سرعت یا شاخص نامطلوب واقعی سیستم. خطا باعث Retry خودکار نمی‌شود.

**دامنهٔ مجاز:** فقط Stage 1 و Stage 2. بدون API، تغییر مدل، انتقال GPU، تغییر مخزن، derived_order، Quota، charRRF، ساخت Context یا Generation.

**پذیرش:** تکمیل فنی Stage 1/2 مستقل از پذیرش نهایی R11.2 است. فایل‌های خروجی فقط در پوشهٔ جدا از مخزن قرار می‌گیرند.

</div>

In [16]:
# NB22E-v4 | C012 | R11.2
# Resume only frozen Stage 1/2; automatic five-candidate stages, immutable checkpoints.

c012_stage_size = 5
c012_onnx_pause_seconds = 0.8
c012_process_rss_limit_gib = 4.0
c012_monitor_interval_seconds = 90.0
c012_query_slow_limit_seconds = 45.0
c012_onnx_slow_limit_seconds = 25.0
c012_retry_id = "golden_v2_candidate_0184f90d79f3dfc4744b6e29"
c012_retry_original_start_sha = "4745577680cbd38d3a212e27ec67e3af98a42088cd2f3aace077fca563e4daf2"
c012_boundary_sha = "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc"
c012_manifest_sha = "a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80"
c012_pilot_audit_sha = "ae2360b309dc5ed6fe44c1735eecdb0407f96324ca394776739bd6fddbccc6a0"
c012_model_sha = "6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b"
c012_index_sha = "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4"
c012_run_start = time.perf_counter()
c012_timing = {name: 0.0 for name in (
    "source_sha", "index_sha", "input_read", "model_setup", "qdrant_setup",
    "query", "tokenization", "onnx", "pause", "checkpoint",
    "system_monitor")}
c012_counts_run = {"new_queries": 0, "new_onnx_attempts": 0,
                   "new_onnx_completed": 0, "onnx_calls": 0,
                   "new_candidates_completed": 0, "new_retries": 0}
c012_stage_history = []
c012_stop = None
c012_active = None
c012_last_sample_at = 0.0
c012_system_samples = []
c012_recent_onnx = []
c012_slow_consecutive = 0


def c012_check(condition, code):
    if not condition:
        raise RuntimeError("C012_" + code)


def c012_canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False).encode("utf-8")


def c012_stamp(path):
    st = path.stat()
    return (st.st_dev, st.st_ino, st.st_size, st.st_mtime_ns)


def c012_sha(path):
    old = c012_stamp(path)
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    c012_check(old == c012_stamp(path), "SOURCE_CHANGED_DURING_SHA:" + path.name)
    return digest.hexdigest()


def c012_read(path):
    started = time.perf_counter()
    c012_check(path.parent.resolve() == c012_dir and path.is_file()
               and not path.is_symlink(), "RECEIPT_UNSAFE:" + path.name)
    raw = path.read_bytes()
    payload = json.loads(raw)
    signature = payload.pop("core_sha256", None)
    c012_check(signature == hashlib.sha256(c012_canonical(payload)).hexdigest(),
               "RECEIPT_SHA:" + path.name)
    c012_timing["checkpoint"] += time.perf_counter() - started
    return payload, hashlib.sha256(raw).hexdigest()


def c012_write(path, core):
    started = time.perf_counter()
    c012_check(path.parent.resolve() == c012_dir and not path.is_symlink(),
               "OUTPUT_UNSAFE:" + path.name)
    raw = c012_canonical({**core,
        "core_sha256": hashlib.sha256(c012_canonical(core)).hexdigest()}) + b"\n"
    if not path.exists():
        temporary = None
        try:
            with tempfile.NamedTemporaryFile("wb", dir=c012_dir, prefix=".C012-",
                                             suffix=".tmp", delete=False) as stream:
                temporary = Path(stream.name)
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            c012_check(temporary.read_bytes() == raw, "TEMP_WRITE_RELOAD")
            try:
                os.link(temporary, path)
            except FileExistsError:
                pass
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
    c012_check(path.is_file() and path.read_bytes() == raw,
               "IMMUTABLE_RECEIPT_CONFLICT:" + path.name)
    c012_timing["checkpoint"] += time.perf_counter() - started
    return hashlib.sha256(raw).hexdigest()


def c012_path(cid, suffix):
    key = hashlib.sha256(cid.encode("utf-8")).hexdigest()
    return c012_dir / ("C012." + key + "." + suffix + ".json")


def c012_memory_peak():
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return round(peak / (1024 ** 3 if sys.platform == "darwin" else 1024 ** 2), 3)


def c012_sample_system(force=False):
    global c012_last_sample_at
    now = time.perf_counter()
    if not force and now - c012_last_sample_at < c012_monitor_interval_seconds:
        return c012_system_samples[-1] if c012_system_samples else None
    started = now
    sample = {"process_current_rss_gib": None, "process_peak_rss_gib": c012_memory_peak(),
              "system_free_memory_percent": None, "thermal_cpu_speed_limit_percent": None,
              "memory_pressure_class": "NOT_VERIFIED", "thermal_throttling": "NOT_VERIFIED"}
    commands = (
        ("ps", ["/bin/ps", "-o", "rss=", "-p", str(os.getpid())]),
        ("memory_pressure", ["/usr/bin/memory_pressure", "-Q"]),
        ("thermal", ["/usr/bin/pmset", "-g", "therm"]),
    )
    for name, command in commands:
        if not Path(command[0]).is_file():
            continue
        try:
            reply = subprocess.run(command, capture_output=True, text=True,
                                   timeout=5, check=False)
            if reply.returncode != 0:
                continue
            if name == "ps":
                sample["process_current_rss_gib"] = round(
                    int(reply.stdout.strip()) / (1024 ** 2), 3)
            elif name == "memory_pressure":
                match = re.search(r"System-wide free memory percentage\s*:\s*(\d+)%",
                                  reply.stdout, re.IGNORECASE)
                if match:
                    sample["system_free_memory_percent"] = int(match.group(1))
            elif name == "thermal":
                match = re.search(r"CPU_Speed_Limit\s*=\s*(\d+)", reply.stdout)
                if match:
                    sample["thermal_cpu_speed_limit_percent"] = int(match.group(1))
                    sample["thermal_throttling"] = (
                        "MEASURED_LIMIT" if int(match.group(1)) < 100
                        else "NO_LIMIT_IN_THIS_SAMPLE")
        except (OSError, ValueError, subprocess.SubprocessError):
            continue
    c012_system_samples.append(sample)
    c012_last_sample_at = time.perf_counter()
    c012_timing["system_monitor"] += c012_last_sample_at - started
    current = sample["process_current_rss_gib"]
    c012_check(current is None or current <= c012_process_rss_limit_gib,
               "PROCESS_CURRENT_RSS_EXCESS")
    c012_check(sample["process_peak_rss_gib"] <= c012_process_rss_limit_gib,
               "PROCESS_HISTORICAL_PEAK_EXCESS")
    # The free-memory number is NOT itself a verified macOS pressure class.
    # A very low observed value is only a conservative stop trigger.
    free = sample["system_free_memory_percent"]
    c012_check(free is None or free > 5, "MEASURED_SYSTEM_FREE_MEMORY_VERY_LOW")
    speed = sample["thermal_cpu_speed_limit_percent"]
    c012_check(speed is None or speed >= 90, "MEASURED_THERMAL_CPU_SPEED_LIMIT")
    return sample


# ============================================================
# Validated frozen boundary and restartable disk receipts
# ============================================================
c012_root = c002_root.resolve()
c012_out = c003_out.resolve()
c012_manifest_path = (c012_out / (
    "C011.population_boundary." + c012_manifest_sha + ".json")).resolve()
c012_check(NB22E_V4_READ_ONLY is True and c012_out.is_dir()
           and not c012_out.is_relative_to(c012_root)
           and c012_manifest_path.parent == c012_out
           and c012_sha(c012_manifest_path) == c012_manifest_sha,
           "C011_MANIFEST_BYTES")
c012_manifest = json.loads(c012_manifest_path.read_bytes())
c012_contract = c012_manifest["execution_contract"]
c012_batches = c012_manifest["execution_queue"]["batches"]
c012_ids = [cid for batch in c012_batches for cid in batch["candidate_ids"]]
c012_members = {row["candidate_id"]: row["question_sha256"]
                for row in c012_contract["members"]}
c012_pilots = c012_manifest["approved_pilot_provenance"]["pilot_ids"]
c012_check(c012_manifest["execution_boundary_sha256"] == c012_boundary_sha
           and hashlib.sha256(c012_canonical(c012_contract)).hexdigest() == c012_boundary_sha
           and c012_contract["historical_identity_inherited"] is False
           and len(c012_ids) == len(set(c012_ids)) == 1112
           and len(c012_members) == 1116 and len(c012_pilots) == 4
           and set(c012_ids).isdisjoint(c012_pilots)
           and set(c012_ids) | set(c012_pilots) == set(c012_members)
           and len(c012_batches) == 45
           and all(batch["batch_number"] == i + 1
                   and 1 <= len(batch["candidate_ids"]) <= 25
                   for i, batch in enumerate(c012_batches))
           and c012_contract["dense_index"]["logical_index_sha256"] == c012_index_sha
           and c012_contract["onnx"]["model_sha256"] == c012_model_sha
           and c012_contract["frozen_parameters"]["onnx_intra_op_threads"] == 4
           and c012_contract["frozen_parameters"]["onnx_batch_size"] == 4
           and c012_contract["frozen_parameters"]["onnx_max_length"] == 512
           and c012_contract["frozen_parameters"]["candidate_parallelism"] == 1
           and c012_manifest["execution_queue"]["fresh_stage1_queries_max"] == 1112
           and c012_manifest["execution_queue"]["fresh_stage2_onnx_pairs_max"] == 55600
           and c012_stage_size == 5 and c012_onnx_pause_seconds == 0.8,
           "FROZEN_POPULATION_OR_EXECUTION_PROFILE")
c012_c010_pin = c012_manifest["approved_pilot_provenance"]["C010"]
c012_c010_path = Path(c012_c010_pin["path"]).resolve()
c012_check(c012_c010_path.parent == c012_out
           and c012_c010_pin["file_bytes_sha256"] == c012_pilot_audit_sha
           and c012_sha(c012_c010_path) == c012_pilot_audit_sha
           and json.loads(c012_c010_path.read_bytes())["status"]
               == "PILOT_EVIDENCE_AUDIT_PASS", "C010_PILOT_PROVENANCE")
c012_dir = (c012_out / ("C012_population_" + c012_boundary_sha)).resolve()
c012_check(c012_dir.parent == c012_out and c012_dir.is_dir()
           and not c012_dir.is_symlink(), "C012_EXISTING_CHECKPOINT_DIRECTORY")


def c012_audit():
    states = {}
    counts = {key: 0 for key in ("COMPLETE", "PARTIAL", "NOT_STARTED", "UNCERTAIN", "BLOCKED")}
    query_attempts = 0
    pair_attempts = 0
    pair_done = 0
    unknown = []
    for cid in c012_ids:
        s1 = c012_path(cid, "stage1_started")
        d1 = c012_path(cid, "stage1_done")
        d2 = c012_path(cid, "stage2_done")
        started1 = None
        finished1 = None
        if s1.is_file():
            started1, _ = c012_read(s1)
            c012_check(started1["status"] == "STARTED_UNCONFIRMED"
                       and started1["candidate_id"] == cid
                       and started1["boundary"] == c012_boundary_sha,
                       "STAGE1_STARTED_INVALID:" + cid)
            query_attempts += 1
        if d1.is_file():
            c012_check(started1 is not None, "STAGE1_DONE_WITHOUT_START:" + cid)
            finished1, _ = c012_read(d1)
            c012_check(finished1["status"] in ("STAGE1_PASS", "STAGE1_BLOCKED")
                       and finished1["candidate_id"] == cid
                       and finished1["boundary"] == c012_boundary_sha
                       and finished1["input_sha256"] == started1["input_sha256"],
                       "STAGE1_DONE_INVALID:" + cid)
            if finished1["status"] == "STAGE1_PASS":
                c012_check(finished1["full_order_matches"] is True
                           and finished1["pool50_membership_matches"] is True
                           and len(set(finished1["observed_hybrid100"])) == 100,
                           "STAGE1_PASS_INVALID:" + cid)
        done_batches = {}
        local_unknown = []
        for start in range(0, 50, 4):
            started_path = c012_path(cid, "stage2_batch_%02d_started" % start)
            done_path = c012_path(cid, "stage2_batch_%02d_done" % start)
            original = None
            original_sha = None
            if started_path.is_file():
                original, original_sha = c012_read(started_path)
                c012_check(original["status"] == "STARTED_UNCONFIRMED"
                           and original["candidate_id"] == cid
                           and original["boundary"] == c012_boundary_sha
                           and original["batch_start"] == start
                           and len(original["chunk_ids"]) == min(4, 50 - start),
                           "STAGE2_START_INVALID:" + cid)
                pair_attempts += len(original["chunk_ids"])
            c012_check(not done_path.is_file() or original is not None,
                       "STAGE2_DONE_WITHOUT_START:" + cid)
            if cid == c012_retry_id and start == 20:
                retry_s = c012_path(cid, "stage2_batch_20_retry1_started")
                retry_d = c012_path(cid, "stage2_batch_20_retry1_done")
                retry_intent = None
                if retry_s.is_file():
                    c012_check(original is not None and not done_path.is_file()
                               and original_sha == c012_retry_original_start_sha,
                               "HISTORICAL_RETRY_ORIGINAL_CHANGED")
                    retry_intent, _ = c012_read(retry_s)
                    c012_check(retry_intent["status"] == "STARTED_UNCONFIRMED"
                               and retry_intent["candidate_id"] == cid
                               and retry_intent["boundary"] == c012_boundary_sha
                               and retry_intent["manual_attempt"] == 1
                               and retry_intent["original_start_sha256"]
                                   == c012_retry_original_start_sha
                               and retry_intent["input_sha256"] == original["input_sha256"]
                               and retry_intent["chunk_ids"] == original["chunk_ids"],
                               "HISTORICAL_RETRY_INTENT_INVALID")
                    pair_attempts += len(retry_intent["chunk_ids"])
                if retry_d.is_file():
                    c012_check(retry_intent is not None, "RETRY_DONE_WITHOUT_START")
                    done_path = retry_d
            if done_path.is_file():
                done, sha = c012_read(done_path)
                c012_check(original is not None
                           and done["status"] == "BATCH_PASS"
                           and done["boundary"] == c012_boundary_sha
                           and done["candidate_id"] == cid
                           and done["input_sha256"] == original["input_sha256"]
                           and done["chunk_ids"] == original["chunk_ids"]
                           and len(done["scores"]) == len(original["chunk_ids"])
                           and all(math.isfinite(float(x)) for x in done["scores"]),
                           "STAGE2_DONE_INVALID:" + cid)
                if cid == c012_retry_id and start == 20:
                    c012_check(done_path.name.endswith("retry1_done.json")
                               and done["manual_attempt"] == 1
                               and done["original_start_sha256"]
                                   == c012_retry_original_start_sha,
                               "RETRY_DONE_CONTRACT")
                done_batches[start] = (done, sha, done_path)
                pair_done += len(done["scores"])
            elif original is not None:
                local_unknown.append(start)
        if done_batches:
            c012_check(finished1 is not None and finished1["status"] == "STAGE1_PASS"
                       and all(row[0]["input_sha256"] == finished1["input_sha256"]
                               for row in done_batches.values()),
                       "STAGE2_WITHOUT_VALID_STAGE1:" + cid)
        if d2.is_file():
            final, _ = c012_read(d2)
            historic = final["historical_pool50_scores"]
            fresh = final["new_scores"]
            hybrid = final["historical_hybrid100"]
            combined = {**historic, **fresh}
            missing = [x for x in hybrid if x not in historic]
            c012_check(finished1 is not None
                       and final["status"] == "STAGE2_PASS"
                       and final["candidate_id"] == cid
                       and final["boundary"] == c012_boundary_sha
                       and final["input_sha256"] == finished1["input_sha256"]
                       and final["historical_hybrid100"] == finished1["observed_hybrid100"]
                       and len(set(hybrid)) == len(combined) == 100
                       and len(historic) == len(fresh) == len(missing) == 50
                       and set(fresh) == set(missing)
                       and all(math.isfinite(float(x)) for x in combined.values())
                       and final["local100_order"] == sorted(
                           hybrid, key=lambda x: (-combined[x], x))
                       and final["rank_artifact_content_sha256"]
                           == hashlib.sha256(c012_canonical({
                               "boundary": c012_boundary_sha, "candidate_id": cid,
                               "scores": combined,
                               "order": final["local100_order"]})).hexdigest()
                       and len(final["batch_receipts"]) == len(done_batches) == 13,
                       "STAGE2_FINAL_CONFLICT:" + cid)
            rebuilt = {}
            for j, pin in enumerate(final["batch_receipts"]):
                start = j * 4
                c012_check(start in done_batches
                           and pin["batch_start"] == start
                           and pin["path"] == str(done_batches[start][2])
                           and pin["sha256"] == done_batches[start][1]
                           and done_batches[start][0]["chunk_ids"] == missing[start:start + 4],
                           "STAGE2_FINAL_BATCH_LINK:" + cid)
                rebuilt.update(zip(done_batches[start][0]["chunk_ids"],
                                   done_batches[start][0]["scores"]))
            c012_check(rebuilt == fresh, "STAGE2_SCORE_RECONSTRUCTION:" + cid)
            progress_path = c012_path(cid, "progress")
            c012_check(progress_path.is_file(), "CANDIDATE_PROGRESS_MISSING:" + cid)
            progress, _ = c012_read(progress_path)
            c012_check(progress["status"] == "CANDIDATE_CHECKPOINTED"
                       and progress["candidate_id"] == cid
                       and progress["boundary"] == c012_boundary_sha,
                       "CANDIDATE_PROGRESS_INVALID:" + cid)
        elif c012_path(cid, "progress").exists():
            c012_check(False, "ORPHAN_PROGRESS:" + cid)
        if started1 is not None and finished1 is None:
            state = "UNCERTAIN"
            unknown.append((cid, "QUERY"))
        elif local_unknown:
            state = "UNCERTAIN"
            unknown.append((cid, "ONNX", local_unknown))
        elif finished1 is not None and finished1["status"] == "STAGE1_BLOCKED":
            state = "BLOCKED"
        elif d2.is_file():
            state = "COMPLETE"
        elif started1 is not None or done_batches:
            state = "PARTIAL"
        else:
            state = "NOT_STARTED"
        states[cid] = state
        counts[state] += 1
    c012_check(query_attempts <= 1112 and pair_attempts <= 55604
               and pair_done <= 55600,
               "CUMULATIVE_APPROVED_ATTEMPTS_EXCEEDED")
    for batch in c012_batches:
        summary_path = c012_dir / ("C012.batch_%03d.done.json" % batch["batch_number"])
        if not summary_path.is_file():
            continue
        summary, _ = c012_read(summary_path)
        c012_check(summary["status"] == "BATCH_COMPLETE"
                   and summary["boundary"] == c012_boundary_sha
                   and summary["batch_number"] == batch["batch_number"]
                   and summary["candidate_ids"] == batch["candidate_ids"]
                   and summary["completed_in_batch"] == len(batch["candidate_ids"])
                   and len(summary["receipts"]) == len(batch["candidate_ids"]),
                   "ORIGINAL_BATCH_SUMMARY_INVALID")
        for cid, pin in zip(batch["candidate_ids"], summary["receipts"]):
            c012_check(states[cid] == "COMPLETE" and pin["candidate_id"] == cid
                       and c012_read(c012_path(cid, "stage1_done"))[1]
                           == pin["stage1_sha256"]
                       and c012_read(c012_path(cid, "stage2_done"))[1]
                           == pin["stage2_sha256"],
                       "ORIGINAL_BATCH_SUMMARY_LINK:" + cid)
    return {"states": states, "counts": counts, "unknown": unknown,
            "query_attempts": query_attempts, "pair_attempts": pair_attempts,
            "pair_done": pair_done}


c012_before = c012_audit()
c012_check(c012_before["counts"]["COMPLETE"] >= 9
           and c012_before["counts"]["PARTIAL"] >= 0
           and c012_before["counts"]["UNCERTAIN"] == 0
           and c012_before["counts"]["BLOCKED"] == 0
           and c012_before["pair_done"] >= 450
           and c012_before["pair_attempts"] >= 454
           and c012_before["pair_attempts"] == c012_before["pair_done"] + 4
           and c012_before["query_attempts"] >= c012_before["counts"]["COMPLETE"],
           "APPROVED_PROGRESS_OR_NO_RETRY_BASELINE_CHANGED")
c012_completed_before = c012_before["counts"]["COMPLETE"]
c012_attempts_before = c012_before["pair_attempts"]
c012_pairs_before = c012_before["pair_done"]
print(json.dumps({"cell": "NB22E-v4:C012", "phase": "AUTOMATIC_RESUME_PREFLIGHT",
    "completed": c012_completed_before, "remaining": 1112 - c012_completed_before,
    "pair_attempts": c012_attempts_before,
    "unique_pairs_remaining": 55600 - c012_pairs_before,
    "max_total_pair_attempts": 55604,
    "stage_size": c012_stage_size, "onnx_pause_seconds": c012_onnx_pause_seconds,
    "no_extra_retry": True}, ensure_ascii=False, indent=2), flush=True)


def c012_run_automatic():
    global c012_active, c012_slow_consecutive
    np_local = np
    ort_local = ort
    c012_check(np_local is not None and ort_local is not None,
               "CENTRALIZED_IMPORT_MISSING")
    sources = {**c012_contract["source_producer_pins"],
               **c012_contract["input_pins"],
               **{"VECTOR_" + name: pin for name, pin in
                  c012_contract["query_vector_cache_pins"].items()}}
    loader = c008_loader
    sources.update({
        "SPARSE_CONFIGURATION": {"path": loader["configuration_path"],
            "file_bytes_sha256": loader["configuration_file_sha256"]},
        "SPARSE_DOCUMENTS": {"path": loader["documents_path"],
            "file_bytes_sha256": loader["documents_sha256"]},
        "SPARSE_POSTINGS": {"path": loader["postings_path"],
            "file_bytes_sha256": loader["postings_sha256"]},
    })
    expected = {}
    for name, pin in sources.items():
        relative = Path(pin["path"])
        path = (c012_root / relative).resolve()
        c012_check(not relative.is_absolute() and path.is_relative_to(c012_root)
                   and path.is_file(), "PIN_PATH:" + name)
        if path in expected:
            c012_check(expected[path] == pin["file_bytes_sha256"],
                       "PIN_ALIAS_CONFLICT:" + name)
        expected[path] = pin["file_bytes_sha256"]
    # Preserve the original model alias, even when it is a symlink into HF cache.
    snapshot = Path(os.path.abspath(str(Path(c004_snapshot).expanduser())))
    model = Path(os.path.abspath(str(Path(c004_model_path).expanduser())))
    c012_check(model.is_file() and model.parent.parent == snapshot
               and model.relative_to(snapshot).as_posix()
                   == c012_contract["onnx"]["model_alias_relative_inside_snapshot"]
               and snapshot.name == c012_contract["onnx"]["revision"],
               "ONNX_ALIAS_FROZEN")
    expected[model] = c012_model_sha
    for name, row in c012_contract["onnx"]["tokenizer"]["files"].items():
        expected[snapshot / name] = row["sha256"]
    c012_check(type(c005_tokenizer).__name__
                   == c012_contract["onnx"]["tokenizer"]["tokenizer_class"]
               and bool(c005_tokenizer.is_fast)
                   == c012_contract["onnx"]["tokenizer"]["is_fast"],
               "TOKENIZER_FROZEN_IDENTITY")
    for name, version in c012_contract["runtime_versions"].items():
        actual = (platform.python_version() if name == "python"
                  else importlib.metadata.version(name))
        c012_check(actual == version, "RUNTIME_VERSION:" + name)
    started = time.perf_counter()
    pinned_stamps = {}
    for path, sha in expected.items():
        c012_check(c012_sha(path) == sha, "PIN_SHA:" + str(path))
        pinned_stamps[path] = c012_stamp(path)
    c012_timing["source_sha"] += time.perf_counter() - started
    original = c004_index_path.resolve()
    c012_check(original.is_dir() and original.is_relative_to(c012_root),
               "ORIGINAL_INDEX_PATH")
    inventory = c004_docs["manifest"]["storage_inventory"]
    frozen_files = inventory["files"]
    c012_check(inventory["file_count"] == len(frozen_files)
               and inventory["inventory_sha256"]
                   == c004_docs["release"]["storage_inventory_sha256"]
                   == hashlib.sha256(c012_canonical(frozen_files)).hexdigest(),
               "QDRANT_FROZEN_INVENTORY")
    index_by_name = {row["path"]: row for row in frozen_files}
    c012_check(len(index_by_name) == len(frozen_files), "INDEX_DUPLICATE")
    index_stamps = {}
    index_bytes = 0
    seen = set()
    started = time.perf_counter()
    for folder, dirs, files in os.walk(original, followlinks=False):
        c012_check(not any((Path(folder) / name).is_symlink()
                           for name in dirs + files), "INDEX_SYMLINK")
        for name in files:
            path = Path(folder) / name
            relative = path.relative_to(original).as_posix()
            c012_check(path.is_file(), "INDEX_NOT_REGULAR")
            if name != ".lock":
                c012_check(relative in index_by_name
                           and path.stat().st_size == index_by_name[relative]["size_bytes"]
                           and c012_sha(path) == index_by_name[relative]["sha256"],
                           "INDEX_STORAGE_SHA:" + relative)
                seen.add(relative)
            index_stamps[path] = c012_stamp(path)
            index_bytes += path.stat().st_size
    c012_timing["index_sha"] += time.perf_counter() - started
    c012_check(seen == set(index_by_name)
               and len(index_stamps) == c005_index_result["inventory_files"]
               and index_bytes == c005_index_result["inventory_bytes"]
               and index_bytes <= 512 * 1024 * 1024,
               "QDRANT_INDEX_DRIFT")

    def guard():
        for path, stamp in pinned_stamps.items():
            c012_check(path.is_file() and c012_stamp(path) == stamp,
                       "INPUT_CHANGED:" + path.name)
        for path, stamp in index_stamps.items():
            c012_check(path.is_file() and c012_stamp(path) == stamp,
                       "INDEX_CHANGED:" + path.name)
        c012_check(c012_memory_peak() <= c012_process_rss_limit_gib,
                   "PROCESS_PEAK_EXCESS")

    # Index the byte offsets just once. Each stage reads only its five candidates.
    started = time.perf_counter()
    index_sources = {}
    opened = []
    client = None
    temp_index = None
    session = None
    model_inputs = None
    previous_fresh_onnx = False
    try:
        for label in ("HYBRID", "RERANKER", "CHUNKS"):
            handle = (c012_root / c012_contract["input_pins"][label]["path"]).open("rb")
            opened.append(handle)
            positions = {}
            while True:
                position = handle.tell()
                line = handle.readline()
                if not line:
                    break
                row = json.loads(line)
                key = (row["candidate_id"] if label != "CHUNKS"
                       else row["chunk_id"])
                c012_check(key not in positions, "SOURCE_DUPLICATE:" + label)
                positions[key] = position
            index_sources[label] = (handle, positions)
        handle = (c012_root / loader["postings_path"]).open("rb")
        opened.append(handle)
        positions = {}
        while True:
            position = handle.tell()
            line = handle.readline()
            if not line:
                break
            row = json.loads(line)
            c012_check(row["token"] not in positions, "POSTING_DUPLICATE")
            positions[row["token"]] = position
        index_sources["POSTINGS"] = (handle, positions)
        vector_pins = c012_contract["query_vector_cache_pins"]
        vector_meta = json.loads((c012_root / vector_pins["metadata"]["path"]).read_bytes())
        vector_order = vector_meta["candidate_ids"]
        c012_check(len(vector_order) == len(set(vector_order)) == 1149
                   and set(c012_ids).issubset(vector_order), "VECTOR_CANDIDATE_BINDING")
        vector_positions = {cid: i for i, cid in enumerate(vector_order)}
        vectors = np_local.load(c012_root / vector_pins["array"]["path"],
                                mmap_mode="r", allow_pickle=False)
        c012_check(vectors.shape == (1149, 384) and vectors.dtype == np_local.float32,
                   "VECTOR_SHAPE_DTYPE")
        document_ids = c008_doc_ids
        document_lengths = np_local.asarray(c008_doc_lengths, dtype=np_local.float64)
        mean_length = float(c008_average_doc_length)
        c012_check(len(document_ids) == len(document_lengths) == 3488
                   and document_ids == c007_chunk_ids and mean_length > 0,
                   "SPARSE_DOCUMENT_CONTRACT")
        c012_timing["input_read"] += time.perf_counter() - started

        def get_row(label, key):
            handle, offsets = index_sources[label]
            c012_check(key in offsets, "SOURCE_MISSING:" + label + ":" + str(key))
            handle.seek(offsets[key])
            return json.loads(handle.readline())

        def ensure_model():
            nonlocal session, model_inputs
            if session is not None:
                return
            started = time.perf_counter()
            c012_check(ort_local.__version__ == "1.16.3", "ORT_VERSION")
            options = ort_local.SessionOptions()
            options.intra_op_num_threads = 4
            options.inter_op_num_threads = 1
            options.execution_mode = ort_local.ExecutionMode.ORT_SEQUENTIAL
            options.graph_optimization_level = ort_local.GraphOptimizationLevel.ORT_ENABLE_ALL
            session = ort_local.InferenceSession(str(model), sess_options=options,
                                                 providers=["CPUExecutionProvider"])
            model_inputs = {item.name for item in session.get_inputs()}
            c012_check(session.get_providers() == ["CPUExecutionProvider"]
                       and {"input_ids", "attention_mask"} <= model_inputs,
                       "ONNX_INPUT_CONTRACT")
            c012_timing["model_setup"] += time.perf_counter() - started

        def ensure_qdrant():
            nonlocal client, temp_index
            if client is not None:
                return
            started = time.perf_counter()
            stat = os.statvfs(str(c012_out))
            c012_check(stat.f_bavail * stat.f_frsize >= max(256 * 1024 * 1024,
                                                            2 * index_bytes),
                       "INSUFFICIENT_QDRANT_CLONE_DISK")
            temp_index = tempfile.TemporaryDirectory(
                prefix="NB22E-C012-", dir=str(c012_out))
            clone = Path(temp_index.name) / "qdrant"
            clone.mkdir()
            for source, stamp in index_stamps.items():
                guard()
                target = clone / source.relative_to(original)
                target.parent.mkdir(parents=True, exist_ok=True)
                with (source.open("rb") as f_in, target.open("xb") as f_out):
                    for data in iter(lambda: f_in.read(1024 * 1024), b""):
                        f_out.write(data)
                c012_check(c012_stamp(source) == stamp,
                           "INDEX_CHANGED_DURING_CLONE")
            client = QdrantClient(path=str(clone))
            c012_check(client.count(collection_name=c005_collection,
                                    exact=True).count == 3488,
                       "CLONED_QDRANT_POINT_COUNT")
            c012_timing["qdrant_setup"] += time.perf_counter() - started

        def sparse_for(cid, words):
            scores = np_local.zeros(3488, dtype=np_local.float64)
            for term, qtf in collections.Counter(words).items():
                if term not in index_sources["POSTINGS"][1]:
                    continue
                posting = get_row("POSTINGS", term)
                ids = np_local.asarray(posting["document_indexes"], dtype=np_local.int64)
                frequencies = np_local.asarray(posting["term_frequencies"], dtype=np_local.float64)
                c012_check(len(ids) == len(frequencies) == posting["document_frequency"]
                           and bool((ids[:-1] < ids[1:]).all()), "SPARSE_POSTINGS")
                df = len(ids)
                idf = math.log(1.0 + (3488 - df + 0.5) / (df + 0.5))
                scores[ids] += (idf * frequencies * 2.2
                                / (frequencies + 1.2 * (
                                    1.0 - 0.3 + 0.3 * document_lengths[ids] / mean_length
                                ))) * float(qtf)
            valid = np_local.flatnonzero(scores > 0)
            ranked = valid[np_local.lexsort((valid, -scores[valid]))][:100]
            return ([document_ids[i] for i in ranked],
                    [float(scores[i]) for i in ranked])

        def run_candidate(cid):
            global c012_active, c012_slow_consecutive
            nonlocal previous_fresh_onnx
            c012_active = cid
            guard()
            c012_check(not c012_path(cid, "stage2_done").exists(),
                       "ALREADY_DONE_CANDIDATE_SELECTED")
            started = time.perf_counter()
            hybrid_row = get_row("HYBRID", cid)
            pool_row = get_row("RERANKER", cid)
            rows100, rows50 = hybrid_row["candidates"], pool_row["candidates"]
            c012_check(len(rows100) == 100 and len(rows50) == 50
                       and [x["rank"] for x in rows100] == list(range(1, 101))
                       and [x["rank"] for x in rows50] == list(range(1, 51))
                       and hybrid_row.get("split") == "test"
                       and pool_row.get("split") in (None, "test"),
                       "HISTORICAL_RANK_SCHEMA:" + cid)
            ids100 = [row["chunk_id"] for row in rows100]
            ids50 = [row["chunk_id"] for row in rows50]
            previous_scores = {row["chunk_id"]: float(row["reranker_score"])
                               for row in rows50}
            missing50 = [chunk_id for chunk_id in ids100
                         if chunk_id not in previous_scores]
            c012_check(len(set(ids100)) == 100
                       and len(set(ids50)) == len(previous_scores) == 50
                       and set(ids50) == set(ids100[:50])
                       and len(missing50) == 50
                       and all(math.isfinite(x) for x in previous_scores.values()),
                       "FROZEN_POOL50:" + cid)
            record = c002_questions[cid]
            question = record["question"]
            c012_check(hashlib.sha256(question.encode("utf-8")).hexdigest()
                       == record["question_sha256"] == c012_members[cid],
                       "QUESTION_SHA:" + cid)
            question = question.strip()
            contents = {chunk_id: get_row("CHUNKS", chunk_id)["content"]
                        for chunk_id in missing50}
            c012_check(len(contents) == 50
                       and all(isinstance(s, str) for s in contents.values()),
                       "MISSING50_CONTENT")
            c012_timing["input_read"] += time.perf_counter() - started
            identity = hashlib.sha256(c012_canonical({
                "boundary": c012_boundary_sha, "candidate_id": cid,
                "question_sha256": record["question_sha256"],
                "historical_hybrid100": ids100,
                "historical_pool50": ids50,
                "historical_pool50_scores": previous_scores,
                "model_sha256": c012_model_sha,
            })).hexdigest()
            s1 = c012_path(cid, "stage1_started")
            d1 = c012_path(cid, "stage1_done")
            d2 = c012_path(cid, "stage2_done")
            s1_intent = {"status": "STARTED_UNCONFIRMED", "boundary": c012_boundary_sha,
                         "candidate_id": cid, "input_sha256": identity,
                         "query_limit": 1, "manual_retry_only_if_uncertain": True}
            if d1.is_file():
                intent, _ = c012_read(s1)
                finished, _ = c012_read(d1)
                c012_check(intent == s1_intent and finished["status"] == "STAGE1_PASS"
                           and finished["input_sha256"] == identity
                           and finished["observed_hybrid100"] == ids100
                           and finished["full_order_matches"] is True
                           and finished["pool50_membership_matches"] is True,
                           "REUSED_STAGE1_CONFLICT:" + cid)
            else:
                c012_check(not s1.exists(), "UNKNOWN_STAGE1_NO_RETRY:" + cid)
                ensure_qdrant()
                vector = np_local.asarray(vectors[vector_positions[cid]],
                                          dtype=np_local.float32)
                c012_check(vector.shape == (384,) and bool(np_local.isfinite(vector).all())
                           and bool(np_local.isclose(np_local.linalg.norm(vector), 1.0)),
                           "DENSE_VECTOR_CONTRACT")
                token_started = time.perf_counter()
                words = c008_tokens(question)
                c012_timing["tokenization"] += time.perf_counter() - token_started
                sparse_started = time.perf_counter()
                sparse_ids, sparse_scores = sparse_for(cid, words)
                c012_timing["input_read"] += time.perf_counter() - sparse_started
                guard()
                c012_check(c012_before["query_attempts"]
                           + c012_counts_run["new_queries"] + 1 <= 1112,
                           "QUERY_ATTEMPT_BUDGET")
                c012_write(s1, s1_intent)
                c012_counts_run["new_queries"] += 1
                query_started = time.perf_counter()
                response = client.query_points(
                    collection_name=c005_collection, query=vector.tolist(),
                    limit=100, with_payload=True, with_vectors=False)
                points = getattr(response, "points", response)
                dense_ids = [str(row.payload["chunk_id"]) for row in points]
                dense_scores = [float(row.score) for row in points]
                observed = c008_fuse(dense_ids, dense_scores,
                                     sparse_ids, sparse_scores)
                elapsed = time.perf_counter() - query_started
                c012_timing["query"] += elapsed
                order_ok = observed == ids100 and len(observed) == 100
                pool_ok = set(observed[:50]) == set(ids50)
                c012_write(d1, {
                    "status": "STAGE1_PASS" if order_ok and pool_ok else "STAGE1_BLOCKED",
                    "boundary": c012_boundary_sha, "candidate_id": cid,
                    "input_sha256": identity, "observed_hybrid100": observed,
                    "expected_hybrid100_sha256": hashlib.sha256(
                        c012_canonical(ids100)).hexdigest(),
                    "full_order_matches": order_ok,
                    "pool50_membership_matches": pool_ok,
                    "dense_count": len(dense_ids), "sparse_count": len(sparse_ids),
                    "query_count": 1, "elapsed_seconds": round(elapsed, 4),
                    "first_mismatch_rank": next((i + 1 for i in range(min(len(observed), 100))
                                                 if observed[i] != ids100[i]), None),
                })
                c012_check(order_ok and pool_ok, "HYBRID100_PARITY_STOP:" + cid)
                c012_check(elapsed <= c012_query_slow_limit_seconds,
                           "SEVERE_QUERY_SLOWDOWN:" + cid)
            fresh = {}
            batch_pins = []
            for start in range(0, 50, 4):
                batch_ids = missing50[start:start + 4]
                start_path = c012_path(cid, "stage2_batch_%02d_started" % start)
                done_path = c012_path(cid, "stage2_batch_%02d_done" % start)
                intent_core = {"status": "STARTED_UNCONFIRMED",
                               "boundary": c012_boundary_sha, "candidate_id": cid,
                               "input_sha256": identity, "batch_start": start,
                               "chunk_ids": batch_ids,
                               "no_automatic_retry_if_uncertain": True}
                c012_check(not (cid == c012_retry_id and start == 20),
                           "HISTORICAL_RETRY_CANNOT_RERUN")
                if done_path.is_file():
                    prev, _ = c012_read(start_path)
                    result, done_sha = c012_read(done_path)
                    c012_check(prev == intent_core and result["status"] == "BATCH_PASS"
                               and result["input_sha256"] == identity
                               and result["chunk_ids"] == batch_ids
                               and len(result["scores"]) == len(batch_ids)
                               and all(math.isfinite(x) for x in result["scores"]),
                               "EXISTING_STAGE2_BATCH_INVALID:" + cid)
                    scores = result["scores"]
                else:
                    c012_check(not start_path.exists(),
                               "STARTED_WITHOUT_DONE_NO_AUTORETRY:" + cid)
                    guard()
                    ensure_model()
                    c012_check(c012_before["pair_attempts"]
                               + c012_counts_run["new_onnx_attempts"]
                               + len(batch_ids) <= 55604,
                               "PAIR_ATTEMPT_LIMIT_55604")
                    tokenize_at = time.perf_counter()
                    encoded = c005_tokenizer(
                        [question] * len(batch_ids),
                        [contents[chunk] for chunk in batch_ids],
                        padding=True, truncation=True, max_length=512,
                        return_tensors="np")
                    c012_timing["tokenization"] += time.perf_counter() - tokenize_at
                    c012_check(model_inputs <= set(encoded), "ONNX_INPUT_TOKENS")
                    feed = {name: np_local.asarray(encoded[name], dtype=np_local.int64)
                            for name in model_inputs}
                    c012_check(all(value.ndim == 2
                                   and value.shape[0] == len(batch_ids)
                                   and value.shape[1] <= 512
                                   for value in feed.values()),
                               "ONNX_SEQUENCE_BOUNDS")
                    if previous_fresh_onnx:
                        pause_at = time.perf_counter()
                        time.sleep(c012_onnx_pause_seconds)
                        c012_timing["pause"] += time.perf_counter() - pause_at
                        guard()
                    c012_write(start_path, intent_core)
                    c012_counts_run["new_onnx_attempts"] += len(batch_ids)
                    c012_counts_run["onnx_calls"] += 1
                    compute_at = time.perf_counter()
                    outputs = session.run(None, feed)
                    compute = time.perf_counter() - compute_at
                    c012_timing["onnx"] += compute
                    scores_np = np_local.asarray(outputs[0], dtype=np_local.float64).reshape(-1)
                    c012_check(scores_np.shape == (len(batch_ids),)
                               and bool(np_local.isfinite(scores_np).all()),
                               "ONNX_OUTPUT_INVALID:" + cid)
                    scores = [float(value) for value in scores_np]
                    done_sha = c012_write(done_path, {
                        "status": "BATCH_PASS", "boundary": c012_boundary_sha,
                        "candidate_id": cid, "input_sha256": identity,
                        "chunk_ids": batch_ids, "scores": scores,
                        "elapsed_seconds": round(compute, 4),
                        "process_peak_rss_gib": c012_memory_peak(),
                        "onnx_calls": 1, "qdrant_queries": 0, "api_calls": 0,
                    })
                    c012_counts_run["new_onnx_completed"] += len(batch_ids)
                    previous_fresh_onnx = True
                    c012_recent_onnx.append(compute)
                    if compute > 4.0:
                        c012_slow_consecutive += 1
                    else:
                        c012_slow_consecutive = 0
                    c012_check(compute <= c012_onnx_slow_limit_seconds
                               and c012_slow_consecutive < 3,
                               "SUSTAINED_OR_SEVERE_ONNX_SLOWDOWN")
                    # Sample hardware only at a long interval, never per ONNX batch.
                    if time.perf_counter() - c012_last_sample_at >= c012_monitor_interval_seconds:
                        c012_sample_system()
                c012_check(not set(batch_ids).intersection(fresh),
                           "DUPLICATE_BATCH_SCORES")
                fresh.update(zip(batch_ids, scores))
                batch_pins.append({"path": str(done_path), "sha256": done_sha,
                                   "batch_start": start, "count": len(batch_ids)})
            combined = {**previous_scores, **fresh}
            c012_check(len(fresh) == 50 and len(combined) == 100
                       and set(combined) == set(ids100)
                       and all(math.isfinite(x) for x in combined.values()),
                       "LOCAL100_COMPLETENESS")
            rank = sorted(ids100, key=lambda x: (-combined[x], x))
            rank_sha = hashlib.sha256(c012_canonical({
                "boundary": c012_boundary_sha, "candidate_id": cid,
                "scores": combined, "order": rank})).hexdigest()
            c012_write(d2, {
                "status": "STAGE2_PASS", "boundary": c012_boundary_sha,
                "candidate_id": cid, "input_sha256": identity,
                "historical_hybrid100": ids100,
                "historical_pool50_scores": previous_scores,
                "new_scores": fresh, "local100_order": rank,
                "rank_artifact_content_sha256": rank_sha,
                "batch_receipts": batch_pins,
                "new_pairs": 50, "historical_reused_pairs": 50,
                "derived_order_quota_char_rrf_executed": False,
                "context_generated": False, "target_gate": "BLOCKED",
            })
            c012_write(c012_path(cid, "progress"), {
                "status": "CANDIDATE_CHECKPOINTED", "boundary": c012_boundary_sha,
                "candidate_id": cid, "stage1_path": str(d1),
                "stage2_path": str(d2),
            })
            completed, _ = c012_read(d2)
            progress, _ = c012_read(c012_path(cid, "progress"))
            c012_check(completed["status"] == "STAGE2_PASS"
                       and completed["rank_artifact_content_sha256"] == rank_sha
                       and progress["status"] == "CANDIDATE_CHECKPOINTED",
                       "CANDIDATE_RELOAD_INVALID:" + cid)
            c012_counts_run["new_candidates_completed"] += 1
            c012_active = None
            # Release the 50 texts and score maps before the next question.

        def close_original_batches():
            for batch in c012_batches:
                summary = c012_dir / ("C012.batch_%03d.done.json"
                                     % batch["batch_number"])
                if summary.is_file():
                    continue
                if not all(c012_path(cid, "stage2_done").is_file()
                           for cid in batch["candidate_ids"]):
                    continue
                rows = []
                for cid in batch["candidate_ids"]:
                    s1, s1sha = c012_read(c012_path(cid, "stage1_done"))
                    s2, s2sha = c012_read(c012_path(cid, "stage2_done"))
                    c012_check(s1["status"] == "STAGE1_PASS"
                               and s2["status"] == "STAGE2_PASS",
                               "ORIGINAL_BATCH_CLOSE_INVALID")
                    rows.append({"candidate_id": cid, "stage1_sha256": s1sha,
                                 "stage2_sha256": s2sha})
                c012_write(summary, {
                    "status": "BATCH_COMPLETE", "boundary": c012_boundary_sha,
                    "batch_number": batch["batch_number"],
                    "candidate_ids": batch["candidate_ids"],
                    "completed_in_batch": len(batch["candidate_ids"]),
                    "receipts": rows,
                })

        # ====================================================
        # Automatic bounded stages, no permission prompts
        # ====================================================
        c012_sample_system(force=True)
        queue = [cid for cid in c012_ids
                 if c012_before["states"][cid] != "COMPLETE"]
        c012_check(len(queue) == 1112 - c012_completed_before, "RESUME_QUEUE_IDENTITY")
        stage_index = 0
        while queue:
            stage_index += 1
            stage = queue[:c012_stage_size]
            stage_started = time.perf_counter()
            start_timing = dict(c012_timing)
            start_work = dict(c012_counts_run)
            for cid in stage:
                guard()
                if time.perf_counter() - c012_last_sample_at >= c012_monitor_interval_seconds:
                    c012_sample_system()
                run_candidate(cid)
            close_original_batches()
            # Verify each stage's completed questions from immutable receipts.
            for cid in stage:
                s2, _ = c012_read(c012_path(cid, "stage2_done"))
                p, _ = c012_read(c012_path(cid, "progress"))
                c012_check(s2["status"] == "STAGE2_PASS"
                           and p["status"] == "CANDIDATE_CHECKPOINTED",
                           "STAGE_CHECKPOINT_RELOAD:" + cid)
            del queue[:len(stage)]
            c012_sample_system()
            stage_elapsed = time.perf_counter() - stage_started
            delta = {key: round(c012_timing[key] - start_timing[key], 3)
                     for key in c012_timing}
            stage_measured = sum(delta.values())
            stage_other = round(max(0.0, stage_elapsed - stage_measured), 3)
            stage_record = {
                "stage": stage_index, "candidate_ids": stage,
                "completed_total": 1112 - len(queue),
                "remaining_candidates": len(queue),
                "pair_attempts_total": c012_attempts_before + c012_counts_run["new_onnx_attempts"],
                "unique_onnx_pairs_done_total": c012_pairs_before + c012_counts_run["new_onnx_completed"],
                "seconds": round(stage_elapsed, 3),
                "timing_seconds": delta, "other_unattributed_seconds": stage_other,
                "work_in_stage": {key: c012_counts_run[key] - start_work[key]
                                  for key in c012_counts_run},
                "memory": c012_system_samples[-1] if c012_system_samples else None,
            }
            c012_stage_history.append(stage_record)
            # Estimate from the latest five observed stages, not a fixed pilot.
            recent = c012_stage_history[-5:]
            seconds_per_candidate = statistics.median(
                item["seconds"] / len(item["candidate_ids"]) for item in recent)
            stage_record["eta_hours_current_rate"] = round(
                seconds_per_candidate * len(queue) / 3600, 2)
            stage_raw = c012_canonical(stage_record) + b"\n"
            stage_sha = hashlib.sha256(stage_raw).hexdigest()
            stage_path = c012_dir / ("C012.auto_stage_%03d.%s.json"
                                    % (stage_index, stage_sha))
            # Reuse the same atomic immutable publisher: stage report has no core receipt.
            if not stage_path.exists():
                temporary = None
                try:
                    with tempfile.NamedTemporaryFile("wb", dir=c012_dir,
                                                     prefix=".C012-auto-", suffix=".tmp",
                                                     delete=False) as stream:
                        temporary = Path(stream.name)
                        stream.write(stage_raw)
                        stream.flush()
                        os.fsync(stream.fileno())
                    try:
                        os.link(temporary, stage_path)
                    except FileExistsError:
                        pass
                finally:
                    if temporary is not None:
                        temporary.unlink(missing_ok=True)
            c012_check(stage_path.read_bytes() == stage_raw,
                       "STAGE_SUMMARY_IMMUTABILITY")
            stage_record["stage_report_sha256"] = stage_sha
            print(json.dumps({"cell": "NB22E-v4:C012", "phase": "AUTO_STAGE_COMPLETE",
                              **stage_record}, ensure_ascii=False, indent=2), flush=True)
        close_original_batches()
    finally:
        for stream in opened:
            stream.close()
        if client is not None:
            client.close()
        session = None
        if temp_index is not None:
            temp_index.cleanup()


try:
    c012_run_automatic()
except KeyboardInterrupt:
    c012_stop = {"kind": "KEYBOARD_INTERRUPT", "candidate_id": c012_active,
                 "action": "STOP_WITHOUT_AUTOMATIC_RETRY_OF_STARTED"}
except Exception as error:
    c012_stop = {"kind": type(error).__name__, "detail": str(error)[:900],
                 "candidate_id": c012_active}
    sys.excepthook(type(error), error, error.__traceback__)

# Final re-audit is read-only; failed receipt validation must never be hidden.
c012_after = None
c012_final_audit_error = None
try:
    c012_after = c012_audit()
except Exception as error:
    c012_final_audit_error = type(error).__name__ + ": " + str(error)[:700]

c012_total_elapsed = time.perf_counter() - c012_run_start
c012_report = {
    "cell": "NB22E-v4:C012",
    "status": ("STOPPED_WITH_CHECKPOINTS" if c012_stop or c012_final_audit_error
               else "STAGE1_STAGE2_EXECUTION_COMPLETED"),
    "execution_boundary_sha256": c012_boundary_sha,
    "starting_complete": c012_completed_before, "pilot_reused": 4,
    "completed_new_candidates": (c012_after["counts"]["COMPLETE"]
                                  if c012_after else None),
    "remaining_candidates": (c012_after["counts"]["NOT_STARTED"]
                             + c012_after["counts"]["PARTIAL"]
                             + c012_after["counts"]["UNCERTAIN"]
                             + c012_after["counts"]["BLOCKED"]
                             if c012_after else None),
    "unique_onnx_pairs_completed": (c012_after["pair_done"]
                                    if c012_after else None),
    "total_pair_attempts": (c012_after["pair_attempts"]
                            if c012_after else None),
    "pair_attempt_ceiling": 55604,
    "unknown": (c012_after["unknown"] if c012_after else None),
    "work_this_invocation": c012_counts_run,
    "stage_count_completed": len(c012_stage_history),
    "total_elapsed_seconds": round(c012_total_elapsed, 3),
    "timing_seconds": {**{key: round(value, 3) for key, value in c012_timing.items()},
                       "other_unattributed": round(max(0.0, c012_total_elapsed
                                                      - sum(c012_timing.values())), 3)},
    "last_system_sample": c012_system_samples[-1] if c012_system_samples else None,
    "system_sample_count": len(c012_system_samples),
    "stop_reason": c012_stop,
    "final_audit_error": c012_final_audit_error,
    "paid_api_calls": 0,
    "repository_modified": False,
    "generation_authorized": False,
    "derived_order_quota_char_rrf_executed": False,
    "R11_2_quality_accepted": False,
}
raw = c012_canonical(c012_report) + b"\n"
sha = hashlib.sha256(raw).hexdigest()
path = c012_dir / ("C012.run_summary." + sha + ".json")
if not path.exists():
    temporary = None
    try:
        with tempfile.NamedTemporaryFile("wb", dir=c012_dir,
                                         prefix=".C012-summary-", suffix=".tmp",
                                         delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
        try:
            os.link(temporary, path)
        except FileExistsError:
            pass
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
c012_check(path.read_bytes() == raw, "FINAL_REPORT_RELOAD_FAILED")
c012_report["report_checkpoint"] = {"path": str(path), "sha256": sha}
NB22E_V4_POPULATION_EXECUTION = c012_report
print(json.dumps(c012_report, ensure_ascii=False, indent=2), flush=True)

{
  "cell": "NB22E-v4:C012",
  "phase": "AUTOMATIC_RESUME_PREFLIGHT",
  "completed": 9,
  "remaining": 1103,
  "pair_attempts": 454,
  "unique_pairs_remaining": 55150,
  "max_total_pair_attempts": 55604,
  "stage_size": 5,
  "onnx_pause_seconds": 0.8,
  "no_extra_retry": true
}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

{
  "cell": "NB22E-v4:C012",
  "phase": "AUTO_STAGE_COMPLETE",
  "stage": 1,
  "candidate_ids": [
    "golden_v2_candidate_024bde300780ba3cb472936f",
    "golden_v2_candidate_024f8781eb4783b0ae854bbc",
    "golden_v2_candidate_037df66a958e7c81f03c0ef6",
    "golden_v2_candidate_039f9f42a79d212add2663f9",
    "golden_v2_candidate_0408bb31681308c71cc2a633"
  ],
  "completed_total": 14,
  "remaining_candidates": 1098,
  "pair_attempts_total": 704,
  "unique_onnx_pairs_done_total": 700,
  "seconds": 103.17,
  "timing_seconds": {
    "source_sha": 0.0,
    "index_sha": 0.0,
    "input_read": 0.061,
    "model_setup": 1.634,
    "qdrant_setup": 3.208,
    "query": 0.223,
    "tokenization": 1.341,
    "onnx": 44.452,
    "pause": 51.472,
    "checkpoint": 0.516,
    "system_monitor": 0.082
  },
  "other_unattributed_seconds": 0.181,
  "work_in_stage": {
    "new_queries": 5,
    "new_onnx_attempts": 250,
    "new_onnx_completed": 250,
    "onnx_calls": 65,
    "new_candidates_completed": 5,


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
  "cell": "NB22E-v4:C012",
  "phase": "AUTO_STAGE_COMPLETE",
  "stage": 2,
  "candidate_ids": [
    "golden_v2_candidate_04098e4135143c1430ec50db",
    "golden_v2_candidate_0424c8ef79eaab7c81ba32c6",
    "golden_v2_candidate_0477a54d2504de5ce0f0359b",
    "golden_v2_candidate_0477b187a45eb76763843cc2",
    "golden_v2_candidate_04a46058d3a357905665d81f"
  ],
  "completed_total": 19,
  "remaining_candidates": 1093,
  "pair_attempts_total": 954,
  "unique_onnx_pairs_done_total": 950,
  "seconds": 88.656,
  "timing_seconds": {
    "source_sha": 0.0,
    "index_sha": 0.0,
    "input_read": 0.074,
    "model_setup": 0.0,
    "qdrant_setup": 0.0,
    "query": 1.02,
    "tokenization": 0.812,
    "onnx": 33.571,
    "pause": 52.27,
    "checkpoint": 0.603,
    "system_monitor": 0.061
  },
  "other_unattributed_seconds": 0.245,
  "work_in_stage": {
    "new_queries": 5,
    "new_onnx_attempts": 250,
    "new_onnx_completed": 250,
    "onnx_calls": 65,
    "new_candidates_completed": 5,
    "n

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
  "cell": "NB22E-v4:C012",
  "phase": "AUTO_STAGE_COMPLETE",
  "stage": 4,
  "candidate_ids": [
    "golden_v2_candidate_059b226265e9b8a146978aac",
    "golden_v2_candidate_05acfe991a2b90c40ccaf873",
    "golden_v2_candidate_05d052d545a5eedefbd01558",
    "golden_v2_candidate_05e1c38a910abe497565084c",
    "golden_v2_candidate_065bbed1933859ad5ba020db"
  ],
  "completed_total": 29,
  "remaining_candidates": 1083,
  "pair_attempts_total": 1454,
  "unique_onnx_pairs_done_total": 1450,
  "seconds": 89.868,
  "timing_seconds": {
    "source_sha": 0.0,
    "index_sha": 0.0,
    "input_read": 0.17,
    "model_setup": 0.0,
    "qdrant_setup": 0.0,
    "query": 0.253,
    "tokenization": 0.517,
    "onnx": 35.432,
    "pause": 52.304,
    "checkpoint": 0.867,
    "system_monitor": 0.061
  },
  "other_unattributed_seconds": 0.264,
  "work_in_stage": {
    "new_queries": 5,
    "new_onnx_attempts": 250,
    "new_onnx_completed": 250,
    "onnx_calls": 65,
    "new_candidates_completed": 5,
   

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


RuntimeError: C012_MEASURED_THERMAL_CPU_SPEED_LIMIT

{
  "cell": "NB22E-v4:C012",
  "status": "STOPPED_WITH_CHECKPOINTS",
  "execution_boundary_sha256": "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc",
  "starting_complete": 9,
  "pilot_reused": 4,
  "completed_new_candidates": 29,
  "remaining_candidates": 1083,
  "unique_onnx_pairs_completed": 1454,
  "total_pair_attempts": 1458,
  "pair_attempt_ceiling": 55604,
  "unknown": [],
  "work_this_invocation": {
    "new_queries": 21,
    "new_onnx_attempts": 1004,
    "new_onnx_completed": 1004,
    "onnx_calls": 261,
    "new_candidates_completed": 20,
    "new_retries": 0
  },
  "stage_count_completed": 4,
  "total_elapsed_seconds": 374.526,
  "timing_seconds": {
    "source_sha": 1.349,
    "index_sha": 0.236,
    "input_read": 2.398,
    "model_setup": 1.634,
    "qdrant_setup": 3.208,
    "query": 1.9,
    "tokenization": 3.128,
    "onnx": 141.127,
    "pause": 209.093,
    "checkpoint": 3.449,
    "system_monitor": 0.576,
    "other_unattributed": 6.429
  },
  "las

<div dir="rtl" align="right">

### NB22E-v4:C013 — R11.2: بررسی تخصصی Batch نامعلوم پس از Interrupt

**هدف:** بررسی وضعیت دقیق چهار جفت ONNX مربوط به Batch شمارهٔ ۲۰ سؤال نیمه‌تمام، بدون اجرای مجدد.

**ورودی:** ممیزی موفق C012، Manifest مستقل C011 و رسیدهای ذخیره‌شدهٔ C012.

**اقدامات:**
- تأیید وضعیت Stage 1 سؤال نیمه‌تمام.
- بازخوانی و اعتبارسنجی رسیدهای Batchهای موفق همان سؤال.
- بررسی STARTED بدون DONE در Batch شمارهٔ ۲۰.
- جست‌وجوی محدود فایل موقت احتمالی با محتوای قابل‌اعتبارسنجی.
- تفکیک نتیجهٔ قطعی از محاسبه‌ای که وضعیت آن نامعلوم است.

**محدودیت:** بدون Retrieval، ONNX، Tokenizer، API، تغییر Checkpoint یا نوشتن فایل.

**شرط پایان:** مشخص شدن شواهد قابل‌بازیابی یا نیاز به تصمیم صریح برای Batch نامعلوم. نتایج سالم قبلی دست‌نخورده باقی می‌مانند.

</div>

In [12]:
# NB22E-v4 | C013 | R11.2
# Read-only forensic audit of the interrupted ONNX batch; no retry or writes.

def c013_need(ok, code):
    if not ok:
        raise RuntimeError("C013_" + code)


c013_target = "golden_v2_candidate_0184f90d79f3dfc4744b6e29"
c013_unknown_start = 20

c013_need(
    c012_execute_new_work is False
    and c012_report["status"] == "AUDIT_ONLY_NO_EXECUTION"
    and c012_report["execution_boundary_sha256"]
    == c012_boundary_sha
    and len(c012_before["uncertain"]) == 1
    and c012_before["uncertain"][0]["candidate_id"]
    == c013_target
    and c012_before["uncertain"][0]["unit"]
    == "ONNX_BATCH"
    and c012_before["uncertain"][0]["batch_starts"]
    == [c013_unknown_start]
    and c013_target in c012_ids
    and c012_before["states"][c013_target]
    == "UNCERTAIN",
    "AUDIT_BASELINE_CONFLICT",
)

c013_stage1, c013_stage1_sha = c012_read(
    c012_path(c013_target, "stage1_done")
)

c013_need(
    c013_stage1["status"] == "STAGE1_PASS"
    and c013_stage1["candidate_id"] == c013_target
    and c013_stage1["boundary"] == c012_boundary_sha
    and c013_stage1["full_order_matches"] is True
    and c013_stage1["pool50_membership_matches"] is True,
    "STAGE1_NOT_VERIFIED",
)

c013_identity = c013_stage1["input_sha256"]

c013_receipts = []
c013_missing = []
c013_completed_pairs = 0

for start in range(0, 50, 4):
    started_path = c012_path(
        c013_target,
        "stage2_batch_%02d_started" % start,
    )
    done_path = c012_path(
        c013_target,
        "stage2_batch_%02d_done" % start,
    )

    started_exists = started_path.is_file()
    done_exists = done_path.is_file()

    c013_need(
        not done_exists or started_exists,
        "DONE_WITHOUT_STARTED:" + str(start),
    )

    started = None
    started_sha = None

    if started_exists:
        started, started_sha = c012_read(started_path)

        c013_need(
            started["candidate_id"] == c013_target
            and started["boundary"] == c012_boundary_sha
            and started["input_sha256"] == c013_identity
            and started["batch_start"] == start
            and started["status"] == "STARTED_UNCONFIRMED",
            "STARTED_RECEIPT_CONFLICT:" + str(start),
        )

    if done_exists:
        done, done_sha = c012_read(done_path)

        c013_need(
            done["status"] == "BATCH_PASS"
            and done["candidate_id"] == c013_target
            and done["boundary"] == c012_boundary_sha
            and done["input_sha256"] == c013_identity
            and done["chunk_ids"] == started["chunk_ids"]
            and len(done["scores"])
            == len(started["chunk_ids"])
            and all(
                math.isfinite(float(value))
                for value in done["scores"]
            ),
            "DONE_RECEIPT_CONFLICT:" + str(start),
        )

        c013_completed_pairs += len(done["scores"])

        c013_receipts.append({
            "batch_start": start,
            "status": "VERIFIED_DONE",
            "pair_count": len(done["scores"]),
            "file_sha256": done_sha,
        })

    elif started_exists:
        c013_missing.append({
            "batch_start": start,
            "status": "STARTED_WITHOUT_DONE",
            "pair_count": len(started["chunk_ids"]),
            "started_sha256": started_sha,
            "started_mtime_ns": started_path.stat().st_mtime_ns,
        })

    else:
        c013_receipts.append({
            "batch_start": start,
            "status": "NOT_STARTED",
        })

c013_need(
    len(c013_missing) == 1
    and c013_missing[0]["batch_start"] == 20
    and c013_missing[0]["pair_count"] == 4
    and c013_completed_pairs == 20
    and not c012_path(
        c013_target,
        "stage2_done",
    ).exists(),
    "INTERRUPTED_WORK_STATE_CHANGED",
)

# Inspect possible temporary receipts without modifying or promoting them.
c013_temp_files = []
c013_recoverable_candidates = []

if c012_dir.is_dir():
    c013_temp_files = sorted(
        p
        for p in c012_dir.iterdir()
        if p.name.startswith(".C012-")
        and p.name.endswith(".tmp")
        and p.is_file()
        and not p.is_symlink()
    )

c013_need(
    len(c013_temp_files) <= 500,
    "UNEXPECTED_TEMP_FILE_VOLUME",
)

c013_started_unknown, _ = c012_read(
    c012_path(
        c013_target,
        "stage2_batch_20_started",
    )
)

for temp_path in c013_temp_files:
    if temp_path.stat().st_size > 1024 * 1024:
        continue

    try:
        raw = temp_path.read_bytes()
        record = json.loads(raw)

    except (OSError, ValueError, UnicodeDecodeError):
        continue

    if not isinstance(record, dict):
        continue

    if record.get("candidate_id") != c013_target:
        continue

    if record.get("chunk_ids") != c013_started_unknown["chunk_ids"]:
        continue

    declared_sha = record.get("core_sha256")

    core = {
        key: value
        for key, value in record.items()
        if key != "core_sha256"
    }

    hash_valid = (
        isinstance(declared_sha, str)
        and hashlib.sha256(
            c012_json(core)
        ).hexdigest() == declared_sha
    )

    content_valid = (
        core.get("status") == "BATCH_PASS"
        and core.get("boundary") == c012_boundary_sha
        and core.get("input_sha256") == c013_identity
        and isinstance(core.get("scores"), list)
        and len(core["scores"]) == 4
        and all(
            isinstance(score, (int, float))
            and math.isfinite(score)
            for score in core["scores"]
        )
    )

    c013_recoverable_candidates.append({
        "filename": temp_path.name,
        "byte_sha256": hashlib.sha256(raw).hexdigest(),
        "core_sha256_valid": hash_valid,
        "content_schema_valid": content_valid,
        "requires_manual_adjudication": True,
    })

c013_report = {
    "cell": "NB22E-v4:C013",
    "status": "INTERRUPTED_BATCH_FORENSIC_AUDIT",
    "execution_boundary_sha256": c012_boundary_sha,
    "candidate_id": c013_target,
    "stage1_verified": True,
    "stage1_receipt_sha256": c013_stage1_sha,
    "candidate_completed_onnx_pairs": c013_completed_pairs,
    "verified_completed_batch_starts": [
        item["batch_start"]
        for item in c013_receipts
        if item["status"] == "VERIFIED_DONE"
    ],
    "uncertain_batch": c013_missing[0],
    "temp_files_found": len(c013_temp_files),
    "potential_temp_receipts": c013_recoverable_candidates,
    "automatic_retry_allowed": False,
    "automatic_checkpoint_promotion_allowed": False,
    "healthy_receipts_preserved": True,
    "new_retrieval_queries": 0,
    "new_onnx_calls": 0,
    "new_api_calls": 0,
    "repository_writes": 0,
    "artifact_writes": 0,
    "next": "MANUAL_ADJUDICATION_OF_SINGLE_UNCERTAIN_ONNX_BATCH",
}

print(
    json.dumps(
        c013_report,
        ensure_ascii=False,
        indent=2,
    ),
    flush=True,
)

{
  "cell": "NB22E-v4:C013",
  "status": "INTERRUPTED_BATCH_FORENSIC_AUDIT",
  "execution_boundary_sha256": "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc",
  "candidate_id": "golden_v2_candidate_0184f90d79f3dfc4744b6e29",
  "stage1_verified": true,
  "stage1_receipt_sha256": "f40eb6ac904d87744da9191803bf8c9bb4773c3c39fc84c62988b378c554e821",
  "candidate_completed_onnx_pairs": 20,
  "verified_completed_batch_starts": [
    0,
    4,
    8,
    12,
    16
  ],
  "uncertain_batch": {
    "batch_start": 20,
    "status": "STARTED_WITHOUT_DONE",
    "pair_count": 4,
    "started_sha256": "4745577680cbd38d3a212e27ec67e3af98a42088cd2f3aace077fca563e4daf2",
    "started_mtime_ns": 1794138552775836347
  },
  "temp_files_found": 0,
  "potential_temp_receipts": [],
  "automatic_retry_allowed": false,
  "automatic_checkpoint_promotion_allowed": false,
  "healthy_receipts_preserved": true,
  "new_retrieval_queries": 0,
  "new_onnx_calls": 0,
  "new_api_calls": 0,
  "repository_


<div dir="rtl" align="right">

### NB22E-v4:C014 — اتصال نتایج نهایی آروان به ادامهٔ محلی

**هدف:** بازیابی وضعیت معتبر C012 از Artifactهای منتقل‌شده به مک، بدون بازاجرای محاسبات قبلی.

**ورودی‌ها:**
- Manifest فریز‌شدهٔ C011
- خروجی‌های کامل ۱۱۱۲ سؤال C012
- گزارش Audit و تأییدیهٔ SHA
- Runner منتقل‌شده از آروان

**معیار موفقیت:** تطابق هویت اجرا، وجود همهٔ خروجی‌های مورد انتظار و معتبر بودن شواهد تکمیل فنی.

**محدودیت:** فقط خواندن فایل‌ها؛ بدون ONNX، Retrieval، API، تغییر Git یا نوشتن Artifact.

**مرحلهٔ بعد:** تعیین بستهٔ اجرایی Rank نهایی، Quota، char_rrf و Context بر اساس قراردادهای موجود.

</div>


In [1]:
# NB22E-v4 | C014 | R11.2
# Local C012 handoff reconciliation; read-only, no API or model calls.

from pathlib import Path
import hashlib
import json

def c014_sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Locate the existing Mac project without executing earlier cells.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates.append(
    Path.home()
    / "AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance"
)

repo = next(
    (
        p for p in candidates
        if (p / "notebooks/phase2/22e_retrieval_experiment_lab_v4.ipynb").is_file()
    ),
    None
)

if repo is None:
    raise RuntimeError("C014_PROJECT_ROOT_NOT_FOUND")

base = repo.parent
outputs = base / "_RAG_finance_NB22E_v4_outputs"

boundary = (
    "aa099a56e76054c30629a35a3acbd1ae9a86970367c9fb60677b907c2f0d2cbc"
)
manifest_sha = (
    "a3ffce372eab24e210bedae9b2b6087dc2493e45805c4c27836d223a6578fe80"
)

manifest_path = outputs / f"C011.population_boundary.{manifest_sha}.json"
c012_dir = outputs / f"C012_population_{boundary}"

transfer = (
    base / "_RAG_finance_Arvan_C012_backups"
    / "ARVAN_FINAL_RETURN/runner_inputs"
)
attestation_path = transfer / "C012_AUDIT_SHA_ATTESTATION_20261110T194001Z.json"
audit_log_path = transfer / "C012_AUDIT_PATCHED_20261110T194001Z.log"
runner_path = transfer / "c012_arvan_resumable_runner.py"

required = [manifest_path, attestation_path, audit_log_path, runner_path]
for path in required:
    if not path.is_file():
        raise RuntimeError(f"C014_MISSING_INPUT: {path.name}")

manifest = json.loads(manifest_path.read_bytes())
attestation = json.loads(attestation_path.read_bytes())
audit_text = audit_log_path.read_text(encoding="utf-8")

receipts = list(c012_dir.glob("C012.*.stage2_done.json"))

checks = {
    "MANIFEST_SHA": c014_sha256(manifest_path) == manifest_sha,
    "EXECUTION_BOUNDARY": manifest["execution_boundary_sha256"] == boundary,
    "POPULATION_1116": len(manifest["execution_contract"]["members"]) == 1116,
    "PILOTS_4": len(manifest["approved_pilot_provenance"]["pilot_ids"]) == 4,
    "LOGICAL_BATCHES_45": len(manifest["execution_queue"]["batches"]) == 45,
    "C012_RECEIPTS_1112": len(receipts) == 1112,
    "RUNNER_SHA": (
        c014_sha256(runner_path)
        == attestation["original_runner_sha256"]
    ),
    "AUDIT_LOG_SHA": (
        c014_sha256(audit_log_path)
        == attestation["original_audit_log_sha256"]
    ),
    "AUDIT_INTERNAL_PASS": (
        "C012_AUDIT_OK completed=1112/1112" in audit_text
        and "pairs_done=55600 attempts=55604" in audit_text
    ),
    "AUDIT_ATTESTATION": (
        attestation["status"]
        == "INTERNAL_AUDIT_PASS_WRAPPER_SHA_COUNT_ERROR"
        and attestation["completed"] == 1112
        and attestation["fresh_pairs"] == 55600
        and attestation["total_pair_attempts"] == 55604
        and attestation["new_work_during_audit"] == 0
    ),
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

if not all(checks.values()):
    raise RuntimeError("C014_HANDOFF_RECONCILIATION_FAILED")

NB22E_V4_C014_HANDOFF = {
    "status": "LOCAL_C012_HANDOFF_READY",
    "repository_root": str(repo),
    "output_directory": str(c012_dir),
    "execution_boundary_sha256": boundary,
    "completed_new_questions": 1112,
    "historical_pilots": 4,
    "completed_onnx_pairs": 55600,
    "total_pair_attempts": 55604,
    "new_api_calls": 0,
    "new_model_calls": 0,
    "artifact_writes": 0,
    "quality_gate": "BLOCKED",
    "next": "FROZEN_RANK_CONTEXT_CONTRACT_REVIEW",
}

print("\nC014_STATUS:", NB22E_V4_C014_HANDOFF["status"])
print("NEXT:", NB22E_V4_C014_HANDOFF["next"])


MANIFEST_SHA: PASS
EXECUTION_BOUNDARY: PASS
POPULATION_1116: PASS
PILOTS_4: PASS
LOGICAL_BATCHES_45: PASS
C012_RECEIPTS_1112: PASS
RUNNER_SHA: PASS
AUDIT_LOG_SHA: PASS
AUDIT_INTERNAL_PASS: PASS
AUDIT_ATTESTATION: PASS

C014_STATUS: LOCAL_C012_HANDOFF_READY
NEXT: FROZEN_RANK_CONTEXT_CONTRACT_REVIEW
